# Preparación del último corte de Gestión-Inversión y control de las mediciones

## 1. Propósito, unidad y evidencia

Preparo las actividades de la instantánea seleccionada, conservando campos fuente y trazabilidad. Distingo fecha observada, corte de publicación y edición del archivo. No concateno el historial de referencia con las filas de salida.

La metodología separa conservación, conversión de escala, aplicabilidad y diagnóstico. Una proporción coherente aritméticamente no certifica que la entidad haya medido correctamente la actividad. Los hallazgos del archivo seleccionado se calculan durante la ejecución; no presupongo tamaño, cobertura o incidencias constantes.

La referencia histórica está incorporada en los metadatos de este notebook e incluye fuentes, fechas de edición y huellas. Es una referencia versionada, no un historial que se amplíe automáticamente. En un corte posterior utilizo los antecedentes comparables disponibles y mantengo visibles sus límites.

**Lectura de los resultados.** La metodología describe el procedimiento para cualquier corte compatible. Los resúmenes que aparecen al ejecutar las celdas describen exclusivamente la fuente seleccionada. Para actualizar el estudio ejecuto el notebook desde el inicio; las salidas guardadas no se actualizan por abrir el archivo. Mantengo las revisiones anteriores en un anexo histórico, sin atribuirlas al corte actual.



In [1]:
# Si faltan dependencias:
# %pip install pandas numpy openpyxl

### Selección del corte de trabajo

En esta celda selecciono una fuente local inequívoca del corte indicado o del último corte común. Registro su huella y dejo que el siguiente bloque lea el mapa de vigencias desde el diccionario, en lugar de escribir los años manualmente.


In [2]:
from pathlib import Path
from datetime import date, datetime, timezone
import csv
import hashlib
import re
import unicodedata
import platform
import numpy as np
import pandas as pd
import openpyxl
from IPython.display import display, Markdown

# CONFIGURACIÓN EDITABLE DEL PLAN
MAPA_VIGENCIAS = None  # La carga lo deriva de las observaciones del diccionario oficial.
CORTE = None  # None: último corte local; o fecha exacta "AAAA-MM-DD".
ARCHIVO = None  # Nombre exacto si hay varias versiones del Excel en ese corte.
# Buscar la carpeta del proyecto, también al abrir un notebook desde una subcarpeta.
import os
import sys
sys.dont_write_bytecode = True
_carpeta_indicada = os.environ.get("PROYECTO_RAIZ")
_candidatas = ([Path(_carpeta_indicada).expanduser().resolve()] if _carpeta_indicada
               else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
RAIZ = next((p for p in _candidatas if (p / "rutas.py").is_file()
             and (p / "01_descarga_ultimo_corte.ipynb").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abre Jupyter desde la carpeta del proyecto.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from rutas import raiz_proyecto, corte_local_comun
RAIZ = raiz_proyecto()
BRONCE = RAIZ / "Capa Bronce"
CLAVE = ["ENTIDAD_CODIGO", "PROYECTO_CODIGO", "PROYECTO_ACTIVIDAD_CODIGO"]
def seleccionar_fuente(base, corte=None, archivo=None):
    cortes = []
    for carpeta in base.iterdir():
        if not carpeta.is_dir() or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", carpeta.name):
            continue
        fecha = date.fromisoformat(carpeta.name)
        candidatos = [p for p in (carpeta / "gestion_inversion").glob("*.xlsx") if not p.name.startswith("~$")]
        if candidatos:
            cortes.append((fecha, carpeta))
    if not cortes:
        raise FileNotFoundError("No hay Excel de Gestión-Inversión en los cortes locales.")
    carpeta = base / corte if corte else max(cortes, key=lambda x: x[0])[1]
    candidatos = sorted(p for p in (carpeta / "gestion_inversion").glob("*.xlsx")
                        if not p.name.startswith("~$") and (archivo is None or p.name == archivo))
    if len(candidatos) != 1:
        raise ValueError(f"Revisa CORTE/ARCHIVO: se requiere una versión única, encontradas {candidatos}")
    return candidatos[0], date.fromisoformat(carpeta.name)

FUENTE, FECHA = seleccionar_fuente(BRONCE, CORTE if CORTE else corte_local_comun(RAIZ), ARCHIVO)
HUELLA = hashlib.sha256(FUENTE.read_bytes()).hexdigest()
DESTINO = RAIZ / "Capa Silver" / FECHA.isoformat() / "gestion_inversion"
print("Fuente:", FUENTE.relative_to(RAIZ))
print("Corte:", FECHA, "| SHA-256:", HUELLA)


Fuente: Capa Bronce\2026-06-30\gestion_inversion\planacciongestioninversion20260630_ver20260728_75dd904475.xlsx
Corte: 2026-06-30 | SHA-256: 5bec7adbcc4215d58afc28125758757f1cc8b2837e7142a657fbbe90a86d329f


## 2. Reconocimiento del Excel y de su diccionario

Identifico la hoja de actividades y su diccionario entre los perfiles implementados. Conservo la lectura fuente y el mapa de homologación; no exijo hojas auxiliares de otra edición. La copia oficial del diccionario mantiene nombre, contenido y formato originales.

Obtengo los años de VIG desde las observaciones del diccionario. Si no son inequívocos, detengo la carga. También verifico la identidad del plan y su correspondencia con Avance del Plan cuando existe la base preparada del mismo corte.

Enumero los campos sin entrada en el diccionario oficial sin inventarles una definición institucional. Compruebo la representación de los porcentajes: si cambia a fracciones con formato porcentual activo, detengo la carga para revisar la escala antes de dividir entre cien.



In [3]:
def cargar_gestion(fuente):
    """Carga versiones 76/77/78 campos, sin exigir hojas auxiliares de otro corte."""
    import re
    import pandas as pd
    import openpyxl
    libro = openpyxl.load_workbook(fuente, read_only=True, data_only=False)
    candidatas = [n for n in libro.sheetnames if n.casefold() in {'datos','para_web_datos'}]
    diccionarios = [n for n in libro.sheetnames if n.casefold().endswith('diccionariodedatos')]
    if len(candidatas)!=1 or len(diccionarios)!=1:
        libro.close()
        raise ValueError('Esquema no validado: no mezclo actividades BCS con metas del plan UNCSAB.')
    nombre_datos, nombre_diccionario = candidatas[0], diccionarios[0]
    original = pd.read_excel(fuente,sheet_name=nombre_datos,dtype=object,keep_default_na=False)
    renombres = {c:str(c).upper() for c in original}
    renombres.update({c:'PLAN_META_PRODUCTO_CODIGO' for c in original if str(c).upper()=='PLAN_META_PRODUCTO_ID'})
    origen = original.rename(columns=renombres)
    if not origen.columns.is_unique or any(str(c).startswith('Unnamed:') for c in origen):
        libro.close()
        raise ValueError('Encabezados de datos vacíos o duplicados.')
    if origen.PLAN_CODIGO.astype(str).str.replace(r'\.0$', '', regex=True).nunique() != 1:
        libro.close()
        raise ValueError('La fuente contiene más de un plan; revisar la definición de actividad.')
    bruto = pd.read_excel(fuente,sheet_name=nombre_diccionario,header=None,dtype=object,keep_default_na=False)
    ubicacion = []
    for i in range(min(12,len(bruto))):
        vals=[str(v).strip().casefold() for v in bruto.iloc[i]]
        if 'columna' in vals and 'tipo' in vals:
            ubicacion.append((i,vals.index('columna')))
    if len(ubicacion)!=1:
        libro.close()
        raise ValueError('Encabezado del diccionario no reconocido.')
    fila, col = ubicacion[0]
    dic = pd.read_excel(fuente,sheet_name=nombre_diccionario,header=fila,dtype=object,keep_default_na=False)
    # La copia Excel oficial se preserva nativa; esta vista sirve solo para validar y documentar.
    dic = dic.iloc[:,col:col+4].copy()
    dic.columns = ['Columna','Tipo','Descripción','Observación']
    dic = dic.loc[dic.Columna.astype(str).str.strip().ne('')].copy()
    dic.Columna = dic.Columna.map(lambda c:renombres.get(c,str(c).upper()))
    faltan = set(origen.columns)-set(dic.Columna)
    sobran = set(dic.Columna)-set(origen.columns)
    if sobran or faltan-{'FE_DE_ERRATAS'}:
        libro.close()
        raise ValueError('El diccionario no cubre exactamente los campos fuente.')
    dic.insert(0,'Grupo','')  # Mantengo la vista de cinco campos utilizada por la documentación del notebook.
    indices = {int(m[1]) for c in origen for m in [re.search(r'_MAGVIG(\d+)_',c)] if m}
    mapa={}
    for indice in sorted(indices):
        candidatas=set()
        for _,r in dic.loc[dic.Columna.str.contains(f'_MAGVIG{indice}_',regex=False)].iterrows():
            candidatas.update(int(y) for y in re.findall(r'\b(?:19|20)\d{2}\b',str(r['Observación'])))
        if len(candidatas)!=1:
            libro.close()
            raise ValueError(f'No se identifica de forma única el año de VIG{indice} en el diccionario.')
        mapa[indice]=candidatas.pop()
    inventario=[]
    for hoja in libro:
        # Diagnostico las hojas auxiliares; solo datos y diccionario definen el contrato de carga.
        formulas=errores=None
        if hoja.title in [nombre_datos,nombre_diccionario]:
            formulas=errores=0
            indices_porcentaje = {i for i,col in enumerate(original.columns) if str(col).upper().endswith(("PORCCOMPROMISOS","PORCENTREGADO"))}
            for numero_fila, row in enumerate(hoja,start=1):
                if hoja.title == nombre_datos and numero_fila > 1:
                    for i in indices_porcentaje:
                        celda = row[i]
                        if celda.value is None:
                            continue
                        formato_activo = re.sub(r'"[^"]*"|\\.', "", celda.number_format)
                        if "%" in formato_activo or celda.data_type == "d":
                            libro.close()
                            raise ValueError("Cambió la representación de los porcentajes fuente; no divido entre cien sin revisar la escala del nuevo formato.")
                for c in row:
                    formulas += c.data_type=='f'; errores += c.data_type=='e'
        inventario.append({'hoja':hoja.title,'filas_fisicas':hoja.max_row,'columnas':hoja.max_column,
                           'formulas':formulas,'errores':errores})
        if hoja.title in [nombre_datos,nombre_diccionario] and (formulas or errores):
            libro.close()
            raise ValueError(f'Fórmulas o errores en los campos consumidos: {hoja.title}')
    libro.close()
    return original, origen, dic, mapa, nombre_diccionario, pd.DataFrame(inventario), renombres

bruta_fuente, origen, dic_origen, MAPA_VIGENCIAS, NOMBRE_DICCIONARIO, inventario, renombres_fuente = cargar_gestion(FUENTE)
display(inventario)
display(pd.DataFrame(sorted(MAPA_VIGENCIAS.items()), columns=["numero_vigencia", "anio"]))
sin_definicion = sorted(set(origen.columns) - set(dic_origen.Columna))
print("Campos fuente sin definición en el diccionario oficial:", sin_definicion)
PCT = [c for c in origen if re.fullmatch(r"PROYECTO_ACTIVIDAD_MAG(?:VIG\d+|TOTAL)_PORC(?:COMPROMISOS|ENTREGADO)", c)]
NUMERICAS = [c for c in origen if re.fullmatch(r"PROYECTO_ACTIVIDAD_(?:MAG(?:VIG\d+|TOTAL)|REC(?:KVIG\d+|VIG\d+|KTOTAL))_.*", c)]
if len(PCT) != 2 * (len(MAPA_VIGENCIAS) + 1):
    raise ValueError("Cambió el esquema de porcentajes físicos.")
display(origen.head(3))

# Si ya preparé Avance del Plan, exijo identidad de plan y corte entre las fuentes.
referencia_plan = RAIZ / "Capa Silver" / FECHA.isoformat() / "avance_plan" / "avance_plan.csv"
if referencia_plan.exists():
    plan_referencia = pd.read_csv(referencia_plan, sep="|", dtype="string", keep_default_na=False)
    if set(plan_referencia.PLAN_CODIGO) != set(origen.PLAN_CODIGO.astype(str)):
        raise ValueError("Avance del Plan y Gestión-Inversión pertenecen a planes diferentes.")

referencia_metas = RAIZ / "Capa Silver" / FECHA.isoformat() / "avance_plan" / "avance_meta_producto.csv"
if referencia_metas.exists():
    metas_referencia = pd.read_csv(referencia_metas, sep="|", dtype="string", keep_default_na=False)
    metas_fuente = set(origen.PLAN_META_PRODUCTO_CODIGO.astype(str))
    if not metas_fuente.issubset(set(metas_referencia.PLAN_META_PRODUCTO_CODIGO)):
        raise ValueError("Hay metas de Gestión-Inversión sin correspondencia en Avance del Plan del mismo corte.")


,hoja,filas_fisicas,columnas,formulas,errores
0,DiccionarioDeDatos,84,5,0.0,0.0
1,datos,1656,78,0.0,0.0
2,FeDeErratas,9,2,NaN,NaN
3,IPC,5,4,NaN,NaN
4,TablasDinamicasIncluidas,11,2,NaN,NaN
5,TD_EjecPptalxSector,20,5,NaN,NaN
6,TD_EjecPptalxEntidad,52,5,NaN,NaN
7,TD_EjecPptalxOE,9,5,NaN,NaN
8,TD_EjecPptalxOEyProg,48,7,NaN,NaN
9,TD_EjecPptalxOEyProgMetaProd,473,9,NaN,NaN


,numero_vigencia,anio
0,1,2024
1,2,2025
2,3,2026
3,4,2027


Campos fuente sin definición en el diccionario oficial: []


,PLAN_CODIGO,PLAN_NOMBRE,PLAN_SIGLA,VIGENCIA,VIGENCIA_FECHA_SEGUIMIENTO,ENTIDAD_CODIGO,ENTIDAD_NOMBRE,ENTIDAD_SIGLA,ENTIDAD_CLASIFICACION,SECTOR_CODIGO,...,PROYECTO_ACTIVIDAD_RECKVIG3_GIROS,PROYECTO_ACTIVIDAD_RECKVIG4_PROGRAMADO,PROYECTO_ACTIVIDAD_RECKVIG4_COMPROMISOS_CONTRATADO,PROYECTO_ACTIVIDAD_RECKVIG4_GIROS,PROYECTO_ACTIVIDAD_RECKTOTAL_PROGRAMADO,PROYECTO_ACTIVIDAD_RECKTOTAL_COMPROMISOS_CONTRATADO,PROYECTO_ACTIVIDAD_RECKTOTAL_GIROS,ODS,ODS_META,FE_DE_ERRATAS
0,17,Bogotá Camina Segura,BCS,2026,30/06/2026 (Terminado),0102,Personería Distrital,PD,1 - Administración central,017,...,0,3691.539568,0,0,9181.349504,5489.808292,2731.565805,"16-Paz, Justicia e Instituciones Sólidas",16.06-Crear a todos los niveles instituciones ...,
1,17,Bogotá Camina Segura,BCS,2026,30/06/2026 (Terminado),0102,Personería Distrital,PD,1 - Administración central,017,...,0,729.016787,0,0,2068.396787,1339.38,1339.38,"16-Paz, Justicia e Instituciones Sólidas",16.06-Crear a todos los niveles instituciones ...,
2,17,Bogotá Camina Segura,BCS,2026,30/06/2026 (Terminado),0102,Personería Distrital,PD,1 - Administración central,017,...,0,2985.640288,0,0,9416.401039,680.7565,671.52832,"16-Paz, Justicia e Instituciones Sólidas",16.06-Crear a todos los niveles instituciones ...,


### Perfil de calidad y tipos antes de limpiar

Construyo un perfil por campo, cuento claves repetidas y vacíos y examino los tipos de anualización y los porcentajes publicados. Necesito esta evidencia para distinguir una transformación de representación de un cambio de significado.


In [4]:
def texto_limpio(v):
    return " ".join(unicodedata.normalize("NFC", v).split()) if isinstance(v, str) else v

def vacio(v):
    return v is None or (isinstance(v, str) and not v.strip()) or pd.isna(v)

perfil = pd.DataFrame([{
    "columna": c, "vacios": sum(vacio(v) for v in origen[c]),
    "distintos": origen[c].nunique(),
    "tipos": ", ".join(sorted({type(v).__name__ for v in origen[c]})),
    "textos_a_normalizar": sum(isinstance(v, str) and v != texto_limpio(v) for v in origen[c])
} for c in origen])
print("Filas:", len(origen), "| Columnas:", len(origen.columns),
      "| Duplicados exactos:", int(origen.duplicated().sum()),
      "| Claves repetidas:", int(origen.duplicated(CLAVE).sum()))
display(perfil.loc[(perfil.vacios > 0) | (perfil.textos_a_normalizar > 0)])
display(origen.PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION.map(texto_limpio).value_counts().rename("actividades").to_frame())
resumen_porcentajes = []
for c in PCT:
    s = pd.to_numeric(origen[c], errors="raise")
    resumen_porcentajes.append({"columna": c, "minimo": s.min(), "maximo": s.max(),
                               "ceros": int((s == 0).sum()), "mayores_100": int((s > 100).sum())})
display(pd.DataFrame(resumen_porcentajes))
display(perfil[["columna", "tipos"]])

Filas: 1655 | Columnas: 78 | Duplicados exactos: 0 | Claves repetidas: 0


,columna,vacios,distintos,tipos,textos_a_normalizar
16,PLAN_META_PRODUCTO_NOMBRE,0,425,str,5
19,PROYECTO_NOMBRE,0,353,str,11
21,PROYECTO_ACTIVIDAD_NOMBRE,0,1655,str,260
22,PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION,0,4,str,728
75,ODS,0,18,str,54
77,FE_DE_ERRATAS,1634,9,str,0


,actividades
PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION,
Suma,927
Constante,553
Creciente,169
Decreciente,6


,columna,minimo,maximo,ceros,mayores_100
0,PROYECTO_ACTIVIDAD_MAGVIG1_PORCCOMPROMISOS,0.0,133.112583,1653,1
1,PROYECTO_ACTIVIDAD_MAGVIG1_PORCENTREGADO,0.0,10000.000000,298,67
2,PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS,0.0,13050.000000,183,83
3,PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO,0.0,10000.000000,191,48
4,PROYECTO_ACTIVIDAD_MAGVIG3_PORCCOMPROMISOS,0.0,6370.000000,295,31
5,PROYECTO_ACTIVIDAD_MAGVIG3_PORCENTREGADO,0.0,6370.000000,338,19
6,PROYECTO_ACTIVIDAD_MAGVIG4_PORCCOMPROMISOS,0.0,0.000000,1655,0
7,PROYECTO_ACTIVIDAD_MAGVIG4_PORCENTREGADO,0.0,0.000000,1655,0
8,PROYECTO_ACTIVIDAD_MAGTOTAL_PORCCOMPROMISOS,0.0,7525.000000,817,27
9,PROYECTO_ACTIVIDAD_MAGTOTAL_PORCENTREGADO,0.0,7525.000000,823,8


,columna,tipos
0,PLAN_CODIGO,int
1,PLAN_NOMBRE,str
2,PLAN_SIGLA,str
3,VIGENCIA,int
4,VIGENCIA_FECHA_SEGUIMIENTO,str
...,...,...
73,PROYECTO_ACTIVIDAD_RECKTOTAL_COMPROMISOS_CONTR...,"float, int"
74,PROYECTO_ACTIVIDAD_RECKTOTAL_GIROS,"float, int"
75,ODS,str
76,ODS_META,str


### 2.1. Contraste entre porcentajes y magnitudes

Comparo el porcentaje publicado con magnitud/programado por cien cuando el denominador es distinto de cero. Admito para este diagnóstico una tolerancia absoluta de 0,0051 puntos porcentuales y relativa de 1e-9, separada de las comprobaciones de conservación.

Una diferencia señala la necesidad de revisar fecha, anualización, unidad y referencias a erratas. No sustituye por sí sola el porcentaje institucional. Muestro los conteos completos y una muestra de incidencias; el detalle íntegro permanece en memoria.



In [5]:
# Comparo con un cociente de referencia para diagnosticar, no para sobrescribir la fuente.
incidencias_cociente = []
contrastes_cociente = []
for token in [*(f"VIG{i}" for i in sorted(MAPA_VIGENCIAS)), "TOTAL"]:
    prefijo = f"PROYECTO_ACTIVIDAD_MAG{token}_"
    programado = pd.to_numeric(origen[prefijo + "PROGRAMADO"], errors="raise").astype(float)
    for medida, porcentaje in [("COMPROMISOS_CONTRATADO", "PORCCOMPROMISOS"), ("ENTREGADO", "PORCENTREGADO")]:
        magnitud = pd.to_numeric(origen[prefijo + medida], errors="raise").astype(float)
        publicado = pd.to_numeric(origen[prefijo + porcentaje], errors="raise").astype(float)
        comparables = programado.ne(0)
        referencia = magnitud.div(programado.where(comparables)) * 100
        diferentes = comparables & ~np.isclose(publicado, referencia, rtol=1e-9, atol=0.0051)
        contrastes_cociente.append({"vigencia_o_total": token, "medida": medida,
                                   "comparables": int(comparables.sum()), "diferencias": int(diferentes.sum())})
        for idx in origen.index[diferentes]:
            incidencias_cociente.append({**{c: str(origen.at[idx,c]) for c in CLAVE},
                "vigencia_o_total": token, "medida": medida,
                "programado_fuente": programado.at[idx], "magnitud_fuente": magnitud.at[idx],
                "porcentaje_publicado": publicado.at[idx], "cociente_referencia_pct": referencia.at[idx],
                "referencia_errata": origen.at[idx,"FE_DE_ERRATAS"] if "FE_DE_ERRATAS" in origen else "campo no publicado"})
contrastes_cociente = pd.DataFrame(contrastes_cociente)
incidencias_cociente = pd.DataFrame(incidencias_cociente)
display(contrastes_cociente)
if not incidencias_cociente.empty:
    display(incidencias_cociente.head(20))
print("Las diferencias son incidencias de diagnóstico; no reemplazo ningún porcentaje fuente.")


,vigencia_o_total,medida,comparables,diferencias
0,VIG1,COMPROMISOS_CONTRATADO,1373,1317
1,VIG1,ENTREGADO,1373,1
2,VIG2,COMPROMISOS_CONTRATADO,1481,0
3,VIG2,ENTREGADO,1481,0
4,VIG3,COMPROMISOS_CONTRATADO,1521,0
5,VIG3,ENTREGADO,1521,0
6,VIG4,COMPROMISOS_CONTRATADO,1485,0
7,VIG4,ENTREGADO,1485,0
8,TOTAL,COMPROMISOS_CONTRATADO,918,0
9,TOTAL,ENTREGADO,918,0


,ENTIDAD_CODIGO,PROYECTO_CODIGO,PROYECTO_ACTIVIDAD_CODIGO,vigencia_o_total,medida,programado_fuente,magnitud_fuente,porcentaje_publicado,cociente_referencia_pct,referencia_errata
0,0102,008199,1,VIG1,COMPROMISOS_CONTRATADO,1.00,1.00,0.0,100.000000,
1,0102,008199,2,VIG1,COMPROMISOS_CONTRATADO,100.00,100.00,0.0,100.000000,
2,0102,008199,3,VIG1,COMPROMISOS_CONTRATADO,100.00,100.00,0.0,100.000000,
3,0102,008199,4,VIG1,COMPROMISOS_CONTRATADO,100.00,100.00,0.0,100.000000,
4,0102,008199,5,VIG1,COMPROMISOS_CONTRATADO,1.00,1.00,0.0,100.000000,
5,0102,008199,9,VIG1,COMPROMISOS_CONTRATADO,100.00,100.00,0.0,100.000000,
6,0102,008201,1,VIG1,COMPROMISOS_CONTRATADO,3300.00,3381.00,0.0,102.454545,
7,0102,008201,2,VIG1,COMPROMISOS_CONTRATADO,1.00,1.00,0.0,100.000000,
8,0102,008201,3,VIG1,COMPROMISOS_CONTRATADO,0.15,0.15,0.0,100.000000,
9,0102,008201,5,VIG1,COMPROMISOS_CONTRATADO,0.15,0.15,0.0,100.000000,


Las diferencias son incidencias de diagnóstico; no reemplazo ningún porcentaje fuente.


### 2.2. Lectura de la exploración del corte seleccionado

Relaciono el perfil de calidad con las transformaciones. Los cambios de espacios y Unicode justifican normalizar representación, no reescribir descripciones. Los tipos de anualización determinan cómo interpreto las magnitudes; sus frecuencias se calculan desde el archivo actual.

Distingo porcentajes extremos, ceros y ausencias. Una referencia de errata vacía no prueba que el registro carezca de problemas; un valor elevado tampoco demuestra por sí solo una mezcla de escala.



### Síntesis calculada del perfil y las discrepancias

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [14]:
_anualizaciones_prosa = origen.PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION.map(texto_limpio).value_counts()
_tipos_prosa = "; ".join(f"{k}: {int(v)}" for k,v in _anualizaciones_prosa.items()) or "sin actividades"
_campos_texto_prosa = perfil.loc[perfil.textos_a_normalizar.gt(0)]
_celdas_texto_prosa = int(_campos_texto_prosa.textos_a_normalizar.sum())
_diferencias_prosa = int(contrastes_cociente.diferencias.sum())
display(Markdown(f"**Corte seleccionado: {FECHA.isoformat()}.** Exploro **{len(origen)} actividades** y **{len(origen.columns)} campos fuente**. Identifico **{int(origen.duplicated(CLAVE).sum())} claves repetidas**, **{int(perfil.vacios.sum())} celdas vacías** y **{_celdas_texto_prosa} celdas de texto candidatas a normalización** en **{len(_campos_texto_prosa)} campos**. La anualización se distribuye así: {_tipos_prosa}. El contraste porcentaje–magnitud registra **{_diferencias_prosa} diferencias** entre medidas; este conteo no representa actividades distintas ni correcciones aplicadas."))


**Corte seleccionado: 2026-06-30.** Exploro **1655 actividades** y **78 campos fuente**. Identifico **0 claves repetidas**, **1634 celdas vacías** y **1058 celdas de texto candidatas a normalización** en **5 campos**. La anualización se distribuye así: Suma: 927; Constante: 553; Creciente: 169; Decreciente: 6. El contraste porcentaje–magnitud registra **1318 diferencias** entre medidas; este conteo no representa actividades distintas ni correcciones aplicadas.

## 3. Transformaciones de representación y trazabilidad temporal

Normalizo espacios y Unicode, mantengo identificadores como texto y convierto medidas con comprobaciones estrictas. Conservo actividades, ceros, magnitudes, porcentajes y bases de precios; no borro filas por presentar incidencias.

Derivo `FECHA_CORTE` y `ESTADO_SEGUIMIENTO` conservando el texto fuente, y añado `CORTE_PUBLICACION`. Identifico los registros cuya fecha observada difiere del corte publicado, sin convertirlos en observaciones nuevas del trimestre.

Exijo coherencia entre vigencia y fecha observada y rechazo fechas posteriores al corte. Al comparar publicaciones necesito identidad de plan, actividad, definición y versión, además de una fecha nominal.



In [6]:
def identificador(v):
    if vacio(v):
        raise ValueError("Identificador vacío; revisar la fuente antes de exportar.")
    s = str(texto_limpio(v))
    if not re.fullmatch(r"\d+", s):
        raise ValueError(f"Identificador con formato inesperado: {v!r}")
    return s  # Conserva literalmente los ceros del texto de origen.

datos = origen.copy(deep=True)
for c in datos:
    datos[c] = datos[c].map(texto_limpio)
    if c.endswith("_CODIGO") or c == "PROYECTO_BPIN":
        datos[c] = datos[c].map(identificador).astype("string")
    elif c == "VIGENCIA":
        datos[c] = pd.to_numeric(datos[c], errors="raise").astype("Int64")
    elif c in NUMERICAS:
        if any(vacio(v) for v in datos[c]):
            raise ValueError(f"Aparecieron vacíos numéricos en {c}; definir tratamiento antes de continuar.")
        datos[c] = pd.to_numeric(datos[c], errors="raise")
        if not np.isfinite(datos[c]).all():
            raise ValueError(f"Valores no finitos en {c}")
    else:
        datos[c] = datos[c].astype("string")

longitudes = {"ENTIDAD_CODIGO": 4, "SECTOR_CODIGO": 3, "PLAN_NIVEL1_CODIGO": 2,
              "PLAN_NIVEL2_CODIGO": 2, "PROYECTO_CODIGO": 6}
for c, largo in longitudes.items():
    if not datos[c].str.len().eq(largo).all():
        raise ValueError(f"Revisar longitud de {c}; no se rellenan códigos automáticamente.")
tipos_validos = {"Suma", "Constante", "Creciente", "Decreciente"}
if not set(datos.PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION).issubset(tipos_validos):
    raise ValueError("Tipo de anualización no reconocido.")

partes = datos.VIGENCIA_FECHA_SEGUIMIENTO.str.extract(r"^(\d{2}/\d{2}/\d{4})\s*\(([^()]*)\)$")
if partes.isna().any().any():
    raise ValueError("Formato de fecha/estado inesperado.")
datos["FECHA_CORTE"] = pd.to_datetime(partes[0], format="%d/%m/%Y", errors="raise")
datos["ESTADO_SEGUIMIENTO"] = partes[1].astype("string")
datos["CORTE_PUBLICACION"] = pd.Timestamp(FECHA)
if (datos.FECHA_CORTE > pd.Timestamp(FECHA)).any() or not datos.VIGENCIA.eq(datos.FECHA_CORTE.dt.year).all():
    raise ValueError("Fechas futuras o discrepancia entre fecha fuente y vigencia.")
rezagados = datos.loc[datos.FECHA_CORTE.ne(pd.Timestamp(FECHA)), [*CLAVE, "VIGENCIA_FECHA_SEGUIMIENTO", "FECHA_CORTE", "CORTE_PUBLICACION"]]
print("Registros cuya fecha fuente difiere del corte de publicación:", len(rezagados))
if not rezagados.empty:
    display(rezagados)

if FECHA.year not in MAPA_VIGENCIAS.values():
    raise ValueError("El año del corte está fuera del mapa del plan; revisa su configuración.")

Registros cuya fecha fuente difiere del corte de publicación: 0


## 4. Cómo separo conversión, medición y diagnóstico temporal

Aplico las mismas reglas a todas las actividades, sin listas de claves o fechas que autoricen una corrección. Separo la conversión de unidades de cualquier hipótesis sobre la medición. Cada medida produce tres resultados:

| Sufijo | Qué contiene | Cómo lo utilizo |
|---|---|---|
| `_NORMALIZADO` | Porcentaje publicado dividido exactamente entre cien | Auditoría y lectura del dato oficial; puede conservar anomalías de origen |
| `_PARA_ANALISIS` | La misma proporción, solo cuando pasa los controles técnicos | Uso analítico condicionado a cobertura y estado; no es una estimación ni una corrección oficial |
| `_ESTADO_CALIDAD` | El motivo que habilita o impide el uso | Distingo impedimentos aritméticos o de aplicabilidad de alertas exploratorias que no excluyen por sí solas |

El total físico solo se habilita para Suma; para los otros tipos marco `TOTAL_NO_APLICABLE_ANUALIZACION`, aunque una edición futura rellene sus totales. Conservo siempre el total publicado.

Una proporción de 1 equivale a 100 %. No aplico min–max, no limito los resultados a uno y no invierto porcentajes por el solo hecho de que la anualización sea Decreciente. Los sobrecumplimientos que pasan los controles permanecen por encima de uno.

Compruebo denominador, valores finitos, fechas y concordancia entre porcentaje y magnitud con una tolerancia de 0,0051 puntos porcentuales y relativa de 1e-9. Una discrepancia impide habilitar la medida sin modificar el valor publicado. También señalo una posible mezcla de escala de un producto único. Es una pista textual, no una unidad certificada: esa alerta por sí sola no modifica ni excluye la proporción.

Para el contraste temporal exijo la misma identidad de plan, entidad, proyecto, actividad, vigencia, definición y tipo de anualización y una programación comparable. Excluyo el corte objetivo, los posteriores, las observaciones de la misma fecha o posteriores y las ediciones que el portal publicó después del corte evaluado. Una edición sin fecha de disponibilidad no actúa como antecedente.

Comparo como máximo tres antecedentes distintos y coherentes. Un salto grande compatible con un factor decimal 10 o 100 se marca para confirmar; no lo divido automáticamente ni lo uso por sí solo para dejar vacía la medida analítica. Una actividad puede concentrar su ejecución al cierre o realizar entregas discontinuas. Los umbrales son criterios de alerta, no probabilidades ni pruebas de que un dato sea falso. Si no tengo historia comparable, lo hago explícito y mantengo los controles del archivo actual.

No uso una observación ni una edición posterior para decidir sobre un corte anterior. La revisión retrospectiva puede ayudar a interpretar un caso, pero las decisiones ejecutables excluyen información futura. La referencia se carga desde este notebook y se comprueba su huella antes de utilizarla.


In [7]:
def huella_definicion(nombre, anualizacion):
    import hashlib
    import unicodedata
    texto = " ".join(unicodedata.normalize("NFC", str(nombre)).casefold().split())
    tipo = " ".join(str(anualizacion).casefold().split())
    return hashlib.sha256((texto + "|" + tipo).encode("utf-8")).hexdigest()


def identidad_actividad(fila):
    campos = ["PLAN_CODIGO", "ENTIDAD_CODIGO", "PROYECTO_CODIGO", "PROYECTO_ACTIVIDAD_CODIGO"]
    return tuple(str(fila[c]).strip().lstrip("0") or "0" for c in campos)


def reconocer_unidad(nombre):
    """Extraigo una pista textual; no la convierto en una unidad certificada."""
    import re
    import unicodedata
    texto = re.sub(r"^\s*\d+\s*-\s*", "", str(nombre))
    match = re.search(r"(?<!\d)(\d+(?:[.,]\d+)?)\s+([A-Za-z\u00c0-\u024f]+(?:\([A-Za-z]+\))?)", texto)
    if not match:
        return None, ""
    cantidad = float(match[1].replace(",", "."))
    unidad = "".join(c for c in unicodedata.normalize("NFKD", match[2].casefold()) if not unicodedata.combining(c))
    return cantidad, unidad


def cargar_referencia_historica(raiz):
    """La referencia pública está dentro de este notebook; no descargo el histórico al limpiar."""
    import json
    import hashlib
    import pandas as pd
    from pathlib import Path
    contenido = json.loads((Path(raiz) / "03_limpieza_gestion_inversion.ipynb").read_text(encoding="utf-8"))
    referencia = contenido.get("metadata", {}).get("referencia_temporal_gestion")
    if referencia is None:
        raise ValueError("Falta la referencia temporal auditada del notebook.")
    serializado = json.dumps(referencia["filas"], ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    if hashlib.sha256(serializado.encode("utf-8")).hexdigest() != referencia["sha256_filas"]:
        raise ValueError("La referencia temporal no conserva su huella de integridad.")
    tabla = pd.DataFrame(referencia["filas"], columns=referencia["columnas"])
    fechas_edicion = {s["corte"]:s.get("fecha_publicacion_portal") for s in referencia["fuentes"]}
    tabla["disponible_desde"] = tabla.publicacion.map(fechas_edicion)
    return tabla, referencia["fuentes"]


def preparar_indice_temporal(tabla, corte_publicacion):
    """No uso la publicación objetivo ni información posterior para decidir sobre ella."""
    import pandas as pd
    indice = {}
    corte = pd.Timestamp(corte_publicacion)
    disponibilidad = pd.to_datetime(tabla["disponible_desde"],errors="coerce") if "disponible_desde" in tabla else pd.Series(pd.NaT,index=tabla.index)
    anteriores = tabla.loc[(pd.to_datetime(tabla.publicacion) < corte) & (disponibilidad <= corte)].copy()
    for fila in anteriores.itertuples(index=False):
        key = (fila.plan, fila.entidad, fila.proyecto, fila.actividad, int(fila.anio), fila.definicion)
        indice.setdefault(key, []).append(fila)
    return indice


def evaluar_medicion(fila, token, medida, anio, corte_publicacion, indice):
    """Separo conversión exacta, elegibilidad técnica y alertas; nunca invento una corrección."""
    import math
    import numpy as np
    import pandas as pd
    prefijo = f"PROYECTO_ACTIVIDAD_MAG{token}_"
    sufijo = "PORCCOMPROMISOS" if medida == "COMPROMISOS_CONTRATADO" else "PORCENTREGADO"
    p = float(fila[prefijo + "PROGRAMADO"])
    m = float(fila[prefijo + medida])
    q = float(fila[prefijo + sufijo])
    if not all(math.isfinite(v) for v in (p,m,q)):
        raise ValueError("No puedo normalizar una medida no finita.")
    publicado = q / 100
    razones = []
    alertas = []
    tipo = " ".join(str(fila["PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION"]).casefold().split())
    if tipo not in {"suma", "constante", "creciente", "decreciente"}:
        raise ValueError("Tipo de anualización no reconocido: revisar su definición antes de medir.")
    if token == "TOTAL" and tipo != "suma":
        razones.append("TOTAL_NO_APLICABLE_ANUALIZACION")
    observacion = fila.get("FECHA_CORTE")
    if observacion is None:
        observacion = pd.to_datetime(str(fila["VIGENCIA_FECHA_SEGUIMIENTO"]).split(" ")[0],format="%d/%m/%Y")
    observacion = pd.Timestamp(observacion)
    if observacion > pd.Timestamp(corte_publicacion):
        raise ValueError("La fecha observada es posterior a la publicación evaluada.")
    if anio is not None and anio > observacion.year:
        razones.append("VIGENCIA_POSTERIOR_AL_DATO")
    if p == 0:
        razones.append("SIN_DENOMINADOR")
        if m != 0:
            razones.append("MAGNITUD_SIN_PROGRAMACION")
    elif p < 0 or m < 0 or q < 0:
        razones.append("VALOR_NEGATIVO_POR_REVISAR")
    elif not np.isclose(q, m/p*100, rtol=1e-9, atol=0.0051):
        razones.append("DISCREPANCIA_PORCENTAJE_MAGNITUD")

    cantidad, unidad = reconocer_unidad(fila["PROYECTO_ACTIVIDAD_NOMBRE"])
    unidad_porcentual = unidad.startswith(("porcient", "porcent", "tasa"))
    if cantidad == 1 and p == 1 and m > 1 and not unidad_porcentual:
        alertas.append("ALERTA_ESCALA_PRODUCTO_UNICO_POR_CONFIRMAR")

    definicion = huella_definicion(fila["PROYECTO_ACTIVIDAD_NOMBRE"], fila["PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION"])
    key = (*identidad_actividad(fila), anio, definicion)
    referentes = []
    if anio is not None and p > 0:
        # Una publicación no es necesariamente una observación independiente: deduplico su fecha fuente.
        por_fecha = {}
        for r in sorted(indice.get(key, []), key=lambda r:r.publicacion):
            if pd.Timestamp(r.observacion) >= observacion:
                continue
            rm = r.comprometido if medida == "COMPROMISOS_CONTRATADO" else r.entregado
            rq = r.pct_compromiso if medida == "COMPROMISOS_CONTRATADO" else r.pct_entrega
            if not np.isclose(r.programado,p,rtol=1e-9,atol=1e-9) or r.programado <= 0 or rm <= 0:
                continue
            if not np.isclose(rq,rm/r.programado*100,rtol=1e-9,atol=0.0051):
                continue
            if cantidad == 1 and p == 1 and rm > 1 and not unidad_porcentual:
                continue
            por_fecha[r.observacion] = rm/r.programado
        referentes = list(por_fecha.values())[-3:]
        if referentes and m/p > 1:
            base = float(np.median(referentes))
            # Es una alerta decimal, no una estimación del porcentaje verdadero ni una corrección.
            if base > 0 and m/p >= 8*base and any(0.5*base <= m/p/factor <= 2*base for factor in (10,100)):
                alertas.append("ALERTA_SALTO_DECIMAL_TEMPORAL_POR_CONFIRMAR")

    if razones:
        return publicado, float("nan"), ";".join(dict.fromkeys(razones + alertas)), len(referentes)
    estado = "COHERENTE_CON_FUENTE" if referentes else "COHERENTE_SIN_REFERENCIA_TEMPORAL"
    if publicado > 1:
        estado += ";SOBRECUMPLIMIENTO_SIN_RECORTAR"
    if alertas:
        estado += ";" + ";".join(alertas)
    return publicado, publicado, estado, len(referentes)


def preparar_mediciones(datos, mapa, corte_publicacion, referencia):
    """Todas las actividades pasan por las mismas reglas; no hay listas de claves o fechas autorizadas."""
    import pandas as pd
    indice = preparar_indice_temporal(referencia, corte_publicacion)
    columnas = {}
    resumen = []
    detalle = []
    for token, anio in [*((f"VIG{i}",a) for i,a in sorted(mapa.items())), ("TOTAL",None)]:
        prefijo = f"PROYECTO_ACTIVIDAD_MAG{token}_"
        controles = [[] for _ in range(len(datos))]
        for medida, sufijo in [("COMPROMISOS_CONTRATADO","PORCCOMPROMISOS"),("ENTREGADO","PORCENTREGADO")]:
            evaluaciones = [evaluar_medicion(row,token,medida,anio,corte_publicacion,indice) for _,row in datos.iterrows()]
            normalizado = pd.Series([r[0] for r in evaluaciones],index=datos.index,dtype="float64")
            analisis = pd.Series([r[1] for r in evaluaciones],index=datos.index,dtype="float64")
            estado = pd.Series([r[2] for r in evaluaciones],index=datos.index,dtype="string")
            columnas[prefijo+sufijo+"_NORMALIZADO"] = normalizado
            columnas[prefijo+sufijo+"_PARA_ANALISIS"] = analisis
            columnas[prefijo+sufijo+"_ESTADO_CALIDAD"] = estado
            revision = analisis.isna() & estado.str.contains("POR_CONFIRMAR|DISCREPANCIA|NEGATIVO|MAGNITUD_SIN_PROGRAMACION",regex=True)
            no_aplica = analisis.isna() & ~revision
            resumen.append({"vigencia_o_total":token,"medida":medida,"observaciones":len(datos),
                            "habilitadas_analisis":int(analisis.notna().sum()),"pendientes_revision":int(revision.sum()),
                            "sin_medicion_aplicable":int(no_aplica.sum()),
                            "con_referencia_temporal":sum(r[3]>0 for r in evaluaciones)})
            if int(analisis.notna().sum()+revision.sum()+no_aplica.sum()) != len(datos):
                raise AssertionError("El resumen de calidad no conserva el número de observaciones.")
            for pos,((idx,row),r) in enumerate(zip(datos.iterrows(),evaluaciones)):
                if pd.isna(r[1]) or "ALERTA_" in r[2]:
                    controles[pos].append(("compromiso" if medida=="COMPROMISOS_CONTRATADO" else "entrega")+"="+r[2])
                    detalle.append({**{c:row[c] for c in ["PLAN_CODIGO","ENTIDAD_CODIGO","PROYECTO_CODIGO","PROYECTO_ACTIVIDAD_CODIGO"]},
                                    "vigencia_o_total":token,"medida":medida,"estado":r[2],
                                    "publicado_normalizado":r[0],"referencias_anteriores":r[3]})
        columnas[f"CONTROL_MEDICION_MAG{token}"] = pd.Series([" | ".join(r) if r else "COHERENTE" for r in controles],index=datos.index,dtype="string")
    return pd.DataFrame(columnas,index=datos.index), pd.DataFrame(resumen), pd.DataFrame(detalle)


def verificar_mediciones(datos, medidas, mapa):
    """Compruebo invariantes numéricos y de selección, independientes de una lista de ejemplos."""
    import numpy as np
    import pandas as pd
    for token in [*(f"VIG{i}" for i in sorted(mapa)),"TOTAL"]:
        pre=f"PROYECTO_ACTIVIDAD_MAG{token}_"
        for sufijo in ["PORCCOMPROMISOS","PORCENTREGADO"]:
            publicado=medidas[pre+sufijo+"_NORMALIZADO"]
            analisis=medidas[pre+sufijo+"_PARA_ANALISIS"]
            estado=medidas[pre+sufijo+"_ESTADO_CALIDAD"]
            np.testing.assert_array_equal(publicado.to_numpy(),datos[pre+sufijo].astype(float).to_numpy()/100)
            habilitado=estado.str.startswith("COHERENTE")
            np.testing.assert_array_equal(analisis.notna().to_numpy(dtype=bool),habilitado.to_numpy(dtype=bool))
            pd.testing.assert_series_equal(analisis.loc[habilitado],publicado.loc[habilitado],check_names=False,check_exact=True)


def razon_giros(datos, prefijo, anio):
    """Relaciono recursos de la misma vigencia y base; no mido años todavía no observados."""
    import numpy as np
    import pandas as pd
    programado=datos[prefijo+"PROGRAMADO"].astype(float)
    giros=datos[prefijo+"GIROS"].astype(float)
    comprometido=datos[prefijo+"COMPROMISOS_CONTRATADO"].astype(float)
    if not np.isfinite(pd.concat([programado,giros,comprometido])).all():
        raise ValueError("Recursos no finitos: no publico la razón.")
    futuro=datos.FECHA_CORTE.dt.year.lt(anio)
    negativos=programado.lt(0)|giros.lt(0)|comprometido.lt(0)
    sin_denominador=programado.eq(0)
    valido=~futuro & ~negativos & ~sin_denominador
    resultado=giros.div(programado.where(valido))
    estado=pd.Series("COHERENTE_ARITMETICO",index=datos.index,dtype="string")
    estado.loc[sin_denominador]="SIN_DENOMINADOR"
    estado.loc[negativos]="VALOR_NEGATIVO_POR_REVISAR"
    estado.loc[futuro]="VIGENCIA_POSTERIOR_AL_DATO"
    estado.loc[valido & giros.gt(comprometido)] += ";GIROS_MAYORES_COMPROMISOS_REPORTADOS"
    estado.loc[valido & giros.gt(programado)] += ";SOBREPROGRAMACION_SIN_RECORTAR"
    np.testing.assert_array_equal(resultado.notna().to_numpy(),valido.to_numpy())
    return resultado,estado

referencia_temporal, fuentes_referencia = cargar_referencia_historica(RAIZ)
mediciones, resumen_mediciones, incidencias_medicion = preparar_mediciones(datos, MAPA_VIGENCIAS, FECHA, referencia_temporal)
print("Registros de referencia:", len(referencia_temporal), "(solo se utilizan publicaciones anteriores al corte evaluado)")
display(resumen_mediciones)


Registros de referencia: 13084 (solo se utilizan publicaciones anteriores al corte evaluado)


,vigencia_o_total,medida,observaciones,habilitadas_analisis,pendientes_revision,sin_medicion_aplicable,con_referencia_temporal
0,VIG1,COMPROMISOS_CONTRATADO,1655,56,1331,268,1093
1,VIG1,ENTREGADO,1655,1372,4,279,1098
2,VIG2,COMPROMISOS_CONTRATADO,1655,1481,8,166,1284
3,VIG2,ENTREGADO,1655,1481,2,172,1279
4,VIG3,COMPROMISOS_CONTRATADO,1655,1521,0,134,1226
5,VIG3,ENTREGADO,1655,1521,1,133,1164
6,VIG4,COMPROMISOS_CONTRATADO,1655,0,0,1655,0
7,VIG4,ENTREGADO,1655,0,0,1655,0
8,TOTAL,COMPROMISOS_CONTRATADO,1655,918,0,737,0
9,TOTAL,ENTREGADO,1655,918,0,737,0


### 4.1. Anualización: interpretación y límites de agregación

Antes de agregar distingo los tipos presentes en el archivo seleccionado. Su frecuencia y cobertura pueden cambiar; no traslado a todo el conjunto la ausencia de un tipo en un sector específico. Contrasto esta interpretación con el diccionario de la fuente, el [manual de programación M-IN-004 de la SDP, versión 9](https://www.sdp.gov.co/sites/default/files/05_1410_manualprogramacion_plan_de_accion-v9.pdf) y el [manual de reprogramación M-IN-006, versión 9](https://www.sdp.gov.co/sites/default/files/10_1410_manual_reprogr_actuliz_plan_accion-v9.pdf). Estas versiones sustentan las reglas implementadas, sin presumir que un instrumento futuro mantenga las mismas definiciones.

| Tipo | Interpretación | Límite de la limpieza |
|---|---|---|
| Suma | Magnitudes anuales aditivas si conservan unidad, definición y cobertura | No sumo porcentajes con denominadores distintos ni instantáneas trimestrales del mismo año |
| Constante | Cada vigencia repite un estándar frente a su programación | No agrego cumplimientos de años anteriores al del año observado |
| Creciente | Niveles acumulados que incorporan lo alcanzado previamente | No sumo niveles; solo interpreto diferencias como incrementos si los registros son comparables |
| Decreciente | Niveles orientados a reducirse, sin agregación entre vigencias | No invierto el porcentaje ni uso automáticamente 1 − p; un indicador de reducción requiere línea base, meta y fórmula |

Uso ejemplos exclusivamente ilustrativos: en Suma, entregas de 15, 35 y 6,27 frente a una meta total de 100 representan 56,27 % acumulado. En Constante, cuatro programaciones de 100 % no conforman una meta de 400 %. En Creciente, una trayectoria 15, 51, 87 y 120 se orienta a 120; no sumo esos niveles. Estas cifras explican las reglas y no describen el corte seleccionado.

El manual M-IN-004 contempla que la programación Constante del primer año pueda ser inferior por corresponder a un semestre. No impongo igualdad ciega entre todos los años. También distingo las reglas de programación de la evolución efectiva de entrega y compromiso.

Los cortes son instantáneas y pueden revisar años anteriores. Una diferencia entre archivos no equivale necesariamente a ejecución del trimestre. Programación, definición, unidad, cobertura o erratas pueden romper la comparabilidad. No exijo crecimiento lineal ni monotonicidad como prueba automática de verdad.

Conservo el porcentaje oficial y lo divido entre cien, sin reconstruirlo desde una trayectoria supuesta. La conversión es igual para todos los tipos porque cambia representación, no anualización: 38,98 % se expresa como 0,3898. Un sobrecumplimiento puede superar uno. En Decreciente, coherencia entre magnitud y programación no demuestra por sí sola logro de reducción.

Mantengo separados compromiso/contratado, entrega física y giros financieros. Una escala proporcional común no autoriza una media simple entre actividades heterogéneas ni constituye automáticamente un indicador sectorial.

En la siguiente celda comparo totales físicos con vigencias únicamente para Suma. Una diferencia exige examinar componentes, versiones y erratas: no determina cuál cifra sería correcta ni autoriza sobrescribir el total publicado. Un cociente coherente con su numerador y denominador todavía puede coexistir con componentes que no concilian. Los resultados de este contraste se calculan para cada archivo.



In [8]:
# Diagnóstico por anualización: agrego magnitudes entre vigencias solo para Suma.
filas_anualizacion = []
for tipo, grupo in datos.groupby("PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION", sort=True):
    registro = {"anualizacion": tipo, "actividades": len(grupo)}
    for medida in ["PROGRAMADO", "COMPROMISOS_CONTRATADO", "ENTREGADO"]:
        total = grupo[f"PROYECTO_ACTIVIDAD_MAGTOTAL_{medida}"].astype(float)
        registro[medida + "_totales_cero"] = int(total.eq(0).sum())
        if str(tipo).casefold() == "suma":
            suma = grupo[[f"PROYECTO_ACTIVIDAD_MAGVIG{i}_{medida}" for i in sorted(MAPA_VIGENCIAS)]].sum(axis=1)
            # Admito como diagnóstico la acumulación máxima del redondeo a dos decimales.
            registro[medida + "_diferencias_total"] = int((~np.isclose(total, suma, rtol=1e-9, atol=0.0051 * (len(MAPA_VIGENCIAS)+1))).sum())
        else:
            registro[medida + "_diferencias_total"] = pd.NA  # No aplico una suma sin significado.
    filas_anualizacion.append(registro)
diagnostico_anualizacion = pd.DataFrame(filas_anualizacion)
display(diagnostico_anualizacion)
# El diagnóstico no modifica ninguna magnitud ni genera columnas nuevas en la salida.


,anualizacion,actividades,PROGRAMADO_totales_cero,PROGRAMADO_diferencias_total,COMPROMISOS_CONTRATADO_totales_cero,COMPROMISOS_CONTRATADO_diferencias_total,ENTREGADO_totales_cero,ENTREGADO_diferencias_total
0,Constante,553,553,<NA>,553,<NA>,553,<NA>
1,Creciente,169,169,<NA>,169,<NA>,169,<NA>
2,Decreciente,6,6,<NA>,6,<NA>,6,<NA>
3,Suma,927,9,1,89,6,95,6


### Lectura calculada de los totales y las fechas observadas

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [15]:
_suma_prosa = diagnostico_anualizacion.loc[diagnostico_anualizacion.anualizacion.str.casefold().eq("suma")]
if _suma_prosa.empty:
    _totales_prosa = "No encuentro actividades Suma; no aplico un contraste aditivo a los otros tipos."
else:
    _r_suma = _suma_prosa.iloc[0]
    _totales_prosa = "; ".join(f"{_m}: {int(_r_suma[_m + '_diferencias_total'])} diferencias" for _m in ["PROGRAMADO", "COMPROMISOS_CONTRATADO", "ENTREGADO"])
display(Markdown(f"Para **{FECHA.isoformat()}**, el contraste de totales frente a vigencias en Suma muestra: {_totales_prosa}. No reconstruyo los totales publicados desde esas sumas. Identifico **{len(rezagados)} registros con fecha observada distinta del corte de publicación**. Las diferencias requieren consultar componentes, versiones y erratas; no determinan por sí solas qué cifra sería correcta."))


Para **2026-06-30**, el contraste de totales frente a vigencias en Suma muestra: PROGRAMADO: 1 diferencias; COMPROMISOS_CONTRATADO: 6 diferencias; ENTREGADO: 6 diferencias. No reconstruyo los totales publicados desde esas sumas. Identifico **0 registros con fecha observada distinta del corte de publicación**. Las diferencias requieren consultar componentes, versiones y erratas; no determinan por sí solas qué cifra sería correcta.

## 5. Cómo compruebo la cobertura de la medición

En el siguiente bloque separo medidas habilitadas, pendientes de revisión y no aplicables. No considero un año futuro o una actividad sin denominador como si tuviera desempeño cero. Un cero observado con programación positiva sí puede ser un resultado válido y se conserva.

Examino mínimos, máximos y valores por encima de uno tanto en la conversión publicada como en la selección para análisis. De esta forma puedo distinguir una anomalía conservada en la fuente de un valor que el procedimiento permite utilizar. No elimino actividades para mejorar los conteos de calidad.

Una alerta `ALERTA_` puede acompañar un valor habilitado: debo examinarla antes de una decisión sustantiva. No he calibrado sensibilidad ni especificidad de esos umbrales; por eso no los trato como un detector probado de errores.

El valor de `_PARA_ANALISIS` vacío necesita interpretarse con su estado: no es equivalente a incumplimiento y no debe rellenarse automáticamente con cero. Al analizar las bases debo informar la cobertura y revisar si las incidencias se concentran en alguna entidad, anualización o clase de actividad.
Desagrego también la cobertura de la vigencia activa por anualización. Cuento las alertas exploratorias por separado: pueden acompañar una proporción disponible y no equivalen a discrepancias aritméticas ni a observaciones excluidas.



In [9]:
# Distingo la cobertura de la medición de la ausencia de programación y de las incidencias.
numero_actual = next(i for i,a in MAPA_VIGENCIAS.items() if a == FECHA.year)
display(resumen_mediciones.loc[resumen_mediciones.vigencia_o_total.eq(f"VIG{numero_actual}")])
perfil_resultados = []
for campo in mediciones:
    if campo.endswith(("_NORMALIZADO", "_PARA_ANALISIS")):
        serie = mediciones[campo]
        perfil_resultados.append({"campo":campo,"disponibles":int(serie.notna().sum()),
                                  "minimo":serie.min(),"maximo":serie.max(),"mayores_1":int(serie.gt(1).sum())})
display(pd.DataFrame(perfil_resultados))

# Examino cobertura y alertas sin mezclar anualizaciones ni convertir faltantes en ceros.
cobertura_por_tipo = []
for medida in ["PORCCOMPROMISOS", "PORCENTREGADO"]:
    pre = f"PROYECTO_ACTIVIDAD_MAGVIG{numero_actual}_{medida}"
    for tipo, posiciones in datos.groupby("PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION").groups.items():
        estados = mediciones.loc[posiciones, pre + "_ESTADO_CALIDAD"]
        valores = mediciones.loc[posiciones, pre + "_PARA_ANALISIS"]
        cobertura_por_tipo.append({"anualizacion": tipo, "medida": medida,
            "actividades": len(posiciones), "proporcion_disponible": int(valores.notna().sum()),
            "sin_proporcion_habilitada": int(valores.isna().sum()),
            "con_alerta_exploratoria": int(estados.str.contains("ALERTA_", regex=False).sum())})
display(pd.DataFrame(cobertura_por_tipo))


,vigencia_o_total,medida,observaciones,habilitadas_analisis,pendientes_revision,sin_medicion_aplicable,con_referencia_temporal
4,VIG3,COMPROMISOS_CONTRATADO,1655,1521,0,134,1226
5,VIG3,ENTREGADO,1655,1521,1,133,1164


,campo,disponibles,minimo,maximo,mayores_1
0,PROYECTO_ACTIVIDAD_MAGVIG1_PORCCOMPROMISOS_NOR...,1655,0.0,1.331126,1
1,PROYECTO_ACTIVIDAD_MAGVIG1_PORCCOMPROMISOS_PAR...,56,0.0,1.331126,1
2,PROYECTO_ACTIVIDAD_MAGVIG1_PORCENTREGADO_NORMA...,1655,0.0,100.000000,67
3,PROYECTO_ACTIVIDAD_MAGVIG1_PORCENTREGADO_PARA_...,1372,0.0,100.000000,66
4,PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS_NOR...,1655,0.0,130.500000,83
5,PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS_PAR...,1481,0.0,130.500000,83
6,PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO_NORMA...,1655,0.0,100.000000,48
7,PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO_PARA_...,1481,0.0,100.000000,48
8,PROYECTO_ACTIVIDAD_MAGVIG3_PORCCOMPROMISOS_NOR...,1655,0.0,63.700000,31
9,PROYECTO_ACTIVIDAD_MAGVIG3_PORCCOMPROMISOS_PAR...,1521,0.0,63.700000,31


,anualizacion,medida,actividades,proporcion_disponible,sin_proporcion_habilitada,con_alerta_exploratoria
0,Constante,PORCCOMPROMISOS,553,530,23,3
1,Creciente,PORCCOMPROMISOS,169,160,9,0
2,Decreciente,PORCCOMPROMISOS,6,6,0,0
3,Suma,PORCCOMPROMISOS,927,825,102,5
4,Constante,PORCENTREGADO,553,530,23,2
5,Creciente,PORCENTREGADO,169,160,9,0
6,Decreciente,PORCENTREGADO,6,6,0,0
7,Suma,PORCENTREGADO,927,825,102,3


### Cobertura calculada de la vigencia correspondiente al corte

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [16]:
_numero_informe = next(i for i,a in MAPA_VIGENCIAS.items() if a == FECHA.year)
_actual_informe = resumen_mediciones.loc[resumen_mediciones.vigencia_o_total.eq(f"VIG{_numero_informe}")]
_lineas_cobertura = [f"- **{r.medida}**: {r.habilitadas_analisis} medidas habilitadas, {r.pendientes_revision} pendientes de revisión y {r.sin_medicion_aplicable} sin medición aplicable." for r in _actual_informe.itertuples(index=False)]
display(Markdown(f"**Corte {FECHA.isoformat()}, vigencia {FECHA.year}.**\n\n" + "\n".join(_lineas_cobertura) + "\n\nEstos grupos describen la selección técnica; una medida sin denominador o de una vigencia posterior no es desempeño cero. Examino las alertas exploratorias por separado y no las interpreto como errores demostrados."))


**Corte 2026-06-30, vigencia 2026.**

- **COMPROMISOS_CONTRATADO**: 1521 medidas habilitadas, 0 pendientes de revisión y 134 sin medición aplicable.
- **ENTREGADO**: 1521 medidas habilitadas, 1 pendientes de revisión y 133 sin medición aplicable.

Estos grupos describen la selección técnica; una medida sin denominador o de una vigencia posterior no es desempeño cero. Examino las alertas exploratorias por separado y no las interpreto como errores demostrados.

### 5.1. Cómo leo los estados y las incidencias

La tabla siguiente no enumera correcciones aplicadas. Muestra restricciones de aplicabilidad, discrepancias y alertas exploratorias. Una alerta por sí sola no implica que la medida esté excluida. En `CONTROL_MEDICION_MAGVIG*` y `CONTROL_MEDICION_MAGTOTAL` reúno los motivos de compromiso y entrega, mientras que cada porcentaje conserva su propio estado.

Distingo, entre otros, `SIN_DENOMINADOR`, `VIGENCIA_POSTERIOR_AL_DATO`, `DISCREPANCIA_PORCENTAJE_MAGNITUD`, `ALERTA_ESCALA_PRODUCTO_UNICO_POR_CONFIRMAR` y `ALERTA_SALTO_DECIMAL_TEMPORAL_POR_CONFIRMAR`. Un registro rezagado se evalúa contra su fecha observada: no lo convierto en medición del año de la publicación.

Limito la vista a 30 casos para facilitar la lectura. Los motivos completos permanecen en la tabla en memoria y en los estados exportados. Una etiqueta COHERENTE indica que pasó estos controles técnicos, no que certifique la medición institucional.


In [10]:
# Las incidencias no son correcciones: el original siempre permanece disponible.
if not incidencias_medicion.empty:
    display(incidencias_medicion.groupby(["vigencia_o_total","medida","estado"]).size().rename("medidas").reset_index())
    display(incidencias_medicion.head(30))
else:
    print("No se detectaron incidencias o medidas no aplicables en este corte.")


,vigencia_o_total,medida,estado,medidas
0,TOTAL,COMPROMISOS_CONTRATADO,COHERENTE_SIN_REFERENCIA_TEMPORAL;SOBRECUMPLIM...,3
1,TOTAL,COMPROMISOS_CONTRATADO,SIN_DENOMINADOR,9
2,TOTAL,COMPROMISOS_CONTRATADO,TOTAL_NO_APLICABLE_ANUALIZACION;SIN_DENOMINADOR,728
3,TOTAL,ENTREGADO,SIN_DENOMINADOR,9
4,TOTAL,ENTREGADO,TOTAL_NO_APLICABLE_ANUALIZACION;SIN_DENOMINADOR,728
5,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,1314
6,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD;ALERTA_ESCALA...,3
7,VIG1,COMPROMISOS_CONTRATADO,SIN_DENOMINADOR,268
8,VIG1,COMPROMISOS_CONTRATADO,SIN_DENOMINADOR;MAGNITUD_SIN_PROGRAMACION,14
9,VIG1,ENTREGADO,COHERENTE_SIN_REFERENCIA_TEMPORAL;SOBRECUMPLIM...,3


,PLAN_CODIGO,ENTIDAD_CODIGO,PROYECTO_CODIGO,PROYECTO_ACTIVIDAD_CODIGO,vigencia_o_total,medida,estado,publicado_normalizado,referencias_anteriores
0,17,0102,008199,1,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,0.0,0
1,17,0102,008199,2,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,0.0,0
2,17,0102,008199,3,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,0.0,1
3,17,0102,008199,4,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,0.0,1
4,17,0102,008199,5,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,0.0,1
5,17,0102,008199,6,VIG1,COMPROMISOS_CONTRATADO,SIN_DENOMINADOR,0.0,0
6,17,0102,008199,7,VIG1,COMPROMISOS_CONTRATADO,SIN_DENOMINADOR,0.0,0
7,17,0102,008199,8,VIG1,COMPROMISOS_CONTRATADO,SIN_DENOMINADOR,0.0,0
8,17,0102,008199,9,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,0.0,1
9,17,0102,008201,1,VIG1,COMPROMISOS_CONTRATADO,DISCREPANCIA_PORCENTAJE_MAGNITUD,0.0,0


### 5.2. Cómo utilizo el contraste temporal

La trayectoria puede sugerir una mezcla de escala, pero no siempre identifica una corrección única. Conservo el porcentaje publicado y su conversión exacta; las alertas exploratorias no imputan un valor esperado ni excluyen por sí solas una ejecución discontinua.

Un cambio de descripción, anualización, programación o disponibilidad de la edición puede impedir el enlace con un antecedente. Si no existe historia comparable lo declaro, sin fabricar una curva de avance ni asumir crecimiento lineal.

Los ejemplos de publicaciones anteriores están en el anexo histórico. Para este corte interpreto los estados y conteos calculados en las tablas anteriores.



## 6. Qué verifico antes de consolidar

Compruebo que las filas y los campos fuente se conserven y que no aparezcan claves duplicadas. Para todas las vigencias y el total exijo que `_NORMALIZADO` sea exactamente el porcentaje fuente dividido entre cien. Cuando habilito `_PARA_ANALISIS`, exijo que sea exactamente esa misma proporción; no admito valores estimados intermedios.

También verifico que la ausencia de un valor analítico corresponda a su estado. Conservo ceros observados y sobrecumplimientos, sin confundirlos con valores no aplicables. Las comprobaciones de fuente, cálculo y elegibilidad son separadas y reproducibles.

La huella del Excel original debe permanecer igual. Si detecto un cambio estructural, un dato no finito o una contradicción que impide identificar la fuente, detengo la etapa antes de publicar una salida inválida. Una detención controlada protege el resultado; no prometo que una fuente futura desconocida nunca cambie.


In [11]:
# Verificación integral de conservación y normalización, medida por medida.
assert len(datos) == len(origen) and not datos.duplicated(CLAVE).any()
for c in origen:
    if c in NUMERICAS:
        np.testing.assert_array_equal(datos[c].to_numpy(), pd.to_numeric(origen[c]).to_numpy())
    elif c.endswith("_CODIGO") or c == "PROYECTO_BPIN":
        assert datos[c].eq(origen[c].map(identificador)).all()
    else:
        assert datos[c].astype(str).eq(origen[c].map(texto_limpio).astype(str)).all()
verificar_mediciones(datos, mediciones, MAPA_VIGENCIAS)
assert hashlib.sha256(FUENTE.read_bytes()).hexdigest() == HUELLA
print(f"Verificadas {len(datos)} filas y {len(origen.columns)} columnas originales; dos porcentajes por cada vigencia.")


Verificadas 1655 filas y 78 columnas originales; dos porcentajes por cada vigencia.


## 7. Cómo construyo y publico la salida del último corte

Conservo las actividades de la instantánea seleccionada, sus campos fuente, fecha observada y corte de publicación. El historial de referencia no añade filas a esta base. Mantengo las vigencias en columnas y no concateno magnitudes físicas con recursos como si fueran observaciones homogéneas.

Para giros/programado utilizo numerador y denominador de la misma vigencia y base de precios. Calculo la razón cuando hay programación positiva, recursos no negativos y un año ya observado. Con programación positiva y giros cero, el resultado es cero. Sin denominador, con negativos o en una vigencia posterior al dato, queda vacío y `CONTROL_RECURSOS_*` explica por qué. Informo sobreprogramación u orden presupuestal sin recortar la razón ni modificar importes oficiales.

Mantengo REC y RECK separados y conservo los totales publicados. No reemplazo un total por la suma de vigencias ni interpreto una conversión de escala como una regla válida de agregación entre actividades.

Genero un diccionario procesado con las definiciones de todos los campos y copio aparte la hoja oficial con sus textos, colores y celdas combinadas intactos. Preparo el CSV y el Excel en archivos temporales y verifico ambos antes de reemplazar las salidas activas. Si falla el segundo reemplazo, intento restaurar el primero desde su respaldo temporal para evitar una pareja de versiones mezcladas.

Releo el CSV y comparo sus valores exactamente. En Excel compruebo identificadores y textos exactos y tolero únicamente la representación numérica finita del formato: tolerancia relativa 1e-14 y absoluta 1e-12. El CSV es la referencia de precisión para intercambio. Los temporales y respaldos se retiran cuando la publicación termina correctamente.


### Funciones auxiliares de exportación

En el bloque siguiente reúno las funciones que utilizo para escribir Excel. Mantengo identificadores como texto, cadenas literales y el contenido completo de las celdas. En Gestión-Inversión compruebo además que la copia nativa del diccionario conserve su presentación y sus valores. Ubico estas funciones después del análisis para facilitar la lectura metodológica.


In [12]:
from collections import OrderedDict
from datetime import date, datetime
from pathlib import Path
from tempfile import NamedTemporaryFile
import math
import os
import textwrap

def _valor_excel(valor):
    import pandas as pd
    if valor is None or pd.isna(valor):
        return None
    if hasattr(valor, "item"):
        valor = valor.item()
    if isinstance(valor, (datetime, date)):
        return valor.strftime("%Y-%m-%d")
    if isinstance(valor, float) and not math.isfinite(valor):
        raise ValueError("No se exportan números infinitos a Excel.")
    if isinstance(valor, str) and len(valor) > 32767:
        raise ValueError("Una celda supera el límite de 32.767 caracteres de Excel; no se trunca el texto.")
    return valor

def _lineas(texto, ancho):
    partes = str(texto or "").splitlines() or [""]
    return sum(max(1, len(textwrap.wrap(p, width=ancho))) for p in partes)

def exportar_excel(ruta, hojas, *, plantilla_diccionario=None):
    """Exporta DataFrames o listas de filas con openpyxl (Windows/macOS/Linux).

    Los identificadores permanecen como texto y las cadenas nunca se interpretan
    como fórmulas. No trunca valores para ajustarlos a una altura visible de fila.
    """
    from openpyxl import Workbook, load_workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.table import Table, TableStyleInfo

    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    if plantilla_diccionario is None:
        libro = Workbook()
        libro.remove(libro.active)
    else:
        # Trabajar sobre una copia en memoria del libro fuente conserva el
        # diccionario nativo, sus estilos, colores de tema y celdas combinadas.
        libro = load_workbook(plantilla_diccionario, data_only=False)
        nombres = [n for n in libro.sheetnames if n.casefold().endswith("diccionariodedatos")]
        if len(nombres) != 1:
            libro.close()
            raise ValueError("La fuente no contiene la hoja DiccionarioDeDatos.")
        for hoja_fuente in list(libro.worksheets):
            if hoja_fuente.title != nombres[0]:
                libro.remove(hoja_fuente)
    azul = "17365D"
    verde = "26734D"
    for indice, (nombre, datos) in enumerate(OrderedDict(hojas).items(), start=1):
        hoja = libro.create_sheet(nombre)
        if hasattr(datos, "itertuples"):
            filas = list(datos.itertuples(index=False, name=None))
            filas.insert(0, list(datos.columns))
        else:
            filas = list(datos)
        for fila in filas:
            hoja.append([_valor_excel(v) for v in fila])
        # Algunas cadenas narrativas comienzan con signos: Excel debe conservarlas literalmente.
        for fila in hoja:
            for celda in fila:
                celda.font = Font(name="Calibri", size=11)
                celda.alignment = Alignment(vertical="top", wrap_text=True)
                if isinstance(celda.value, str):
                    celda.data_type = "s"
        if not filas:
            continue
        encabezados = [str(c.value) for c in hoja[1]]
        es_narrativa = encabezados == ["PLAN_META_PRODUCTO_CODIGO", "RETRASOS_Y_SOLUCIONES", "AVANCES_Y_LOGROS"]
        for celda in hoja[1]:
            celda.fill = PatternFill("solid", fgColor=verde if celda.value.startswith("CONTROL_") or celda.value.endswith(("_NORMALIZADO", "_PARA_ANALISIS", "_ESTADO_CALIDAD", "_PORCGIROS_SOBRE_PROGRAMADO")) else azul)
            celda.font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
            hoja.column_dimensions[celda.column_letter].width = 32
            if celda.value.endswith("_CODIGO") or celda.value == "PROYECTO_BPIN":
                for fila in hoja.iter_rows(min_row=2, min_col=celda.column, max_col=celda.column):
                    fila[0].number_format = "@"
            if "_NOMBRE" in celda.value or celda.value.startswith("CONTROL_"):
                hoja.column_dimensions[celda.column_letter].width = 65
        if es_narrativa:
            for letra, ancho in {"A": 21, "B": 72, "C": 142}.items():
                hoja.column_dimensions[letra].width = ancho
            hoja.row_dimensions[1].height = 48
            for i, fila in enumerate(filas[1:], start=2):
                alto = max(_lineas(fila[1], 86), _lineas(fila[2], 172)) * 14 + 10
                hoja.row_dimensions[i].height = min(409, max(30, alto))
        elif nombre == "Diccionario_procesado":
            for celda in hoja[1]:
                hoja.column_dimensions[celda.column_letter].width = 75 if celda.column >= 5 else 45
            hoja.row_dimensions[1].height = 48
            for i in range(2, hoja.max_row + 1):
                hoja.row_dimensions[i].height = 210
        else:
            hoja.row_dimensions[1].height = 105
            hoja.sheet_format.defaultRowHeight = 24
        if hoja.max_row > 1:
            tabla = Table(displayName=f"Tabla_{indice}", ref=hoja.dimensions)
            tabla.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
            hoja.add_table(tabla)
        hoja.freeze_panes = "A2"
    if "Datos" in libro.sheetnames:
        libro.move_sheet("Datos", offset=-libro.index(libro["Datos"]))
        libro.active = 0
    temporal = None
    try:
        with NamedTemporaryFile(dir=ruta.parent, suffix=".xlsx", delete=False) as archivo:
            temporal = Path(archivo.name)
        libro.save(temporal)
        if plantilla_diccionario is not None:
            verificar_diccionario_original(plantilla_diccionario, temporal)
        os.replace(temporal, ruta)
    finally:
        libro.close()
        if temporal is not None:
            temporal.unlink(missing_ok=True)
    return ruta

def verificar_diccionario_original(fuente, resultado):
    """Comprueba contenido y presentación contra la hoja oficial del corte."""
    from copy import copy
    from openpyxl import load_workbook
    from xml.etree.ElementTree import tostring

    original = load_workbook(fuente, data_only=False)
    exportado = load_workbook(resultado, data_only=False)
    try:
        nombre = next(n for n in original.sheetnames if n.casefold().endswith("diccionariodedatos"))
        a = original[nombre]
        b = exportado[nombre]
        if a.calculate_dimension() != b.calculate_dimension():
            raise ValueError("El diccionario exportado cambió de dimensiones.")
        if set(map(str, a.merged_cells.ranges)) != set(map(str, b.merged_cells.ranges)):
            raise ValueError("Cambiaron las celdas combinadas del diccionario.")
        for fila in a:
            for celda in fila:
                copia = b[celda.coordinate]
                if (celda.value, celda.data_type) != (copia.value, copia.data_type):
                    raise ValueError(f"Cambió el contenido de {celda.coordinate}.")
                for atributo in ["font", "fill", "border", "alignment", "number_format", "protection"]:
                    if copy(getattr(celda, atributo)) != copy(getattr(copia, atributo)):
                        raise ValueError(f"Cambió {atributo} de {celda.coordinate}.")
        for atributo in ["row_dimensions", "column_dimensions"]:
            origen, destino = getattr(a, atributo), getattr(b, atributo)
            if set(origen) != set(destino):
                raise ValueError(f"Cambiaron las dimensiones: {atributo}.")
            for clave in origen:
                if dict(origen[clave]) != dict(destino[clave]):
                    raise ValueError(f"Cambió la dimensión {atributo}/{clave}.")
        for atributo in ["sheet_format", "sheet_properties", "views", "page_margins", "page_setup", "print_options", "auto_filter"]:
            if tostring(getattr(a, atributo).to_tree()) != tostring(getattr(b, atributo).to_tree()):
                raise ValueError(f"Cambió la presentación del diccionario: {atributo}.")
        if original.loaded_theme != exportado.loaded_theme:
            raise ValueError("Cambió el tema de colores del diccionario.")
    finally:
        original.close()
        exportado.close()



def publicar_salidas_validadas(pares):
    """Publico el CSV y Excel ya validados; revierto un reemplazo si el siguiente falla."""
    from pathlib import Path
    from tempfile import NamedTemporaryFile
    import os
    import shutil
    respaldos, publicados = {}, []
    conservar_respaldos = False
    try:
        for temporal, destino in pares:
            destino = Path(destino)
            if destino.exists():
                with NamedTemporaryFile(dir=destino.parent,prefix=".respaldo_gestion_",suffix=destino.suffix,delete=False) as f:
                    respaldo = Path(f.name)
                respaldos[destino] = respaldo
                shutil.copyfile(destino,respaldo)
        try:
            for temporal,destino in pares:
                destino = Path(destino)
                os.replace(temporal,destino)
                publicados.append(destino)
        except Exception:
            for destino in reversed(publicados):
                try:
                    if destino in respaldos:
                        os.replace(respaldos[destino],destino)
                    else:
                        destino.unlink(missing_ok=True)
                except Exception as restauracion:
                    conservar_respaldos = True
                    raise RuntimeError(f"No pude restaurar {destino}; conservo los respaldos {respaldos} para recuperación.") from restauracion
            raise
    finally:
        if not conservar_respaldos:
            for respaldo in respaldos.values():
                respaldo.unlink(missing_ok=True)



def verificar_datos_excel(ruta, datos):
    """Verifico texto/identificadores exactos y solo tolero representación finita de Excel en números."""
    from openpyxl import load_workbook
    from numbers import Real
    import math
    libro = load_workbook(ruta, read_only=True, data_only=False)
    try:
        hoja = libro["Datos"]
        if (hoja.max_row, hoja.max_column) != (len(datos)+1,len(datos.columns)):
            raise ValueError("El Excel no conserva las dimensiones de la base preparada.")
        if [c.value for c in next(hoja.iter_rows())] != list(datos.columns):
            raise ValueError("El Excel cambió los encabezados de la base preparada.")
        for numero,(guardada,original) in enumerate(zip(hoja.iter_rows(min_row=2),datos.itertuples(index=False,name=None)),start=2):
            for celda,valor in zip(guardada,original):
                esperado = _valor_excel(valor)
                if esperado is None or esperado == "":
                    correcto = celda.value is None or celda.value == ""
                elif isinstance(esperado,Real) and not isinstance(esperado,bool):
                    correcto = isinstance(celda.value,Real) and math.isclose(esperado,celda.value,rel_tol=1e-14,abs_tol=1e-12)
                else:
                    correcto = celda.value == esperado and (not isinstance(esperado,str) or celda.data_type == "s")
                if not correcto:
                    raise ValueError(f"El Excel no conserva el contenido esperado en {celda.coordinate}.")
    finally:
        libro.close()


### Consolidación, diccionarios y comprobación de escritura

En esta celda reúno los campos fuente y derivados, calculo las razones financieras, documento cada columna y verifico el CSV mediante una lectura de vuelta. Después genero el Excel y compruebo la fidelidad de la hoja oficial del diccionario.


In [13]:
from tempfile import NamedTemporaryFile
from decimal import Decimal


def formato_numero_csv(valor):
    numero = float(valor)
    if not np.isfinite(numero):
        raise ValueError("No se exportan valores infinitos.")
    return str(int(numero)) if numero.is_integer() else format(Decimal(repr(numero)), "f").replace(".", ",")


# Una observación por actividad y corte: todos los campos originales, sin joins
# entre las antiguas trece tablas ni repetición de identificadores.
consolidado = pd.concat([datos.copy(deep=True), mediciones], axis=1)

for familia in ["REC", "RECK"]:
    for numero, anio in sorted(MAPA_VIGENCIAS.items()):
        prefijo = f"PROYECTO_ACTIVIDAD_{familia}VIG{numero}_"
        proporcion, estado = razon_giros(datos, prefijo, anio)
        consolidado[prefijo + "PORCGIROS_SOBRE_PROGRAMADO"] = proporcion
        consolidado[f"CONTROL_RECURSOS_{familia}VIG{numero}"] = estado

assert len(consolidado) == len(origen)
assert not consolidado.duplicated([*CLAVE, "FECHA_CORTE"]).any()
pd.testing.assert_frame_equal(consolidado[list(origen.columns)], datos[list(origen.columns)], check_exact=True)
assert not any(c == "SENTIDO_CALCULO" or c.endswith("_CALCULADO") or "SOBRE_COMPROMISOS" in c for c in consolidado)

# Un diccionario para la base completa, conservando también el diccionario fuente.
definiciones = {str(r.iloc[1]): r for _, r in dic_origen.iterrows()}
filas_diccionario = []
for campo in consolidado:
    m = re.search(r"VIG(\d+)(?:_|$)", campo)
    anio = str(MAPA_VIGENCIAS[int(m[1])]) if m else "Total del plan publicado" if "TOTAL" in campo else "Contexto del corte"
    origen_campo, tipo = "FUENTE_EXCEL", "texto"
    unidad = "sin unidad numérica sumable"
    formula = "No aplica: dato fuente conservado."
    nulos = "Se conserva el valor de la fuente; blanco no equivale a cero."
    ejemplo, notas = "", ""
    if campo in definiciones:
        r = definiciones[campo]
        descripcion, tipo, notas = str(r.iloc[3]), str(r.iloc[2]), str(r.iloc[4])
        if campo.endswith("_CODIGO") or campo == "PROYECTO_BPIN":
            tipo = "texto identificador"
            notas += ". Importar como texto para conservar ceros iniciales y el BPIN completo."
        elif campo in PCT:
            unidad = "porcentaje publicado, escala 0–100"
            notas += ". Original conservado; _NORMALIZADO es su conversión exacta a proporción. _PARA_ANALISIS solo contiene valor si pasa los controles técnicos; _ESTADO_CALIDAD explica su elegibilidad."
        elif re.search(r"_REC(?:K)?(?:VIG\d+|TOTAL)_", campo):
            unidad = "millones de pesos constantes del año VIGENCIA" if "_RECK" in campo else "millones de pesos corrientes de la vigencia indicada"
            notas += ". No se convierte a pesos ni se modifica su base de precios."
        elif "_MAG" in campo:
            unidad = "magnitud física propia de la actividad; consultar su descripción"
            notas += ". Se conserva la magnitud. No dividir automáticamente entre 100 ni sumar niveles de actividades no aditivas."
        if campo == "VIGENCIA":
            notas += ". Es el año del reporte, no el año de todas las columnas anuales."
        if "TOTAL" in campo:
            notas += ". Total publicado, no reconstruido sumando las vigencias."
    elif campo == "FE_DE_ERRATAS":
        origen_campo = "FUENTE_SIN_ENTRADA_EN_DICCIONARIO"
        descripcion = "Campo presente en datos, sin entrada en el diccionario oficial de este corte."
        notas = "Conservo el valor fuente y la ausencia de definición; consultar la hoja FeDeErratas si está disponible."
    elif campo == "CORTE_PUBLICACION":
        origen_campo, tipo, unidad = "METADATO_PUBLICACION", "fecha", "AAAA-MM-DD"
        descripcion = "Corte de la publicación que contiene el registro; puede diferir de la fecha del dato fuente."
        formula = "Fecha del corte seleccionado en el catálogo/carpeta, sin sustituir FECHA_CORTE."
        notas = "Identifica la instantánea y permite separar registros rezagados de observaciones del corte."
    elif campo == "FECHA_CORTE":
        origen_campo, tipo, unidad = "DERIVADO_FECHA_FUENTE", "fecha", "AAAA-MM-DD"
        descripcion = "Fecha del seguimiento extraída de VIGENCIA_FECHA_SEGUIMIENTO."
        formula = "Extraer y convertir la fecha DD/MM/AAAA de VIGENCIA_FECHA_SEGUIMIENTO."
        nulos = "No se admiten fechas vacías ni posteriores al corte publicado; se conserva la fecha fuente de registros rezagados."
    elif campo == "ESTADO_SEGUIMIENTO":
        origen_campo = "DERIVADO_ESTADO_FUENTE"
        descripcion = "Estado del seguimiento informado entre paréntesis en VIGENCIA_FECHA_SEGUIMIENTO."
        formula = "Extraer el texto entre paréntesis, sin modificar su significado."
    elif campo.startswith("CONTROL_RECURSOS_"):
        origen_campo, tipo, unidad = "CONTROL_TECNICO_RECURSOS", "texto", "estado de elegibilidad y alertas"
        descripcion = "Estado de la razon giros/programado de la misma vigencia y base de precios."
        formula = "Verificar anio observado, denominador positivo y recursos no negativos; informar sobreprogramacion u orden presupuestal."
        nulos = "No se deja vacio. Las razones no aplicables quedan vacias en su campo numerico."
        notas = "La coherencia aritmetica no certifica la correccion presupuestal de la fuente."
    elif campo.startswith("CONTROL_MEDICION_MAG"):
        origen_campo, tipo = "CONTROL_TECNICO", "texto"
        descripcion = "Resumen de estados de compromiso y entrega; no representa una corrección de magnitud."
        unidad = "código de calidad"
        formula = "Concatenar los estados de las dos medidas de la vigencia o total."
        notas = "Un estado pendiente no se reemplaza por cero ni por un porcentaje estimado."
        nulos = "No se deja vacío. COHERENTE indica que ambas medidas pasan los controles."
    elif campo.endswith("_NORMALIZADO"):
        origen_campo, tipo, unidad = "CONVERSION_EXACTA_FUENTE", "numérico decimal", "proporción: 1 equivale a 100 %"
        descripcion = "Porcentaje publicado convertido a proporción, sin corregir su significado."
        formula = campo.removesuffix("_NORMALIZADO") + " / 100"
        notas = "Puede conservar anomalías del productor. No se recorta a 1. Para uso filtrado consultar _PARA_ANALISIS y _ESTADO_CALIDAD."
        nulos = "La lectura exige un porcentaje fuente numérico y finito."
        ejemplo = "49 → 0,49; un porcentaje publicado de 6370 permanece 63,7 y puede quedar pendiente de revisión."
    elif campo.endswith("_PARA_ANALISIS"):
        origen_campo, tipo, unidad = "SELECCION_TECNICA_PARA_ANALISIS", "numérico decimal", "proporción: 1 equivale a 100 %"
        descripcion = "Proporción publicada habilitada por los controles; no es una estimación ni una corrección oficial."
        formula = "Usar _NORMALIZADO si _ESTADO_CALIDAD comienza por COHERENTE; en otro caso dejar vacío."
        nulos = "Vacío sin denominador, vigencia futura, total no Suma o impedimento aritmético. Las alertas exploratorias por sí solas no vacían el valor; vacío no equivale a cero."
        notas = "La coherencia técnica no certifica veracidad institucional. Informar cobertura de estas medidas y no borrar las actividades excluidas."
    elif campo.endswith("_ESTADO_CALIDAD"):
        origen_campo, tipo, unidad = "DIAGNOSTICO_DE_MEDICION", "texto", "estado técnico"
        descripcion = "Motivo que habilita o impide usar la proporción en análisis."
        formula = "Comprobar denominador, fecha observada, cociente, pista de unidad y publicaciones anteriores de la misma definición y programación."
        notas = "El contraste temporal excluye el corte objetivo y posteriores. Una alerta decimal es una hipótesis que no modifica ni excluye por sí sola la proporción. Coherencia aritmética no certifica logro sustantivo ni acumulación entre vigencias."
        nulos = "No se admiten estados vacíos."
    elif campo.endswith("_PORCGIROS_SOBRE_PROGRAMADO"):
        origen_campo, tipo, unidad = "CALCULADO_DESDE_RECURSOS", "numérico decimal", "proporción: 1 equivale a 100 %"
        descripcion = "Parte de los recursos previstos que se ha girado, para la misma actividad, vigencia y base de precios."
        prefijo = campo.removesuffix("PORCGIROS_SOBRE_PROGRAMADO")
        formula = prefijo + "GIROS / " + prefijo + "PROGRAMADO"
        nulos = "Vacío si programado=0, incluso si giros=0. Con programado distinto de cero y giros=0, el resultado es 0."
        notas = "Sin dividir entre 100, sin ajuste MAG ni tope de 1. No mide giros sobre compromisos. PROGRAMADO no se equipara automáticamente con apropiación definitiva. Para agregar: suma de giros/suma de programado; no promediar porcentajes simples ni sumar recursos corrientes y constantes."
        ejemplo = "Giros 600/programado 1000 = 0,6 = 60 %."
    else:
        raise ValueError(f"Falta documentar {campo}")
    filas_diccionario.append({
        "VARIABLE": campo, "ORIGEN": origen_campo, "ARCHIVOS": "gestion_inversion.csv",
        "ANIO_MEDICION": anio, "DESCRIPCION": descripcion, "TIPO": tipo,
        "UNIDAD_Y_ESCALA": unidad, "FORMULA_DIRECTA": formula,
        "FORMULA_INVERSA": "No aplica.", "SIGNIFICADO_DE_VACIOS": nulos,
        "EJEMPLO": ejemplo, "OBSERVACIONES": notas})
diccionario_salida = pd.DataFrame(filas_diccionario).astype("string")
assert diccionario_salida.VARIABLE.tolist() == list(consolidado)

DESTINO.mkdir(parents=True, exist_ok=True)
# Valido ambas salidas temporales antes de publicar el par CSV/Excel.
ruta = DESTINO / "gestion_inversion.csv"
RUTA_EXCEL = DESTINO / "gestion_inversion.xlsx"
temporal_csv = temporal_excel = None
try:
    with NamedTemporaryFile(dir=DESTINO, suffix=".part", delete=False) as f:
        temporal_csv = Path(f.name)
    with NamedTemporaryFile(dir=DESTINO, suffix=".xlsx", delete=False) as f:
        temporal_excel = Path(f.name)
    consolidado.to_csv(temporal_csv, sep="|", decimal=",", float_format=formato_numero_csv,
                       index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL,
                       date_format="%Y-%m-%d", na_rep="")
    tipos = {c:"string" for c in consolidado if str(consolidado[c].dtype) == "string"}
    fechas = [c for c in consolidado if pd.api.types.is_datetime64_any_dtype(consolidado[c])]
    vuelta = pd.read_csv(temporal_csv, sep="|", decimal=",", encoding="utf-8-sig", dtype=tipos,
                         parse_dates=fechas, keep_default_na=False,
                         na_values={c:[""] for c in consolidado if pd.api.types.is_float_dtype(consolidado[c])},
                         float_precision="round_trip")
    pd.testing.assert_frame_equal(consolidado.reset_index(drop=True), vuelta, check_dtype=False, check_exact=True)
    with temporal_csv.open(encoding="utf-8-sig", newline="") as f:
        filas_csv = list(csv.reader(f, delimiter="|", strict=True))
    assert filas_csv[0] == list(consolidado)
    assert len(filas_csv) == len(consolidado)+1 and all(len(r)==len(consolidado.columns) for r in filas_csv)
    exportar_excel(temporal_excel, {"Datos":consolidado,"Diccionario_procesado":diccionario_salida},
                   plantilla_diccionario=FUENTE)
    verificar_datos_excel(temporal_excel, consolidado)
    assert hashlib.sha256(FUENTE.read_bytes()).hexdigest() == HUELLA
    publicar_salidas_validadas([(temporal_csv,ruta),(temporal_excel,RUTA_EXCEL)])
finally:
    if temporal_csv is not None: temporal_csv.unlink(missing_ok=True)
    if temporal_excel is not None: temporal_excel.unlink(missing_ok=True)
print(f"Publicadas {len(consolidado)} actividades, {len(origen.columns)} campos fuente y {len(consolidado.columns)} campos totales.")
print("Normalización exacta y estados verificados; el diccionario oficial conserva su contenido y formato.")
print("CSV:",ruta.relative_to(RAIZ),"| Excel:",RUTA_EXCEL.relative_to(RAIZ))


C:\Users\USER\AppData\Local\Temp\ipykernel_23592\129144657.py:142: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  vuelta = pd.read_csv(temporal_csv, sep="|", decimal=",", encoding="utf-8-sig", dtype=tipos,


Publicadas 1655 actividades, 78 campos fuente y 132 campos totales.
Normalización exacta y estados verificados; el diccionario oficial conserva su contenido y formato.
CSV: Capa Silver\2026-06-30\gestion_inversion\gestion_inversion.csv | Excel: Capa Silver\2026-06-30\gestion_inversion\gestion_inversion.xlsx


## 8. Uso de la base preparada

La base conserva las filas de la instantánea seleccionada y añade metadatos, conversiones y estados. El tamaño depende de la fuente; el historial no incorpora actividades adicionales al resultado operativo.

Utilizo como referencia el porcentaje original y `_NORMALIZADO`, junto con anualización, vigencia y `_ESTADO_CALIDAD`. `_PARA_ANALISIS` es una selección opcional con controles aritméticos y de aplicabilidad: no es un porcentaje corregido ni un indicador universal de desempeño.

Informo cobertura y alertas antes de analizar. Una celda analítica vacía no equivale a cero. Compartir escala no hace comparables actividades de unidades distintas ni convierte una revisión retrospectiva en ejecución del trimestre.



## 9. Alcance de la actualización periódica

La operación prepara el corte seleccionado mediante reglas generales y aplica controles a todos sus registros. No depende de una lista de actividades autorizadas. El mapa de vigencias y los conteos se obtienen del archivo; la referencia temporal solo aporta antecedentes cuando son comparables y estaban disponibles.

La referencia incorporada conserva las ediciones auditadas; no se actualiza por el solo hecho de descargar una fuente nueva. Su ausencia para una actividad no implica error ni acredita su medición. Un plan, esquema o unidad nuevos requieren revisión antes de ampliar compatibilidad.

Compruebo conservación, conversión, aplicabilidad y reproducción de las salidas. No certifico la medición institucional ni deduzco una corrección única a partir de una serie temporal. Una fuente incompatible detiene la etapa antes de publicar una salida no verificada.



## 10. Interpretación del resultado de cada ejecución

Leo los resúmenes junto con el archivo seleccionado, su huella, las fechas observadas y el mapa de vigencias. No traslado a una actualización posterior las cardinalidades, porcentajes de cobertura o excepciones de otra publicación.

Los controles de las celdas anteriores se repiten al ejecutar el notebook. Las auditorías independientes y pruebas históricas se conservan por separado como evidencia de las fuentes probadas: su fecha no es la fecha del corte actual ni una certificación de datos futuros.



### Identificación calculada de la base preparada y su referencia

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [17]:
from datetime import datetime
from zoneinfo import ZoneInfo
_momento_informe = datetime.now(ZoneInfo("America/Bogota")).isoformat(timespec="seconds")
_anios_informe = ", ".join(f"VIG{i} = {a}" for i,a in sorted(MAPA_VIGENCIAS.items()))
_ediciones_informe = [f["corte"] for f in fuentes_referencia]
_intervalo_ref = f"{min(_ediciones_informe)} a {max(_ediciones_informe)}" if _ediciones_informe else "sin ediciones registradas"
_referencias_disponibles = preparar_indice_temporal(referencia_temporal, FECHA)
display(Markdown(f"**Informe generado:** {_momento_informe}. Preparo **{len(consolidado)} actividades y {len(consolidado.columns)} campos** del corte **{FECHA.isoformat()}**. El mapa leído del diccionario es: {_anios_informe}.\n\nLa referencia incorporada contiene **{len(referencia_temporal)} observaciones** de **{len(fuentes_referencia)} ediciones** ({_intervalo_ref}); para este corte, el filtro temporal retiene **{sum(len(v) for v in _referencias_disponibles.values())} antecedentes** antes del contraste individual. Esta cantidad no implica que todas las actividades tengan una trayectoria comparable. La referencia no se amplía automáticamente con esta ejecución."))


**Informe generado:** 2026-10-29T20:49:04-05:00. Preparo **1655 actividades y 132 campos** del corte **2026-06-30**. El mapa leído del diccionario es: VIG1 = 2024, VIG2 = 2025, VIG3 = 2026, VIG4 = 2027.

La referencia incorporada contiene **13084 observaciones** de **8 ediciones** (2024-12-31 a 2026-09-30); para este corte, el filtro temporal retiene **9756 antecedentes** antes del contraste individual. Esta cantidad no implica que todas las actividades tengan una trayectoria comparable. La referencia no se amplía automáticamente con esta ejecución.

---

<details>
<summary>Anexo histórico: evidencias y redacción de las revisiones anteriores — no describe el corte seleccionado</summary>

Conservo estos textos como registro de revisiones puntuales, realizadas sobre las fuentes y fechas que cada apartado identifica. Sus conteos, ejemplos, resultados y límites pertenecen a esas versiones. No son resultados recalculados ni condiciones satisfechas para una nueva ejecución. La metodología operativa y los informes del corte seleccionado están en el cuerpo principal. Las revisiones históricas no se amplían automáticamente.

# Preparación del último corte de Gestión-Inversión y control de las mediciones

## 1. Qué preparo y qué puedo comprobar

En este notebook preparo únicamente las actividades del corte seleccionado. Uso los cortes anteriores para reconocer cómo se reportan las medidas y detectar inconsistencias, pero no los concateno con las observaciones de salida ni los descargo cada vez que limpio el último archivo.

La revisión cubrió 13.089 registros de ocho instantáneas de BCS desde diciembre de 2024 hasta septiembre de 2026. A partir de ella conservo dentro de los metadatos de este mismo notebook una referencia de 13.084 observaciones de su vigencia activa. Excluí cinco registros rezagados como puntos temporales nuevos. La referencia incluye fuentes, huellas SHA-256, corte y fecha de publicación de cada edición, sin requerir archivos auxiliares para usar el proyecto en otro equipo.

Mi objetivo es garantizar controles comprobables de representación, conservación y cálculo, no atribuir veracidad institucional a una cifra por haberla limpiado. Una proporción algebraicamente coherente puede provenir de una medición sustantiva incorrecta. Por ello separo el valor publicado, su conversión exacta, la elegibilidad técnica para análisis y el estado que explica las incidencias.

La descarga del notebook 01 se orienta por defecto al último corte numérico común disponible. Si no fijo `CORTE`, selecciono el último corte local con Excel de ambas fuentes. Mantengo la fecha observada y el corte de publicación separados y conservo la fuente original. Las cifras históricas que comento corresponden a las versiones revisadas el 29 de octubre de 2026; las tablas ejecutables se recalculan para la fuente seleccionada.


---

## 2. Cómo reconozco las versiones del Excel

En el siguiente bloque identifico la hoja de actividades y su diccionario por los perfiles observados. En diciembre de 2024 encontré `para_web_datos` con 76 campos y `para_web_DiccionarioDeDatos`; en 2025 y 2026 encontré versiones de 77 y 78 campos. No exijo hojas auxiliares que solo aparecieron en una publicación posterior.

Homologo mayúsculas y el nombre histórico `PLAN_META_PRODUCTO_ID`, cuya correspondencia con el código de meta contrasté contra Avance del Plan. Conservo `bruta_fuente` y el mapa de renombres. La copia oficial del diccionario en Excel mantiene su nombre, contenido y formato originales; la vista homologada sirve para controles y documentación analítica.

Obtengo el año de cada VIG directamente de las observaciones del diccionario, sin imponer 2024–2027 como regla universal. Si una vigencia no tiene un año inequívoco, detengo la carga. También exijo un único plan y compruebo su correspondencia con Avance del Plan cuando ya está preparado.

Compruebo además que las celdas de porcentaje mantengan la representación numérica auditada. Si el Excel cambia a fracciones con formato porcentual activo, detengo la carga para revisar la escala y evitar una segunda división entre cien.

En marzo y septiembre de 2026 encontré `FE_DE_ERRATAS` en datos sin una entrada en el diccionario. Conservo el campo y registro esa omisión, sin modificar el diccionario oficial ni inventar una definición atribuida a la SDP.


---

### 2.1. Por qué contrasto porcentajes y magnitudes sin sustituirlos

En este bloque comparo el porcentaje publicado con magnitud/programado por cien, solo cuando programado es distinto de cero. Uso una tolerancia absoluta de 0,0051 puntos porcentuales y relativa de 1e-9 para no atribuir a un problema sustantivo un redondeo a dos decimales. Esta tolerancia es exclusiva del diagnóstico: la conservación de la fuente y la relectura del CSV se comprueban por separado.

No declaro que cualquier diferencia sea un error. La contrasto con la fecha, el tipo de anualización y las referencias a erratas. Encontré, por ejemplo, un total de entrega de la actividad `0265 / 007535 / 1` en junio de 2025: programado 1, entrega 0,9 y porcentaje publicado 9. También encontré la actividad `0200 / 007934 / 2` con programado 755, entrega 1.005 y porcentaje publicado 755 en las versiones de 2026, asociada a `3 - IPES` en las erratas. Conservo esos valores y señalo la discrepancia; no los convierto automáticamente en 90 o en 133,11 %.

Este control me permite revisar todas las medidas, no solo los ejemplos que seleccioné para explicar una mezcla de escala. Las primeras 20 incidencias se muestran para lectura; la tabla completa queda en memoria.


---

### 2.2. Qué encontré al explorar las actividades

En junio de 2026 encontré 1.655 filas y 78 campos fuente, sin claves repetidas. El perfil mostró diferencias de espacios en 5 nombres de meta, 11 nombres de proyecto, 260 descripciones de actividad, 728 tipos de anualización y 54 etiquetas ODS. Por eso normalicé representación, sin reescribir las palabras.

Después de esa normalización identifiqué 927 actividades de Suma, 553 de Constante, 169 de Creciente y 6 de Decreciente. No adopté una misma suma de magnitudes para todas ellas. Conservé los 1.634 vacíos de `FE_DE_ERRATAS` como ausencia de referencia, no como prueba de que el registro carezca de problemas.

La exploración de porcentajes mostró valores muy elevados y ceros históricos que cambiaban entre versiones. No los traté como errores solo por su rango. Primero contrasté programación, compromiso, entrega, porcentaje y trayectoria de cada caso señalado.


---

## 3. Qué limpio y qué conservo

En la siguiente celda normalizo espacios y Unicode, mantengo los identificadores como texto y convierto las medidas con comprobaciones numéricas estrictas. No borro actividades, columnas constantes, ceros futuros ni porcentajes extremos. También mantengo las unidades y las bases de precios.

Extraigo `FECHA_CORTE` y `ESTADO_SEGUIMIENTO` del texto fuente sin retirarlo. Además añado `CORTE_PUBLICACION` para identificar la instantánea descargada. Esta separación fue necesaria porque el Excel de marzo de 2025 incluye cinco actividades con fecha fuente de diciembre de 2024. Conservo ambas fechas y muestro esos registros, sin imputarlos a marzo ni eliminarlos.

Exijo que la vigencia corresponda al año de la fecha observada y que no haya observaciones posteriores al corte publicado. La presencia de un registro rezagado queda documentada y no se interpreta como una observación nueva del trimestre. Al combinar publicaciones, la identidad del plan y de la instantánea debe formar parte de la trazabilidad.


---

### 5.2. Qué cambió al contrastar la serie sin adivinar un porcentaje

| Caso observado | Conversión publicada que conservo | Tratamiento para análisis |
|---|---|---|
| Una estrategia pasa de entrega `0,26` en marzo de 2026 a `63,7` en junio y `0,82` en septiembre; junio publica 6.370 % | `6370/100 = 63,7` | No presento `0,637` como una corrección cierta: marco una alerta de escala sin excluir por esa sola conjetura; el valor oficial continúa visible |
| Una actividad publica programado 1, entrega 0,9 y porcentaje 9 | `9/100 = 0,09` | Señalo la discordancia con el cociente; no sustituyo automáticamente por 0,9 |
| Un compromiso porcentual histórico pasa a cero, conservando una magnitud positiva | `0/100 = 0` | No recupero un supuesto porcentaje desde la magnitud; dejo visible la discrepancia |
| El porcentaje publicado es 49 y concuerda con una magnitud de 0,49 sobre una unidad programada | `49/100 = 0,49` | Habilito 0,49 si los demás controles no detectan incidencias |
| Programado y entregado son cero | Conservo el cero publicado | No calculo una medición `0/0`: dejo vacío el campo analítico y explico el denominador ausente |

La trayectoria puede respaldar una hipótesis de mezcla de escala, pero no identifica siempre una única corrección. El caso de eventos estudiado anteriormente podía admitir interpretaciones distintas; eliminé su corrección automática. Prefiero conservar el dato y la incertidumbre que asignar un porcentaje aparentemente preciso sin suficiente respaldo.

La revisión también mostró cambios de descripción bajo una misma clave. Por eso no enlazo automáticamente toda fila que comparta códigos, ni uso una interpolación o un suavizado para fabricar el avance esperado del trimestre.


---

## 8. Cómo utilizar el resultado sin perder la trazabilidad

Para el corte local de junio de 2026 conservo 1.655 actividades y 78 campos fuente. La base preparada contiene 132 columnas, incluidos metadatos temporales, conversiones, medidas habilitadas y estados. La ampliación no crea nuevas actividades ni reemplaza sus datos originales.

Adopto como referencia principal el porcentaje original y su conversión exacta `_NORMALIZADO`, acompañados por anualización, vigencia y `_ESTADO_CALIDAD`. Si necesito una selección con controles aritméticos y de aplicabilidad utilizo `_PARA_ANALISIS`, reportando su cobertura y las alertas: no es un porcentaje corregido ni una medida universal de desempeño. No interpreto una celda analítica vacía como cero ni elimino la actividad completa por una incidencia en una sola medida.

El porcentaje normalizado, la razón financiera y un indicador físico de distinta definición no se vuelven comparables sustantivamente por compartir escala. Tampoco tomo una revisión retrospectiva o un registro rezagado como una variación trimestral. El resultado delimita qué cálculo hice y qué aspectos de la fuente continúan pendientes de confirmación.


---

## 9. Qué queda automatizado y qué no afirmo

La operación rutinaria adquiere el último corte numérico y prepara sus datos con las reglas generales contrastadas. No depende de una lista de actividades o fechas autorizadas y no necesita descargar todo el histórico en cada ejecución. La referencia incorporada aporta antecedentes cuando existen y se utiliza con filtros de identidad, disponibilidad y tiempo.

Puedo comprobar conservación, conversión de escala, cálculos, estados y reproducción en los archivos probados. No puedo certificar que la entidad haya medido bien una actividad, ni deducir una corrección única de cualquier inconsistencia mediante su serie temporal. Por eso una incidencia no se maquilla como un valor corregido.

Mi control de calidad permite continuar cuando la medida actual es técnicamente coherente y señala las que requieren revisión. Ante una fuente incompatible o un fallo que impide verificar la salida, preservo la evidencia y comunico el problema. Este es el alcance verificable de la limpieza, no una garantía de desempeño institucional ni una promesa de ausencia de fallos de datos o del entorno.


---

## 10. Comprobación del procedimiento en los ocho cortes auditados

Ejecuté el procedimiento sobre todas las filas de las ocho fuentes, en una copia trasladada del proyecto. Verifiqué conservación, conversiones, selección de medidas y construcción del diccionario. Las cifras siguientes se refieren a la vigencia activa de cada publicación y distinguen revisión de falta de medición aplicable.

| Corte | Actividades | Campos finales | Compromisos: habilitados / revisión / no aplica | Entrega: habilitados / revisión / no aplica |
|---|---:|---:|---|---|
| 2024-12-31 | 1602 | 130 | 1400 / 4 / 198 | 1400 / 3 / 199 |
| 2025-03-31 | 1618 | 131 | 314 / 1189 / 115 | 1501 / 2 / 115 |
| 2025-06-30 | 1619 | 131 | 1510 / 6 / 103 | 1510 / 6 / 103 |
| 2025-09-30 | 1634 | 131 | 1515 / 4 / 115 | 1515 / 4 / 115 |
| 2025-12-31 | 1637 | 132 | 1505 / 1 / 131 | 1505 / 1 / 131 |
| 2026-03-31 | 1651 | 132 | 1518 / 0 / 133 | 1518 / 0 / 133 |
| 2026-06-30 | 1655 | 132 | 1521 / 0 / 134 | 1521 / 1 / 133 |
| 2026-09-30 | 1673 | 132 | 1516 / 0 / 157 | 1516 / 1 / 156 |

En marzo de 2025, por ejemplo, el elevado número de compromisos pendientes corresponde a porcentajes que no concuerdan con la magnitud reportada; no lo oculto sustituyendo el porcentaje por un cociente. Una base sin alertas después de esa sustitución parecería más completa, pero perdería trazabilidad.

La referencia utiliza 13.084 observaciones de vigencia activa de las ocho instantáneas revisadas; sus fuentes y huellas están en los metadatos del notebook. Las ediciones publicadas después del corte evaluado no actúan como antecedentes. Esta validación no reconstruye todas las versiones que pudieron existir antes de una revisión del proveedor.


### Fuentes de las ocho instantáneas contrastadas

Registro las ediciones utilizadas; las huellas SHA-256 y las observaciones de referencia permanecen en los metadatos de este notebook. La fecha de edición permite evitar usar una revisión que todavía no estaba publicada al evaluar un corte anterior.

| Corte | Edición publicada en el portal | Fuente oficial auditada |
|---|---|---|
| 2024-12-31 | 2025-02-11 | [Excel original](https://www.sdp.gov.co/sites/default/files/pa_cg_ci_20241231_0.xlsx) |
| 2025-03-31 | 2025-05-19 | [Excel original](https://www.sdp.gov.co/sites/default/files/pa_cg_ci_20250331_act20250519.xlsx) |
| 2025-06-30 | 2025-07-30 | [Excel original](https://www.sdp.gov.co/sites/default/files/pa_cg_ci_20250630_ver20250730.xlsx) |
| 2025-09-30 | 2025-11-20 | [Excel original](https://www.sdp.gov.co/sites/default/files/planacion_gestion-inversion_20250930_entidad_v20251120.xlsx) |
| 2025-12-31 | 2026-04-15 | [Excel original](https://www.sdp.gov.co/sites/default/files/planaciongestioninventidad20251231_v20260415.xlsx) |
| 2026-06-30 | 2026-07-28 | [Excel original](https://www.sdp.gov.co/sites/default/files/planacciongestioninversion20260630_ver20260728.xlsx) |
| 2026-09-30 | 2026-10-27 | [Excel original](https://www.sdp.gov.co/sites/default/files/planacciongestioninversion20260930_ver20261027.xlsx) |
| 2026-03-31 | 2026-05-06 | [Excel original](https://www.sdp.gov.co/sites/default/files/planacciongestioninversionentidad20260331_ver20260505_0.xlsx) |



---

### 10.1. Cobertura del contraste aritmético de entregas

Recorrí los porcentajes de entrega de las cuatro vigencias y del total en las ocho publicaciones. El denominador de esta tabla cuenta medidas comparables, no actividades únicas. No comparo un cociente cuando su programación es cero.

| Corte | Medidas con programado distinto de cero | Diferencias frente al cociente de referencia |
|---|---:|---:|
| 2024-12-31 | 6756 | 0 |
| 2025-03-31 | 6755 | 0 |
| 2025-06-30 | 6784 | 1 |
| 2025-09-30 | 6814 | 0 |
| 2025-12-31 | 6790 | 0 |
| 2026-03-31 | 6774 | 1 |
| 2026-06-30 | 6778 | 1 |
| 2026-09-30 | 6803 | 1 |

Las excepciones y las referencias a erratas sustentan mi decisión de conservar los porcentajes publicados en lugar de recalcularlos indiscriminadamente. El diagnóstico no convierte una anomalía de origen en una corrección oficial.


---

### 10.2. Pruebas de frontera y recuperación

Además de recorrer los ocho archivos, probé los siguientes casos diseñados para verificar las decisiones del procedimiento:

- Cuatro anualizaciones conservan el porcentaje anual sin invertirlo ni agregarlo.
- Total físico no Suma no habilitado incluso con valores fuente positivos.
- Anualización desconocida requiere revisión explícita.
- Porcentaje literal 63,7 convertido una sola vez a 0,637.
- Sobrecumplimiento válido conservado sobre 1.
- Cero de ejecución distinguido de faltante.
- 0/0 no se presenta como porcentaje medido.
- Pista textual de escala alerta sin corregir ni excluir por conjetura.
- Discordancia 0,9/1 frente a 9 % detectada.
- Cero publicado no sustituido por una magnitud positiva.
- Valor negativo enviado a revisión.
- Registro rezagado no se convierte en medición de la vigencia posterior.
- No finitos impiden publicar.
- Salto temporal alerta sin excluir una ejecución discontinua por sí solo.
- Añadir publicaciones futuras no cambia una decisión pasada.
- Una publicación futura aislada no actúa como antecedente.
- Cambiar la definición impide enlazar la serie.
- Cambiar la programación impide comparar el denominador temporal.
- Una revisión publicada después del corte no actúa como antecedente.
- Recursos: cero real, falta de denominador, rezago, negativos y sobreprogramación diferenciados.
- Fallo al publicar Excel restaura el CSV anterior.
- Relectura Excel conserva códigos, literales y nulos; detecta pérdida de ceros.
- Cambio al formato porcentual de Excel bloqueado antes de dividir entre cien con otra escala.

Estas pruebas verifican la implementación y los controles declarados; no certifican una verdad sustantiva que no esté respaldada por la fuente. Las pruebas y descargas de revisión se mantuvieron fuera de la carpeta de entrega.



---

---

<details>
<summary>Diagnostico previo conservado (28 de octubre de 2026)</summary>

---

# Anexo: diagnóstico previo conservado

Documento histórico de la revisión inicial. Sus referencias a etapas pendientes describen ese momento; el procesamiento actual está en las celdas anteriores.

# Diagnóstico integral de Gestión-Inversión

Fecha de revisión: 28 de octubre de 2026. Corte de los datos: 30 de junio de 2026. Proyecto de grado, etapa de comprensión y diagnóstico previo a la limpieza.

## 1. Conclusión principal

Los archivos son complementarios y tienen correspondencia estructural, pero no son intercambiables ni deben fusionarse mediante una unión directa por indicador o proyecto. El Excel contiene actividades, magnitudes y recursos; el PDF incorpora indicadores, sus programaciones y ejecuciones, y los relatos institucionales. Las diferencias de escala porcentual, anualización, base de precios y fe de erratas requieren reglas específicas.

No se ha limpiado ni exportado Gestión-Inversión a Silver. No se modificó ningún archivo original. Los hallazgos siguientes distinguen valores observados, interpretación apoyada por el diccionario y recomendaciones para el diseño posterior.

## 2. Cobertura y método de revisión

- Se extrajo y recorrió el contenido textual de las 264 páginas del PDF: 3.060.958 caracteres; ninguna página carece de texto y todas contienen el corte 30/06/2026. La página 264 es metodológica; las anteriores contienen entidades y seguimiento.
- Se leyeron todas las celdas y hojas del Excel, con inventario de dimensiones, vacíos, errores y fórmulas, perfiles de las 78 columnas de datos y verificación aritmética completa de familias de variables.
- Se realizó revisión visual dirigida del diseño, encabezados, campos narrativos y metodología. La extracción integral y las comprobaciones automáticas no equivalen a una inspección visual manual de cada celda de las 264 páginas ni a una validación externa de cada afirmación institucional.
- Se reconstruyeron por geometría las etiquetas y los contextos de las narrativas, teniendo en cuenta la rotación del PDF. Una extracción textual lineal puede colocar el cuerpo de un relato antes de su etiqueta.
- Se cotejaron catálogos y claves entre archivos; factores de IPC, sumas de vigencias, porcentajes y todas las filas de las tablas dinámicas.
- Se contrastaron importes explícitos en las filas financieras de proyecto del PDF con agregados del Excel, separando valores corrientes y constantes y considerando redondeos. Los espacios en blanco del PDF no se interpretaron automáticamente como ceros.
- Se revisaron muestras sustantivas de relatos de todas las entidades y todos los casos de anomalías de longitud, marcadores y diferencias financieras identificadas. No se realizó codificación cualitativa manual exhaustiva de los 1.034 campos narrativos: esa fase requiere una unidad analítica y un protocolo de anotación.

Las comprobaciones se hicieron con openpyxl/pandas para Excel, PyMuPDF para texto y coordenadas y una segunda extracción con pypdf para confirmar los cuatro finales aparentemente truncados. No se requirió OCR: el PDF contiene texto seleccionable. Que sea seleccionable no garantiza por sí solo que su orden de lectura sea correcto.

### Archivos originales y huellas de integridad

| Fuente | Ruta | SHA-256 |
| --- | --- | --- |
| PDF | Capa Bronce/2026-06-30/gestion_inversion/planacciongestioninversion20260630_entidades_a2ddd1077b.pdf | c473c0b452ddf172d03af549ae7f86ca8315247cfaeed9090f8b7b27c61865b3 |
| XLSX | Capa Bronce/2026-06-30/gestion_inversion/planacciongestioninversion20260630_ver20260728_75dd904475.xlsx | 5bec7adbcc4215d58afc28125758757f1cc8b2837e7142a657fbbe90a86d329f |

El PDF identifica el reporte `PA_RepoCGI_01_Entidad`, versión Seguimiento, y muestra impresión el 24 de septiembre de 2026. Su metadato de creación es del mismo día. El nombre del Excel contiene `ver20260728`; eso sugiere una versión de julio, pero el nombre por sí solo no demuestra su historia de revisiones. Compartir corte no garantiza compartir versión o estado de corrección. La fecha de corte, la fecha de impresión y una posible fecha de versión son conceptos distintos.

## 3. Inventario completo del Excel

| Hoja | Filas físicas | Columnas | Función |
| --- | --- | --- | --- |
| DiccionarioDeDatos | 84 | 5 | Diccionario de 78 campos y notas |
| datos | 1656 | 78 | 1.655 registros de actividad |
| FeDeErratas | 9 | 2 | 8 notas de corrección |
| IPC | 5 | 4 | 4 vigencias, IPC y conversión de precios |
| TablasDinamicasIncluidas | 11 | 2 | Índice explicativo de 6 tablas dinámicas |
| TD_EjecPptalxSector | 20 | 5 | Agregación financiera; contiene total general y, cuando corresponde, subtotales |
| TD_EjecPptalxEntidad | 52 | 5 | Agregación financiera; contiene total general y, cuando corresponde, subtotales |
| TD_EjecPptalxOE | 9 | 5 | Agregación financiera; contiene total general y, cuando corresponde, subtotales |
| TD_EjecPptalxOEyProg | 48 | 7 | Agregación financiera; contiene total general y, cuando corresponde, subtotales |
| TD_EjecPptalxOEyProgMetaProd | 473 | 9 | Agregación financiera; contiene total general y, cuando corresponde, subtotales |
| TD_EjecPptalxODS_MetaODS | 97 | 5 | Agregación financiera; contiene total general y, cuando corresponde, subtotales |

No se detectaron fórmulas ni celdas de error en los valores de las 11 hojas. Las tablas dinámicas contienen resultados y estructura de agregación, no observaciones adicionales. No debe usarse la primera fila como encabezado en todas las hojas: `datos`, `IPC` y `FeDeErratas` tienen encabezados iniciales; el diccionario comienza su encabezado en la fila 6 y las tablas dinámicas en la fila 3. `TablasDinamicasIncluidas` contiene introducción e índice, no una base analítica.

## 4. Unidad de observación, cobertura y relaciones

En este archivo la clave `(ENTIDAD_CODIGO, PROYECTO_CODIGO, PROYECTO_ACTIVIDAD_CODIGO)` identifica las 1.655 filas sin duplicados ni claves repetidas. Una fila es una actividad de proyecto vinculada a una meta producto para este corte. Para una base histórica se debe agregar la fecha de corte a la clave. La clave proyecto–actividad también resulta única en el archivo, pero conservar entidad aporta contexto y un control adicional.

La jerarquía no debe confundirse con una cadena de correspondencias uno a uno: una entidad tiene proyectos y actividades; cada actividad se vincula a una meta producto; una meta pertenece a un programa y este a un objetivo. Un proyecto puede aportar a varias metas producto.

| Dimensión | Cantidad |
| --- | --- |
| ENTIDAD_CODIGO | 48 |
| SECTOR_CODIGO | 16 |
| PLAN_NIVEL1_CODIGO | 5 |
| PLAN_NIVEL2_CODIGO | 39 |
| PLAN_META_PRODUCTO_CODIGO | 425 |
| PROYECTO_CODIGO | 353 |
| PROYECTO_BPIN | 352 |
| ODS | 18 |
| ODS_META | 77 |

Las 18 etiquetas textuales ODS representan **16 códigos ODS**, no 18 objetivos: hay variantes de capitalización en los ODS 04 y 11, y el ODS 14 no aparece en este corte. No se deben crear entidades distintas por cambiar “Calidad” a “calidad”. Las etiquetas de meta ODS usan una sintaxis diferente de Avance del Plan: por ejemplo `16.06-...` frente a `16.06. ...`; los sufijos pueden ser numéricos o alfabéticos.

Los 48 registros únicos de entidad son las entidades efectivamente incluidas en los archivos; el título del PDF “Entidad o Alcaldía Local” no demuestra cobertura de las 20 alcaldías locales. No se encontraron entidades cuyo nombre las identifique como alcaldía local.

Comprobaciones de dependencias:
- Cada entidad pertenece a un solo sector en este archivo.
- Cada proyecto se vincula a una sola entidad y un solo BPIN.
- Cada programa se vincula a un solo objetivo y cada meta a un programa.
- **119 proyectos se vinculan a más de una meta**, con un máximo de 16; por eso un enlace solamente por proyecto puede multiplicar filas.
- Hay 353 códigos de proyecto y 352 BPIN: los proyectos 007501 y 007519 comparten el BPIN 2020110010193, relacionado con la Primera Línea del Metro. BPIN no es una clave única de proyecto en este conjunto.
- No se detectaron variantes de nombre, después de normalizar espacios, para el mismo código de entidad, sector, objetivo, programa, meta o proyecto.

El conjunto de 425 metas coincide exactamente con el Excel de Avance del Plan del mismo corte. Los 443 códigos distintos de indicador del PDF coinciden con los de la hoja Indicadores de ese Excel. Esta igualdad de cobertura no demuestra igualdad de avances agregados por entidad ni autoriza repetirlos en cada actividad.

## 5. Cómo leer las 78 columnas

La hoja `datos` contiene 23 campos de identificación y anualización, 25 de magnitudes físicas, 12 de recursos corrientes, 15 de recursos constantes y 3 campos de ODS/erratas. El diccionario describe exactamente los 78 campos: no faltan campos documentados ni sobran definiciones sin columna.

### Identificadores y contexto

Los códigos institucionales, sectoriales, de objetivo, programa, proyecto y BPIN deben tratarse como identificadores de texto. En particular, `0102`, `017` y `008199` contienen ceros iniciales significativos para la representación y los enlaces. No se debe aplicar la regla de “quitar ceros” usada para algunas columnas de Avance del Plan a estos campos. Los códigos de meta producto y actividad son numéricos en el original, pero tampoco son medidas sumables.

`VIGENCIA` es 2026 en todas las filas. `VIGENCIA_FECHA_SEGUIMIENTO` contiene `30/06/2026 (Terminado)`: se debe conservar el texto original y, si se necesita una fecha, derivarla separadamente. El estado entre paréntesis no es una fecha ni debe descartarse sin registro.

### Magnitudes y porcentajes

`MAGVIG1` = 2024, `MAGVIG2` = 2025, `MAGVIG3` = 2026 y `MAGVIG4` = 2027. Cada vigencia tiene programado, compromisos/contratado, porcentaje comprometido, entregado y porcentaje entregado. `MAGTOTAL` contiene los cinco totales equivalentes, pero el diccionario indica que esos totales se muestran solo para actividades de tipo Suma.

Los porcentajes de estas columnas están en escala **0–100**, a diferencia de las fracciones internas del Excel de Avance del Plan. Se puede crear su representación proporcional dividiendo por 100, conservando el dato original para auditar. El IPC ya es fracción: `0.052` representa 5,2 % y **no debe dividirse de nuevo**. Las magnitudes físicas no deben convertirse a fracción solo porque la descripción mencione “Porciento”: es necesario diferenciar una cantidad programada, como 100 unidades porcentuales, de una columna que expresa el cociente de cumplimiento.

### Recursos

`RECVIG`: recursos corrientes de cada año, en **millones de pesos**.

`RECKVIG`: recursos constantes de cada año, expresados en millones a precios de la vigencia del reporte, aquí **2026**.

`RECKTOTAL`: totales reportados en precios constantes 2026.

Las tres medidas financieras son programación, compromisos/contratado y giros; no existe una columna financiera separada de “entregado” en el Excel. “Entregado” corresponde a magnitudes físicas. No se deben confundir giros con avance físico o cumplimiento de resultados. La hoja IPC y el diccionario desaconsejan sumar recursos corrientes de años diferentes para comparaciones reales.

### Anualización

| Tipo de actividad | Filas | Interpretación para la limpieza |
| --- | --- | --- |
| Suma | 927 | Se pueden comprobar totales físicos entre vigencias |
| Constante | 553 | No sumar automáticamente magnitudes entre años; los totales físicos aparecen como cero estructural |
| Creciente | 169 | No sumar automáticamente magnitudes entre años; los totales físicos aparecen como cero estructural |
| Decreciente | 6 | No sumar automáticamente magnitudes entre años; los totales físicos aparecen como cero estructural |

Las 728 actividades no Suma tienen cero en los totales de magnitud aun cuando sus valores anuales sean positivos. No significan necesariamente falta de ejecución. Se debe conservar el cero de origen y agregar, si se necesita, una marca de aplicabilidad; convertirlo automáticamente en un avance cero interpretado como desempeño sería un error.

La página 264 del PDF explica el cálculo de avance **de los indicadores**:
- Suma: cocientes basados en sumas de ejecución y programación.
- Constante: la ejecución es independiente por vigencia y el reporte utiliza promedios para horizontes acumulados.
- Creciente sin línea base: utiliza la última ejecución reportada.
- Decreciente sin línea base: utiliza cocientes de programación sobre ejecución, en sentido inverso a un indicador creciente.
- Creciente/decreciente con línea base: utiliza diferencias respecto de la base o de la ejecución anterior y condiciones especiales. Deben conservarse esas reglas; no reemplazarlas por un cociente universal. La formulación impresa de decrecientes con línea base amerita validación metodológica antes de reproducirla.
- El porcentaje de giros del reporte se define como giros/programado.

Los tipos de indicador del PDF y los tipos de actividad del Excel pertenecen a niveles diferentes. Un indicador de tipo Suma no obliga a que todas sus actividades también lo sean. La codificación `(K)` significa Constante; `(C)` significa Creciente.

Los rangos gráficos del PDF son 0–50, >50–79, >79–90 y >90. Son categorías del reporte, no umbrales estadísticos aprendidos ni una definición universal de éxito. El pie de 263 páginas advierte que la programación/ejecución de magnitudes incorpora la vigencia actual y anterior según la desagregación de la entidad; no deben interpretarse como un panel de mediciones independientes sin revisar su alcance.

## 6. Diagnóstico de calidad del Excel

No hay filas exactamente duplicadas. No hay vacíos en las columnas analíticas principales; `FE_DE_ERRATAS` tiene 1.634 celdas vacías y 21 filas con referencia. Ese vacío significa que no hay una nota asociada en esa celda, no necesariamente un defecto de completitud. No se encontraron números negativos ni valores no numéricos en las 52 columnas de magnitudes y recursos.

Sí hay diferencias de espacios que afectan comparaciones y categorías:

| Columna | Celdas con espacios/Unicode por normalizar |
| --- | --- |
| PLAN_META_PRODUCTO_NOMBRE | 5 |
| PROYECTO_NOMBRE | 11 |
| PROYECTO_ACTIVIDAD_NOMBRE | 260 |
| PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION | 728 |
| ODS | 54 |

Las constantes del archivo no se deben eliminar automáticamente: incluyen el plan, la vigencia y ejecución cero de la vigencia futura 2027. Esas columnas documentan el horizonte del plan y pueden variar en otros cortes.

### Porcentajes superiores a 100 y cocientes incoherentes

Hay **284 celdas porcentuales superiores a 100** entre las diez columnas porcentuales; son celdas, no 284 actividades distintas. El máximo observado es 13.050 %: proyecto 008157, actividad 4, programado 2 y comprometido 261 en 2025. El cociente 261/2 × 100 reproduce 13.050. Es un valor extremo que exige interpretación de programación y unidad, pero no puede recortarse a 100 como una supuesta limpieza. En escala proporcional sería 130,5.

En `MAGVIG1_PORCCOMPROMISOS`, 1.653 de 1.655 filas tienen cero; solo dos valores son distintos de cero. Al contrastar con comprometido/programado en las filas con denominador no nulo, **1.317 casos no coinciden**. Ejemplo: proyecto 008199, actividad 1, programa 1 y compromete 1, pero el campo porcentual comprometido de 2024 contiene 0; el porcentaje entregado de esa misma fila es 100. La inconsistencia está en el original y no proviene de una conversión a CSV. No se debe sobrescribir el campo: si se requiere el cociente se agrega como variable calculada independiente, con una bandera de discrepancia.

En las columnas equivalentes de 2025, 2026 y 2027 y los totales físicos, los cocientes con denominador no nulo coinciden dentro de 0,00001 puntos porcentuales. Hay una discrepancia adicional en el porcentaje entregado de 2024. Los denominadores cero deben tratarse por separado: en 2026, Veeduría Distrital, proyecto 007981, actividad 10, tiene programado 0, entregado 3 y porcentaje entregado reportado 100, con la fe de erratas 7. No se debe convertir en cero ni provocar una división silenciosa por cero.

### Reglas financieras de orden

Se observan compromisos mayores que la programación en 1 fila de 2024, 276 de 2025 y 45 de 2026. En 2025 hay 11 filas con giros superiores a compromisos. Estos controles son alertas de coherencia; sin la historia de ajustes, desagregaciones y modificaciones no demuestran por sí solos un hecho irregular. No se corrigen por recorte o por intercambio de columnas.

## 7. IPC, sumas y conciliación interna

Los factores registrados son:

| Vigencia | IPC registrado | Operación | Factor |
| --- | --- | --- | --- |
| 2024 | 0.052 | Multiplica | 1.117213 |
| 2025 | 0.051 | Multiplica | 1.063 |
| 2026 | 0.063 | Multiplica | 1.0 |
| 2027 | 0.0425 | Divide | 1.0425 |

Se verificaron las 12 conversiones de columna: 4 vigencias × 3 medidas × 1.655 filas = **19.860 conversiones**. Coinciden con los factores declarados a una tolerancia absoluta de 0,0000001 millones. Por ejemplo, 2024 se multiplica por 1,117213, 2025 por 1,063, 2026 por 1 y 2027 se divide por 1,0425. El IPC de una fila no debe confundirse con su factor acumulado: el factor 2024 reproduce 1,051 × 1,063, no 1,052.

### Totales constantes reportados frente a suma de vigencias

Las columnas de recursos constantes contienen discrepancias entre el total reportado y la suma de los cuatro años: 1 caso en programado, 9 en compromisos y 13 en giros usando tolerancia 0,0000001 millones. Dos de los casos de giros son diferencias del orden de 0,000001 millones; los otros once son sustantivamente mayores. **Todas estas filas tienen fe de erratas.** Esto apoya una relación con ajustes, pero las notas no aportan todo el antes/después necesario para decidir qué valor debe prevalecer.

Caso relevante: Salud/FFDS, proyecto 007914, actividad 2, tiene programación anual de recursos cero en los cuatro años, pero un total constante programado de 28.336,78431 millones. Es un motivo para conservar tanto los valores anuales como el total reportado y registrar la diferencia, no para elegir silenciosamente uno.

En magnitudes de tipo Suma hay 1 discrepancia de programación total, 6 de compromiso total y 6 de entrega total frente a la suma de los años; todas se encuentran en filas con fe de erratas. Las 728 actividades no Suma quedan fuera de esa prueba aditiva.

### Tablas dinámicas

Se recalcularon las agregaciones de todas las filas de las seis hojas dinámicas, incluidos subtotales y totales generales: **681 filas agregadas**. Todas coinciden con las columnas `RECKTOTAL` de `datos` dentro de la tolerancia numérica. Las diferencias máximas son menores que 0,000001 millones y compatibles con acumulación de coma flotante. Se consideró que Excel agrupa variantes de capitalización/espacios de ODS.

Esta coincidencia significa que las tablas dinámicas reproducen la hoja de datos, **no que resuelvan sus inconsistencias entre totales y años**. Tampoco deben concatenarse sus filas como nuevos registros, porque se duplicaría la información y se sumarían niveles jerárquicos.

| Hoja dinámica | Filas agregadas comprobadas | Discrepancias |
| --- | --- | --- |
| TD_EjecPptalxSector | 17 | 0 |
| TD_EjecPptalxEntidad | 49 | 0 |
| TD_EjecPptalxOE | 6 | 0 |
| TD_EjecPptalxOEyProg | 45 | 0 |
| TD_EjecPptalxOEyProgMetaProd | 470 | 0 |
| TD_EjecPptalxODS_MetaODS | 94 | 0 |

## 8. Fe de erratas: parte sustantiva de la fuente

Las ocho notas están relacionadas con 21 actividades. Deben mantenerse como tabla de referencia, enlazadas por el número de nota y con el texto original completo. No constituyen instrucciones suficientemente detalladas para reescribir automáticamente valores.

La nota IDT menciona actividades 1 y 4 del proyecto 8107, pero en los datos también aparece la actividad 3 marcada con la nota 2. Es una diferencia de alcance entre la nota y el etiquetado que se debe registrar. La nota UD-FJC menciona “proyecto 8217 actividad 3”, mientras que su texto de radicado contiene `2026IE_195`; se conserva tal como está, sin reinterpretar el número del radicado como valor analítico.

| Nota | Filas relacionadas | Texto completo de la nota |
| --- | --- | --- |
| 1 | 7 | Radicado SDHT 2-2026-6747, SDP 1-2026-05672. Solicita ajuste de cifras de magnitudes y recursos no quedaron reflejados los reportes de seguimiento por parte de la entidad. |
| 2 | 3 | Radicado IDT 20261000001761, SDP 1-2026-06074. Solicita corrección de registro mediante fe de erratas para el proyecto 8107, actividades 1 y 4. |
| 3 | 2 | Radicados IPES 10-816-2026-001327 y 10-816-2026-001708, SDP 1-2026-05590 y 1-2026-08033. Solicita corrección de registro mediante fe de erratas para el proyecto 7927 actividad 3 y proyecto 7934 actividad 2. |
| 4 | 1 | Radicado SGRAL 2-2026-2540, SDP 1-2026-05978. Solicita corrección mediante fe de erratas para el proyecto 8098, actividad 3 |
| 5 | 2 | Radicado SDA 2026EE17877, SDP 1-2026-06054. Solicita corrección mediante fe de erratas para el proyecto 7961, actividades 1 y 2 |
| 6 | 4 | Radicado SDS/FFDS 2026-EE-15018, SDP 1-2026-09838. Solicita correción mediante fe de erratas para los proyectos 7914, 8124, 8141 y 8105. |
| 7 | 1 | Radicados VD 20261200007501 y 20261200021271, SDP  1-2026-09590 y 1-2026-14287. Solicita corrección mediante fe de erratas para el proyecto 7981 actividad 10. |
| 8 | 1 | Radicado UD-FJC OAP_2026IE_195, SDP 1-2026-18138. Solicita corrección mediante fe de erratas para el proyecto 8217 actividad 3. |

## 9. Qué aporta el PDF que no está en el Excel

El PDF sigue esta organización: entidad → objetivo → programa → meta producto → indicador y tabla de seguimiento → retrasos/soluciones y avances/logros → proyectos y actividades con recursos y magnitudes. No es una tabla plana ni una colección de un relato por proyecto.

- Las 48 entidades del Excel aparecen en el PDF y no se detectaron entidades adicionales.
- Se localizaron las mismas **1.655 claves entidad–proyecto–actividad**, sin faltantes ni adicionales después de manejar la continuación de páginas y los títulos divididos en varias líneas.
- Hay 591 combinaciones entidad–meta–proyecto en cada archivo; un mismo proyecto puede reaparecer bajo varias metas.
- El PDF contiene **517 bloques entidad–meta–indicador**, correspondientes a **443 códigos distintos**. Un indicador compartido puede tener varios reportantes; por ejemplo, el 4218 aparece seis veces. Es incorrecto borrar automáticamente esas apariciones como duplicados.
- En cada bloque se identificaron dos campos narrativos: **517 de retrasos/soluciones y 517 de avances/logros**, 1.034 campos y aproximadamente 1,13 millones de caracteres extraídos de sus áreas geométricas.
- El conteo literal de la frase “Avances y Logros” da 520 ocurrencias porque también aparece dentro de textos; no equivale al número de etiquetas ni al de registros.
- Los bloques de indicador se distribuyen en 294 Suma, 132 Constante, 85 Creciente y 6 Decreciente; son apariciones por entidad y meta, no necesariamente códigos de indicador únicos.
- La plantilla muestra un encabezado 2028, pero no se detectaron importes monetarios en esa banda; el Excel solo documenta 2024–2027. No se debe crear un quinto año de datos a partir de un encabezado vacío.

Los campos de indicadores (código, nombre, programación inicial del plan, programación actualizada, ejecución por año, porcentaje de vigencia, porcentaje transcurrido y porcentaje del plan) no están en la tabla `datos` de Gestión-Inversión. No debe afirmarse que faltan por limpieza: son un nivel adicional disponible en el PDF y, parcialmente, en el Excel de Avance del Plan.

## 10. Diagnóstico de los relatos y análisis cualitativo preliminar

Los relatos mezclan productos entregados, gestión contractual, obstáculos, soluciones propuestas, resultados de periodos anteriores y acciones futuras. Algunos citan cifras en pesos corrientes, porcentajes o billones dentro de la prosa; no debe asumirse que las unidades narrativas coinciden con los millones de las tablas. La extensión del relato no es una medida de desempeño.

Se detectaron 26 campos con marcadores literales breves: dos “0”, quince “NA”, siete “N/A” y dos “No aplica”. El indicador 4046 de Desarrollo Económico, página 48, tiene “0” en ambos campos narrativos. No se deben eliminar o convertir automáticamente todos estos valores en un mismo nulo: “no aplica”, “sin reporte” y “sin retrasos declarados” tienen significados diferentes. Las declaraciones de ausencia de retrasos son autorreportes, no verificaciones independientes.

Hay cuatro campos de avances/logros de exactamente 4.000 caracteres que terminan de manera abrupta:
- Página 9, indicador 4205, Secretaría General: termina en una palabra cortada.
- Página 43, indicador 4210, Movilidad: termina después de una coma.
- Página 73, indicador 3972, Planeación: termina en una frase incompleta.
- Página 189, indicador 3920, IDIPRON: termina en una cifra incompleta.

Los mismos finales se confirmaron con pypdf; el siguiente contenido es un encabezado de proyecto, no la continuación del relato. Es evidencia de truncamiento ya presente en el documento, probablemente relacionado con un límite de campo o de generación. No se puede determinar la causa exacta ni recuperar el texto faltante con estos dos archivos. La extracción y la limpieza no deben inventarlo.

### Temas observados y ejemplos localizables

Los siguientes ejemplos describen relatos de las entidades; no son comprobaciones externas de los hechos ni una clasificación exhaustiva validada. La taxonomía es una propuesta inicial que debe pasar por un protocolo de codificación.

| Tema | Referencia PDF | Contenido observado e implicación |
| --- | --- | --- |
| Contratación, personal y capacidad operativa | Salud/FFDS, p. 151, indicador 3946 | El texto vincula una baja temporal de visitas con contratación del personal de verificación. Separar causa declarada, efecto y medida correctiva. |
| Trámites, predios y licencias | Hábitat, p. 60, indicador 4187 | Se describen demoras de licenciamiento y revisión técnica de planes parciales. No reducirlo a una categoría genérica de bajo presupuesto. |
| Obras y condiciones externas | IPES, p. 135, indicador 3918 | Se mencionan estructuración contractual, lluvias y concertación con comerciantes. Un mismo relato puede requerir varias etiquetas. |
| Coordinación interinstitucional y suspensiones | IDU, p. 169, indicador 4142 | Se reporta suspensión de convenio y dependencia de decisiones de otras entidades. Diferenciar estado, causa y responsabilidad declarada. |
| Demanda y cambios de población | Educación, p. 32, indicador 4013 | Se atribuye menor matrícula a dinámicas demográficas y movilidad, y se describen estrategias de búsqueda. No inferir causalidad solo por la explicación institucional. |
| Tecnología y validaciones | Catastro, p. 221, indicador 4229 | La salida a producción depende de ajustes y pruebas adicionales. Diferenciar avance técnico de puesta en servicio. |
| Condiciones de operación | Transmilenio, p. 249, indicador 4144 | Se señalan energía, capacitación y condiciones de patios para entregas de flota. |
| Calendario de reporte | Secretaría General, p. 14, indicador 4249 | Declara no presentar retrasos y programar reportes para septiembre y diciembre. Cero avance de corte no equivale necesariamente a atraso. |
| Contenido ubicado en un campo poco informativo | Personería, p. 1, indicador 4727 | Retrasos/soluciones describe beneficios del marco metodológico sin identificar un retraso. El nombre del campo no basta para etiquetar su contenido. |
| Reprogramación y eventos futuros | Empresa Metro, p. 260, indicador 4148 | Menciona entrada en operación estimada para marzo de 2028. Una fecha futura narrativa no crea una vigencia adicional del Excel. |

Para la tesis conviene una codificación multietiqueta con: tipo de obstáculo, causa declarada, acción correctiva, estado de ejecución, entidad involucrada, periodo mencionado, población/territorio y evidencia textual. Debe conservarse el relato original y la página. Antes de entrenar o evaluar una clasificación se requiere una muestra anotada, definiciones operativas y revisión de desacuerdos. Búsquedas de palabras clave sirven para explorar, no para producir etiquetas definitivas ni medir desempeño automáticamente.

Los 517 bloques no proporcionan 517 observaciones independientes de avance del plan: algunos indicadores son compartidos. Tampoco se debe copiar un mismo relato sobre cada actividad y luego tratar esas copias como documentos independientes al entrenar un modelo.

## 11. Conciliación financiera PDF–Excel

Se contrastaron **7.534 importes monetarios explícitos** de las 591 filas de proyecto del PDF con agregados del Excel por entidad–meta–proyecto. Se consideró una tolerancia de `0.005 × (número de componentes + 1) + 0.000001` millones para comparar valores mostrados con dos decimales. Los componentes son el número de actividades para un año y cuatro veces ese número para los totales de cuatro años. La prueba no equipara celdas vacías con cero ni certifica todos los porcentajes y magnitudes del PDF.

### La base de precios explica la mayor parte de las diferencias aparentes

En la página 1, el proyecto 8201 muestra total programado 7.796,86 millones. Se reproduce como 822,87 + 1.465,00 + 3.301,14 + 2.207,85: suma de las cifras corrientes de 2024 a 2027. El total constante 2026 del Excel para ese grupo es aproximadamente 7.895,597788 millones. Son bases distintas; no corresponde declarar que falta dinero ni modificar el Excel para hacerlo coincidir con el PDF.

Por ello los totales del PDF se contrastaron contra sumas nominales de recursos corrientes **solo como prueba de reproducción del reporte**, no como recomendación analítica de sumar años sin ajustar precios. Para comparaciones interanuales se recomienda utilizar la base constante declarada y conservar los importes reportados originales.

### Resultado sobre la misma base

**7.504 importes coinciden dentro de la tolerancia; 30 quedan señalados para revisión, distribuidos en 9 grupos entidad–meta–proyecto. Todos pertenecen a grupos con filas marcadas con fe de erratas.** La tolerancia de un total de cuatro años contempla más componentes que la de una sola vigencia, para no clasificar diferencias de redondeo como errores de fuente.

Ejemplos sustantivos:
- Salud/FFDS, p. 154, proyecto 007914: programación 2025 de 26.657,37 en el PDF frente a 0 en el agregado anual del Excel; el grupo contiene la nota 6.
- IPES, p. 136, proyecto 007934: compromisos 2025 de 3.605,03 frente a 14.116,771905 millones del Excel; el grupo contiene la nota 3.
- Hábitat, p. 57, proyecto 008085: compromisos 2025 de 8.147,28 frente a 5.363,825331 millones del Excel; el grupo contiene la nota 1.

El patrón es compatible con diferencias de actualización o aplicación de erratas, pero las notas no permiten reconstruir completamente la secuencia. No se declara una fuente numérica “correcta” por tener una fecha de impresión posterior. El detalle completo de diferencias está en el anexo.

## 12. Lectura cuantitativa descriptiva del corte

Estas cifras resumen la fuente y no constituyen una evaluación de eficiencia, impacto o cumplimiento de metas físicas. Los cocientes financieros se calculan como razón de sumas, no como promedio simple de porcentajes de actividades.

| Vigencia | Programado corriente (millones) | Comprometido corriente (millones) | Giros corrientes (millones) |
| --- | --- | --- | --- |
| 2024 | 12,252,520.96 | 10,534,430.91 | 7,917,435.7 |
| 2025 | 36,147,008.6 | 33,924,846.046398 | 26,548,113.519121 |
| 2026 | 42,297,666.72 | 26,370,819.86545 | 11,418,301.370605 |
| 2027 | 31,984,527.33 | 0 | 0 |

Para 2026, el agregado de compromisos/programado es aproximadamente **62,35 %**, y giros/programado **27,00 %**. Es ejecución financiera frente a programación de la vigencia al corte de junio; no demuestra por sí sola retraso frente al cronograma ni cumplimiento de resultados.

La suma de los totales constantes reportados por actividad es: programado 125.119.551,100790 millones, compromisos 74.192.279,318377 millones y giros 48.479.803,577750 millones a precios de 2026. Estas sumas reproducen las tablas dinámicas; deben acompañarse de la advertencia sobre las filas con totales inconsistentes y no reemplazarse sin decisión metodológica por la suma anual recalculada.

## 13. Diseño recomendado para la siguiente etapa

1. Conservar los archivos de Bronce y mantener la fecha de corte en toda tabla derivada.
2. Preparar una tabla de actividades que preserve las 1.655 filas y los 78 campos originales. No eliminar columnas constantes, filas con erratas, porcentajes extremos ni campos opcionales.
3. Separar la normalización de representación de las correcciones sustantivas: espacios y tipos son reglas técnicas; cambiar totales o porcentajes oficiales requiere decisión y evidencia.
4. Conservar las ocho notas de erratas y las cuatro filas de IPC como tablas de referencia, con su contenido íntegro. No mezclar el índice de tablas dinámicas con registros de actividad.
5. Si se necesita un panel anual, derivarlo de forma reversible: 1.655 × 4 = 6.620 registros actividad–vigencia, conservando en una tabla aparte los totales reportados y su aplicabilidad. No asumir aditividad física de todos los tipos.
6. Extraer indicadores y relatos a su propio nivel entidad–meta–indicador–corte, conservando número de página, tipo de anualización y texto original. Preservar por separado programación inicial y programación actualizada.
7. Para unir relatos con actividades, usar una tabla puente sustentada en los bloques del PDF; no asumir una relación uno a uno. Evitar multiplicar recursos por el número de indicadores de una meta.
8. Estandarizar los códigos con sus longitudes y ceros de origen; no reutilizar sin cambios la normalización de códigos del notebook de Avance del Plan.
9. Convertir únicamente campos de porcentaje identificados a fracción, conservando su escala original y sin recortar valores mayores que 1. No tocar la escala del IPC ni de las magnitudes por coincidencias de nombre.
10. Agregar controles de conservación celda a celda y claves antes/después, además de conciliaciones de vigencias, porcentajes y fuentes. Las incidencias deben quedar visibles, sin reemplazar los originales.
11. Mantener los CSV derivados con delimitador `|`, comillas y UTF-8, según la convención ya adoptada. Los relatos pueden incluir puntos y coma, saltos y comillas, por lo que deben releerse para verificar la exportación.
12. No usar los relatos que mencionan explícitamente avance o porcentajes como predictores de ese mismo avance sin discutir fuga de información. Una partición aleatoria por actividad podría repartir copias del mismo relato entre entrenamiento y prueba; conviene particionar por grupo o por tiempo según el objetivo.

No se recomienda diseñar aún una tabla única que mezcle el avance físico de indicador, la entrega de actividad y la ejecución presupuestal: son niveles, unidades y denominadores diferentes.

## 14. Asuntos que siguen abiertos

- Resolver qué versión y qué criterio usar cuando valores anuales, totales y PDF discrepan en filas con erratas. Requiere aclaración de la fuente o documentos de corrección con valores antes/después.
- Determinar si el campo porcentual comprometido de 2024 es un error de exportación o responde a una regla no documentada. La inconsistencia aritmética está comprobada; su causa no.
- Recuperar los cuatro relatos cortados de una fuente alternativa, si existe; estos dos archivos no permiten completarlos.
- Definir si el objetivo de la tesis será descriptivo, predictivo o de clasificación cualitativa, y cuál es la unidad y horizonte temporal de estudio.
- Validar fichas y líneas base antes de recalcular indicadores crecientes/decrecientes; el Excel de actividades no proporciona todos esos insumos.
- Las afirmaciones de los relatos son autorreportadas. No se verificaron contra contratos, expedientes, sistemas misionales u otras evidencias administrativas.

## Anexo A. Mapa de las 48 entidades en el PDF

Este índice permite localizar los contenidos revisados sin mezclar las entidades ni confundir sus narrativas. La última página, 264, es metodológica.

| Entidad | Nombre de encabezado PDF | Páginas | Actividades Excel | Bloques de indicador |
| --- | --- | --- | --- | --- |
| 0102 | Personeria Distrital | 1–3 | 36 | 4 |
| 0104 | Secretaria General | 4–16 | 60 | 23 |
| 0105 | Veeduria Distrital | 17–18 | 18 | 2 |
| 0110 | Secretaria Distrital de Gobierno | 19–26 | 53 | 19 |
| 0111 | Secretaria Distrital de Hacienda | 27–29 | 25 | 4 |
| 0112 | Secretaria de Educacion del Distrito | 30–37 | 43 | 18 |
| 0113 | Secretaria Distrital de Movilidad | 38–45 | 69 | 15 |
| 0117 | Secretaria Distrital de Desarrollo Economico | 46–54 | 46 | 23 |
| 0118 | Secretaria Distrital de Habitat | 55–62 | 56 | 17 |
| 0119 | Secretaria Distrital de Cultura Recreacion y Deporte | 63–72 | 44 | 22 |
| 0120 | Secretaria Distrital de Planeacion | 73–82 | 49 | 19 |
| 0121 | Secretaria Distrital de la Mujer | 83–91 | 61 | 19 |
| 0122 | Secretaria Distrital de Integracion Social | 92–103 | 67 | 29 |
| 0125 | Departamento Administrativo del Servicio Civil Distrital | 104–105 | 14 | 1 |
| 0126 | Secretaria Distrital de Ambiente | 106–115 | 77 | 21 |
| 0127 | Departamento Administrativo de la Defensoria del Espacio Publico-DADEP | 116–118 | 20 | 5 |
| 0131 | Unidad Administrativa Especial Cuerpo Oficial de Bomberos | 119–120 | 21 | 2 |
| 0136 | Secretaria Juridica Distrital | 121–124 | 17 | 11 |
| 0137 | Secretaria Distrital de Seguridad, Convivencia y Justicia | 125–133 | 65 | 18 |
| 0200 | Instituto para la Economia Social | 134–139 | 26 | 11 |
| 0201 | Secretaria Distrital de Salud / Fondo Financiero Distrital de Salud | 140–157 | 91 | 34 |
| 0203 | Instituto Distrital de Gestion de Riesgos y Cambio Climatico | 158–161 | 25 | 8 |
| 0204 | Instituto de Desarrollo Urbano | 162–171 | 105 | 20 |
| 0206 | Fondo de Prestaciones Economicas Cesantias y Pensiones | 172–172 | 4 | 1 |
| 0208 | Caja de Vivienda Popular | 173–176 | 23 | 8 |
| 0211 | Instituto Distrital de Recreacion y Deporte | 177–182 | 44 | 12 |
| 0213 | Instituto Distrital de Patrimonio Cultural-IDPC | 183–188 | 17 | 10 |
| 0214 | Instituto Distrital para la Protección de la Niñez y la Juventud | 189–190 | 10 | 5 |
| 0215 | Fundacion Gilberto Alzate Avendaño | 191–195 | 18 | 10 |
| 0216 | Orquesta Filarmonica de Bogota | 196–197 | 13 | 6 |
| 0218 | Jardin Botanico - Jose Celestino Mutis | 198–202 | 50 | 10 |
| 0219 | Instituto para la Investigacion Educativa y el Desarrollo Pedagogico | 203–203 | 6 | 2 |
| 0220 | Instituto Distrital de la Participacion y Accion Comunal-IDPAC | 204–207 | 19 | 8 |
| 0221 | Instituto Distrital de Turismo | 208–212 | 28 | 9 |
| 0222 | Instituto Distrital de las Artes | 213–219 | 49 | 12 |
| 0226 | Unidad Administrativa Especial de Catastro Distrital-UAEC | 220–221 | 15 | 4 |
| 0227 | Unidad Administrativa Especial de Rehabilitacion y Mantenimiento Vial-UAERMV | 222–224 | 13 | 6 |
| 0228 | Unidad Administrativa Especial de Servicios Publicos-UAESP | 225–231 | 25 | 13 |
| 0229 | Instituto Distrital de Proteccion y Bienestar Animal | 232–236 | 24 | 9 |
| 0230 | Universidad Distrital Francisco Jose de Caldas | 237–239 | 45 | 1 |
| 0235 | Contraloria de Bogota | 240–241 | 25 | 1 |
| 0240 | Loteria de Bogota | 242–242 | 2 | 1 |
| 0260 | Canal Capital | 243–243 | 7 | 1 |
| 0262 | Empresa de Transporte del Tercer Milenio Transmilenio S.A | 244–251 | 24 | 22 |
| 0263 | Empresa de Renovación y Desarrollo Urbano | 252–254 | 15 | 4 |
| 0265 | Empresa de Acueducto y Alcantarillado de Bogota | 255–258 | 43 | 9 |
| 0266 | Empresa Metro de Bogota S.A | 259–261 | 22 | 5 |
| 0501 | Agencia Distrital para la Educación Superior, La Ciencia y la Tecnología | 262–263 | 26 | 3 |

## Anexo B. Perfil de los 78 campos originales

Los mínimos y máximos describen los datos originales, antes de cualquier limpieza. “Vacíos” incluye la celda no informada; no se contabiliza como vacío un cero. Las columnas de porcentaje conservan aquí la escala 0–100.

| Campo | Tipo según diccionario | Vacíos | Distintos | Mínimo | Máximo | Descripción |
| --- | --- | --- | --- | --- | --- | --- |
| PLAN_CODIGO | Numérico | 0 | 1 |  |  | Código del Plan de Desarrollo |
| PLAN_NOMBRE | Texto | 0 | 1 |  |  | Nombre del Plan de Desarrollo |
| PLAN_SIGLA | Texto | 0 | 1 |  |  | Sigla del Plan de Desarrollo |
| VIGENCIA | Numérico | 0 | 1 |  |  | Vigencia del reporte de información |
| VIGENCIA_FECHA_SEGUIMIENTO | Texto | 0 | 1 |  |  | Fecha del reporte de información |
| ENTIDAD_CODIGO | Texto | 0 | 48 |  |  | Código de la entidad (único para el Distrito) |
| ENTIDAD_NOMBRE | Texto | 0 | 48 |  |  | Nombre de la entidad |
| ENTIDAD_SIGLA | Texto | 0 | 48 |  |  | Sigla de la entidad |
| ENTIDAD_CLASIFICACION | Texto | 0 | 3 |  |  | Clasificación de la entidad |
| SECTOR_CODIGO | Texto | 0 | 16 |  |  | Código del sector administrativo |
| SECTOR_NOMBRE | Texto | 0 | 16 |  |  | Nombre del sector administrativo |
| PLAN_NIVEL1_CODIGO | Texto | 0 | 5 |  |  | Código del primer nivel del Plan Distrital de Desarrollo |
| PLAN_NIVEL1_NOMBRE | Texto | 0 | 5 |  |  | Nombre del primer nivel del Plan Distrital de Desarrollo |
| PLAN_NIVEL2_CODIGO | Texto | 0 | 39 |  |  | Código del segundo nivel del Plan Distrital de Desarrollo |
| PLAN_NIVEL2_NOMBRE | Texto | 0 | 39 |  |  | Nombre del segundo nivel del Plan Distrital de Desarrollo |
| PLAN_META_PRODUCTO_CODIGO | Numérico | 0 | 425 |  |  | Código de la meta producto del Plan Distrital de Desarrollo |
| PLAN_META_PRODUCTO_NOMBRE | Texto | 0 | 425 |  |  | Nombre de la meta producto del Plan Distrital de Desarrollo |
| PROYECTO_CODIGO | Texto | 0 | 353 |  |  | Código del proyecto de inversión (para Bogotá) |
| PROYECTO_BPIN | Texto | 0 | 352 |  |  | Código del proyecto de inversión (Banco de nación) |
| PROYECTO_NOMBRE | Texto | 0 | 353 |  |  | Nombre del proyecto de inversión |
| PROYECTO_ACTIVIDAD_CODIGO | Numérico | 0 | 61 |  |  | Código de la actividad |
| PROYECTO_ACTIVIDAD_NOMBRE | Texto | 0 | 1655 |  |  | Nombre de la actividad |
| PROYECTO_ACTIVIDAD_TIPO_ANUALIZACION | Texto | 0 | 4 |  |  | Tipo de anualización de la actividad |
| PROYECTO_ACTIVIDAD_MAGVIG1_PROGRAMADO | Numérico | 0 | 354 | 0 | 15,522,714 | Magnitud programada vigencia 1 |
| PROYECTO_ACTIVIDAD_MAGVIG1_COMPROMISOS_CONTRATADO | Numérico | 0 | 375 | 0 | 15,522,714 | Magnitud comprometida vigencia 1 |
| PROYECTO_ACTIVIDAD_MAGVIG1_PORCCOMPROMISOS | Numérico | 0 | 3 | 0 | 133.112583 | Porcentaje magnitud comprometida vigencia 1 |
| PROYECTO_ACTIVIDAD_MAGVIG1_ENTREGADO | Numérico | 0 | 421 | 0 | 15,522,714 | Magnitud entregada vigencia 1 |
| PROYECTO_ACTIVIDAD_MAGVIG1_PORCENTREGADO | Numérico | 0 | 148 | 0 | 10,000 | Porcentaje magnitud entregada vigencia 1 |
| PROYECTO_ACTIVIDAD_MAGVIG2_PROGRAMADO | Numérico | 0 | 452 | 0 | 29,522,897 | Magnitud programada vigencia 2 |
| PROYECTO_ACTIVIDAD_MAGVIG2_COMPROMISOS_CONTRATADO | Numérico | 0 | 517 | 0 | 29,522,897 | Magnitud comprometida vigencia 2 |
| PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS | Numérico | 0 | 160 | 0 | 13,050 | Porcentaje magnitud comprometida vigencia 2 |
| PROYECTO_ACTIVIDAD_MAGVIG2_ENTREGADO | Numérico | 0 | 528 | 0 | 29,522,897 | Magnitud entregada vigencia 2 |
| PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO | Numérico | 0 | 164 | 0 | 10,000 | Porcentaje magnitud entregada vigencia 2 |
| PROYECTO_ACTIVIDAD_MAGVIG3_PROGRAMADO | Numérico | 0 | 424 | 0 | 22,000,000 | Magnitud programada vigencia 3 |
| PROYECTO_ACTIVIDAD_MAGVIG3_COMPROMISOS_CONTRATADO | Numérico | 0 | 627 | 0 | 11,954,719 | Magnitud comprometida vigencia 3 |
| PROYECTO_ACTIVIDAD_MAGVIG3_PORCCOMPROMISOS | Numérico | 0 | 599 | 0 | 6,370 | Porcentaje magnitud comprometida vigencia 3 |
| PROYECTO_ACTIVIDAD_MAGVIG3_ENTREGADO | Numérico | 0 | 629 | 0 | 11,954,719 | Magnitud entregada vigencia 3 |
| PROYECTO_ACTIVIDAD_MAGVIG3_PORCENTREGADO | Numérico | 0 | 678 | 0 | 6,370 | Porcentaje magnitud entregada vigencia 3 |
| PROYECTO_ACTIVIDAD_MAGVIG4_PROGRAMADO | Numérico | 0 | 415 | 0 | 9,954,389 | Magnitud programada vigencia 4 |
| PROYECTO_ACTIVIDAD_MAGVIG4_COMPROMISOS_CONTRATADO | Numérico | 0 | 1 | 0 | 0 | Magnitud comprometida vigencia 4 |
| PROYECTO_ACTIVIDAD_MAGVIG4_PORCCOMPROMISOS | Numérico | 0 | 1 | 0 | 0 | Porcentaje magnitud comprometida vigencia 4 |
| PROYECTO_ACTIVIDAD_MAGVIG4_ENTREGADO | Numérico | 0 | 1 | 0 | 0 | Magnitud entregada vigencia 4 |
| PROYECTO_ACTIVIDAD_MAGVIG4_PORCENTREGADO | Numérico | 0 | 1 | 0 | 0 | Porcentaje magnitud entregada vigencia 4 |
| PROYECTO_ACTIVIDAD_MAGTOTAL_PROGRAMADO | Numérico | 0 | 322 | 0 | 77,000,000 | Total magnitud programada |
| PROYECTO_ACTIVIDAD_MAGTOTAL_COMPROMISOS_CONTRATADO | Numérico | 0 | 561 | 0 | 57,000,330 | Total magnitud comprometida |
| PROYECTO_ACTIVIDAD_MAGTOTAL_PORCCOMPROMISOS | Numérico | 0 | 536 | 0 | 7,525 | Total porcentaje magnitud comprometida |
| PROYECTO_ACTIVIDAD_MAGTOTAL_ENTREGADO | Numérico | 0 | 558 | 0 | 57,000,330 | Total magnitud entregada |
| PROYECTO_ACTIVIDAD_MAGTOTAL_PORCENTREGADO | Numérico | 0 | 551 | 0 | 7,525 | Total porcentaje magnitud entregada |
| PROYECTO_ACTIVIDAD_RECVIG1_PROGRAMADO | Numérico | 0 | 1364 | 0 | 2,022,767.86 | Recursos CORRIENTES programados de la vigencia 1 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG1_COMPROMISOS_CONTRATADO | Numérico | 0 | 1360 | 0 | 2,022,038.81 | Recursos CORRIENTES comprometidos de la vigencia 1 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG1_GIROS | Numérico | 0 | 1284 | 0 | 1,985,858.91 | Recursos CORRIENTES girados de la vigencia 1 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG2_PROGRAMADO | Numérico | 0 | 1486 | 0 | 4,044,806.97 | Recursos CORRIENTES programados de la vigencia 2 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG2_COMPROMISOS_CONTRATADO | Numérico | 0 | 1490 | 0 | 4,044,321.451625 | Recursos CORRIENTES comprometidos de la vigencia 2 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG2_GIROS | Numérico | 0 | 1457 | 0 | 4,034,519.80147 | Recursos CORRIENTES girados de la vigencia 2 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG3_PROGRAMADO | Numérico | 0 | 1471 | 0 | 5,275,343.57 | Recursos CORRIENTES programados de la vigencia 3 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG3_COMPROMISOS_CONTRATADO | Numérico | 0 | 1388 | 0 | 4,908,785.014848 | Recursos CORRIENTES comprometidos de la vigencia 3 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG3_GIROS | Numérico | 0 | 1341 | 0 | 2,073,542.209494 | Recursos CORRIENTES girados de la vigencia 3 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG4_PROGRAMADO | Numérico | 0 | 1311 | 0 | 5,490,657.71 | Recursos CORRIENTES programados de la vigencia 4 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG4_COMPROMISOS_CONTRATADO | Numérico | 0 | 1 | 0 | 0 | Recursos CORRIENTES comprometidos de la vigencia 4 (millones) |
| PROYECTO_ACTIVIDAD_RECVIG4_GIROS | Numérico | 0 | 1 | 0 | 0 | Recursos CORRIENTES girados de la vigencia 4 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG1_PROGRAMADO | Numérico | 0 | 1364 | 0 | 2,259,862.549174 | Recursos CONSTANTES programados de la vigencia 1 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG1_COMPROMISOS_CONTRATADO | Numérico | 0 | 1360 | 0 | 2,259,048.045037 | Recursos CONSTANTES comprometidos de la vigencia 1 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG1_GIROS | Numérico | 0 | 1284 | 0 | 2,218,627.390418 | Recursos CONSTANTES girados de la vigencia 1 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG2_PROGRAMADO | Numérico | 0 | 1486 | 0 | 4,299,629.80911 | Recursos CONSTANTES programados de la vigencia 2 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG2_COMPROMISOS_CONTRATADO | Numérico | 0 | 1490 | 0 | 4,299,113.703077 | Recursos CONSTANTES comprometidos de la vigencia 2 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG2_GIROS | Numérico | 0 | 1457 | 0 | 4,288,694.548963 | Recursos CONSTANTES girados de la vigencia 2 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG3_PROGRAMADO | Numérico | 0 | 1471 | 0 | 5,275,343.57 | Recursos CONSTANTES programados de la vigencia 3 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG3_COMPROMISOS_CONTRATADO | Numérico | 0 | 1388 | 0 | 4,908,785.014848 | Recursos CONSTANTES comprometidos de la vigencia 3 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG3_GIROS | Numérico | 0 | 1341 | 0 | 2,073,542.209494 | Recursos CONSTANTES girados de la vigencia 3 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG4_PROGRAMADO | Numérico | 0 | 1311 | 0 | 5,266,817.947242 | Recursos CONSTANTES programados de la vigencia 4 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG4_COMPROMISOS_CONTRATADO | Numérico | 0 | 1 | 0 | 0 | Recursos CONSTANTES comprometidos de la vigencia 4 (millones) |
| PROYECTO_ACTIVIDAD_RECKVIG4_GIROS | Numérico | 0 | 1 | 0 | 0 | Recursos CONSTANTES girados de la vigencia 4 (millones) |
| PROYECTO_ACTIVIDAD_RECKTOTAL_PROGRAMADO | Numérico | 0 | 1623 | 0 | 14,471,321.682409 | Total recursos CONSTANTES programados (millones) |
| PROYECTO_ACTIVIDAD_RECKTOTAL_COMPROMISOS_CONTRATADO | Numérico | 0 | 1552 | 0 | 8,302,554.434919 | Total recursos CONSTANTES comprometidos (millones) |
| PROYECTO_ACTIVIDAD_RECKTOTAL_GIROS | Numérico | 0 | 1518 | 0 | 8,235,880.430716 | Total recursos CONSTANTES girados (millones) |
| ODS | Texto | 0 | 18 |  |  | Objetivo de Desarrollo Sostenible - ODS |
| ODS_META | Texto | 0 | 77 |  |  | Meta del Objetivo de Desarrollo Sostenible - ODS |
| FE_DE_ERRATAS | Texto | 1634 | 9 |  |  | Numeración y entidad relacionada con la hoja FeDeErratas |

## Anexo C. Descuadres internos de recursos constantes

La diferencia es total reportado menos suma de las vigencias. Se conserva la identificación por fila Excel y la nota asociada. No se han aplicado correcciones.

| Fila Excel | Entidad | Proyecto | Actividad | Medida | Total reportado | Suma anual | Diferencia | Errata |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 879 | 0201 | 007914 | 2 | PROGRAMADO | 28,336.78431 | 0 | 28,336.78431 | 6 - SDS/FFDS |
| 52 | 0104 | 008098 | 3 | COMPROMISOS_CONTRATADO | 1,556.897257 | 1,554.947275 | 1.949982 | 4 - SGRAL |
| 362 | 0118 | 008084 | 2 | COMPROMISOS_CONTRATADO | 4,096.901935 | 2,914.883078 | 1,182.018857 | 1 - SDHT |
| 364 | 0118 | 008084 | 4 | COMPROMISOS_CONTRATADO | 2,265.749179 | 2,300.977286 | -35.228107 | 1 - SDHT |
| 365 | 0118 | 008084 | 5 | COMPROMISOS_CONTRATADO | 834.744314 | 839.770108 | -5.025794 | 1 - SDHT |
| 371 | 0118 | 008085 | 5 | COMPROMISOS_CONTRATADO | 7,998.401615 | 5,039.593151 | 2,958.808464 | 1 - SDHT |
| 852 | 0200 | 007934 | 2 | COMPROMISOS_CONTRATADO | 13,345.110128 | 24,519.095118 | -11,173.98499 | 3 - IPES |
| 1315 | 0221 | 008107 | 1 | COMPROMISOS_CONTRATADO | 514.233514 | 880.670874 | -366.43736 | 2 - IDT |
| 1317 | 0221 | 008107 | 3 | COMPROMISOS_CONTRATADO | 455.154673 | 450.902673 | 4.252 | 2 - IDT |
| 1318 | 0221 | 008107 | 4 | COMPROMISOS_CONTRATADO | 2,785.375179 | 5,206.782879 | -2,421.4077 | 2 - IDT |
| 52 | 0104 | 008098 | 3 | GIROS | 1,006.82998 | 794.19422 | 212.63576 | 4 - SGRAL |
| 361 | 0118 | 008084 | 1 | GIROS | 8,504.58093 | 6,314.945354 | 2,189.635576 | 1 - SDHT |
| 362 | 0118 | 008084 | 2 | GIROS | 3,679.822705 | 2,497.803848 | 1,182.018857 | 1 - SDHT |
| 364 | 0118 | 008084 | 4 | GIROS | 1,425.43291 | 1,830.466537 | -405.033627 | 1 - SDHT |
| 365 | 0118 | 008084 | 5 | GIROS | 573.904386 | 578.930169 | -5.025783 | 1 - SDHT |
| 371 | 0118 | 008085 | 5 | GIROS | 3,023.583055 | 1,938.995741 | 1,084.587314 | 1 - SDHT |
| 388 | 0118 | 008125 | 3 | GIROS | 397.908262 | 509.438363 | -111.530101 | 1 - SDHT |
| 392 | 0118 | 008125 | 7 | GIROS | 315.452046 | 392.677784 | -77.225738 | 1 - SDHT |
| 643 | 0126 | 007961 | 1 | GIROS | 6,245.083988 | 6,245.083989 | -0.000001 | 5 - SDA |
| 644 | 0126 | 007961 | 2 | GIROS | 3,765.57921 | 3,765.579209 | 0.000001 | 5 - SDA |
| 852 | 0200 | 007934 | 2 | GIROS | 7,549.524866 | 13,755.132668 | -6,205.607802 | 3 - IPES |
| 1315 | 0221 | 008107 | 1 | GIROS | 458.140387 | 548.782397 | -90.64201 | 2 - IDT |
| 1318 | 0221 | 008107 | 4 | GIROS | 2,108.188385 | 4,486.555215 | -2,378.36683 | 2 - IDT |

## Anexo D. Diferencias financieras PDF–Excel después de armonizar la base

Las etiquetas TOTAL corresponden al total del PDF comparado con suma corriente anual del Excel para reproducir el reporte, **no** con RECKTOTAL. Los importes están en millones. La tolerancia contempla el número de actividades. La presencia de una nota en el grupo no demuestra que explique exactamente cada diferencia.

| Página | Entidad | Meta | Proyecto | Medida PDF | Importe PDF | Comparable Excel | Diferencia | Tolerancia | Notas en el grupo |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 10 | 0104 | 2302 | 008098 | RECVIG2_COMPROMISOS_CONTRATADO | 11,203.97 | 11,202.135553 | 1.834447 | 0.045001 | 4 - SGRAL |
| 10 | 0104 | 2302 | 008098 | RECVIG2_GIROS | 10,025.47 | 9,825.434354 | 200.035646 | 0.045001 | 4 - SGRAL |
| 10 | 0104 | 2302 | 008098 | TOTAL_PDF_COMPROMISOS_CONTRATADO | 23,591.12 | 23,589.285763 | 1.834237 | 0.165001 | 4 - SGRAL |
| 10 | 0104 | 2302 | 008098 | TOTAL_PDF_GIROS | 15,648.7 | 15,448.650994 | 200.049006 | 0.165001 | 4 - SGRAL |
| 57 | 0118 | 2164 | 008085 | RECVIG2_COMPROMISOS_CONTRATADO | 8,147.28 | 5,363.825331 | 2,783.454669 | 0.030001 | 1 - SDHT |
| 57 | 0118 | 2164 | 008085 | RECVIG2_GIROS | 3,946.05 | 2,925.738492 | 1,020.311508 | 0.030001 | 1 - SDHT |
| 57 | 0118 | 2164 | 008085 | TOTAL_PDF_COMPROMISOS_CONTRATADO | 13,059.63 | 10,276.189654 | 2,783.440346 | 0.105001 | 1 - SDHT |
| 57 | 0118 | 2164 | 008085 | TOTAL_PDF_GIROS | 6,891.41 | 5,871.09764 | 1,020.31236 | 0.105001 | 1 - SDHT |
| 60 | 0118 | 2261 | 008084 | RECVIG2_COMPROMISOS_CONTRATADO | 1,767.38 | 1,805.252521 | -37.872521 | 0.025001 | 1 - SDHT |
| 60 | 0118 | 2261 | 008084 | RECVIG2_GIROS | 1,419.5 | 1,805.252512 | -385.752512 | 0.025001 | 1 - SDHT |
| 60 | 0118 | 2261 | 008084 | TOTAL_PDF_COMPROMISOS_CONTRATADO | 4,103.29 | 4,141.164133 | -37.874133 | 0.085001 | 1 - SDHT |
| 60 | 0118 | 2261 | 008084 | TOTAL_PDF_GIROS | 2,695.29 | 3,081.043384 | -385.753384 | 0.085001 | 1 - SDHT |
| 60 | 0118 | 2263 | 008084 | RECVIG2_GIROS | 5,385.63 | 3,325.76346 | 2,059.86654 | 0.010001 | 1 - SDHT |
| 60 | 0118 | 2263 | 008084 | TOTAL_PDF_GIROS | 8,009.34 | 5,949.47036 | 2,059.86964 | 0.025001 | 1 - SDHT |
| 61 | 0118 | 2267 | 008084 | RECVIG2_COMPROMISOS_CONTRATADO | 2,223.93 | 1,111.965058 | 1,111.964942 | 0.010001 | 1 - SDHT |
| 61 | 0118 | 2267 | 008084 | RECVIG2_GIROS | 2,223.93 | 1,111.965058 | 1,111.964942 | 0.010001 | 1 - SDHT |
| 61 | 0118 | 2267 | 008084 | TOTAL_PDF_COMPROMISOS_CONTRATADO | 3,829.2 | 2,717.231208 | 1,111.968792 | 0.025001 | 1 - SDHT |
| 61 | 0118 | 2267 | 008084 | TOTAL_PDF_GIROS | 3,424.19 | 2,312.220228 | 1,111.969772 | 0.025001 | 1 - SDHT |
| 61 | 0118 | 2279 | 008125 | RECVIG2_GIROS | 2,921.68 | 3,099.253882 | -177.573882 | 0.045001 | 1 - SDHT |
| 61 | 0118 | 2279 | 008125 | TOTAL_PDF_GIROS | 5,234.72 | 5,412.280645 | -177.560645 | 0.165001 | 1 - SDHT |
| 136 | 0200 | 2137 | 007934 | RECVIG2_COMPROMISOS_CONTRATADO | 3,605.03 | 14,116.771905 | -10,511.741905 | 0.015001 | 3 - IPES |
| 136 | 0200 | 2137 | 007934 | RECVIG2_GIROS | 3,555.56 | 9,393.38169 | -5,837.82169 | 0.015001 | 3 - IPES |
| 136 | 0200 | 2137 | 007934 | TOTAL_PDF_COMPROMISOS_CONTRATADO | 15,263.23 | 25,774.968394 | -10,511.738394 | 0.045001 | 3 - IPES |
| 136 | 0200 | 2137 | 007934 | TOTAL_PDF_GIROS | 8,962.63 | 14,800.449334 | -5,837.819334 | 0.045001 | 3 - IPES |
| 154 | 0201 | 2255 | 007914 | RECVIG2_PROGRAMADO | 26,657.37 | 0 | 26,657.37 | 0.015001 | 6 - SDS/FFDS |
| 154 | 0201 | 2255 | 007914 | TOTAL_PDF_PROGRAMADO | 147,597.42 | 120,940.05 | 26,657.37 | 0.045001 | 6 - SDS/FFDS |
| 211 | 0221 | 2297 | 008107 | RECVIG2_COMPROMISOS_CONTRATADO | 1,887.57 | 4,506.187631 | -2,618.617631 | 0.025001 | 2 - IDT |
| 211 | 0221 | 2297 | 008107 | RECVIG2_GIROS | 1,292.39 | 3,615.066581 | -2,322.676581 | 0.025001 | 2 - IDT |
| 211 | 0221 | 2297 | 008107 | TOTAL_PDF_COMPROMISOS_CONTRATADO | 6,261.87 | 8,880.473202 | -2,618.603202 | 0.085001 | 2 - IDT |
| 211 | 0221 | 2297 | 008107 | TOTAL_PDF_GIROS | 4,542.4 | 6,865.088052 | -2,322.688052 | 0.085001 | 2 - IDT |

## Anexo E. Verificación aritmética de porcentajes físicos

La prueba usa 100 × magnitud/programación solamente donde el denominador es distinto de cero. La tolerancia es 0,00001 puntos porcentuales. Un desacuerdo es una incidencia respecto de esa fórmula, no una autorización para reemplazar el valor.

| Periodo | Porcentaje | Denominador cero | Numerador no cero con denominador cero | Desacuerdos de cociente | Valores >100 |
| --- | --- | --- | --- | --- | --- |
| VIG1 | PORCCOMPROMISOS | 282 | 14 | 1317 | 1 |
| VIG1 | PORCENTREGADO | 282 | 3 | 1 | 67 |
| VIG2 | PORCCOMPROMISOS | 174 | 8 | 0 | 83 |
| VIG2 | PORCENTREGADO | 174 | 2 | 0 | 48 |
| VIG3 | PORCCOMPROMISOS | 134 | 0 | 0 | 31 |
| VIG3 | PORCENTREGADO | 134 | 1 | 0 | 19 |
| VIG4 | PORCCOMPROMISOS | 170 | 0 | 0 | 0 |
| VIG4 | PORCENTREGADO | 170 | 0 | 0 | 0 |
| TOTAL | PORCCOMPROMISOS | 737 | 0 | 0 | 27 |
| TOTAL | PORCENTREGADO | 737 | 0 | 0 | 8 |

## 15. Ampliación: anualización, unidades y organización de Silver

Esta sección incorpora la explicación del investigador sobre Suma, Constante, Creciente y Decreciente, contrastada con las actividades del Excel. No se aplicaron transformaciones a los originales ni se creó todavía la capa Silver de Gestión-Inversión.

### 15.1. Tres dimensiones distintas

La unidad de medida indica qué representa la magnitud: personas, documentos, estrategias, minutos, tasas, porcentajes u otra unidad. El tipo de anualización indica cómo se interpreta su trayectoria entre vigencias. El porcentaje de cumplimiento expresa una relación entre ejecución y referencia. No se puede deducir una dimensión a partir de otra ni del tamaño del número.

Una cifra 150 puede ser 150 personas, una tasa de 150 por una población de referencia o 150 % de cumplimiento. Dividirla por 100 o recortarla sin identificar el campo y su unidad destruiría su significado.

### 15.2. Interpretación por tipo

| Tipo | Interpretación temporal | Operación admisible | Precaución |
|---|---|---|---|
| Suma | Entregas aditivas entre vigencias de la misma actividad | Sumar entregas del horizonte observado y compararlas con la programación correspondiente | No sumar diferentes actividades con unidades distintas ni agregar entregado y comprometido como cantidades independientes |
| Constante | Cumplimiento que se renueva o evalúa por vigencia | Comparar ejecución de una vigencia con su propia programación; para acumulados, seguir la regla oficial específica | No sumar cuatro programaciones de 100 % para crear una meta de 400 %. Puede estar medida en cantidades diferentes de porcentajes |
| Creciente | Hitos o niveles que incorporan lo alcanzado antes | Usar el último nivel; calcular diferencias entre observaciones comparables cuando se requiera variación | No sumar los niveles de vigencias sucesivas ni confundir una variación con la ejecución original |
| Decreciente | Nivel o resultado cuya trayectoria prevista es descendente | Usar el nivel observado y su referencia; seguir la fórmula oficial de avance | No es necesariamente un stock acumulado: puede ser una tasa, prevalencia o duración. No imponer un cociente creciente |

El ejemplo Suma produce 56,27 unidades entregadas sobre 100 programadas para todo el plan: 56,27 % del plan. Frente a los hitos programados hasta 2026 (15 + 35 + 15 = 65), el cociente sería 56,27/65 = 86,5692 %. Ambos responden a denominadores distintos; ninguno demuestra por sí solo cumplimiento del cronograma al día de junio. Las cifras del ejemplo son ilustrativas del investigador y no un registro adicional del archivo.

En una actividad constante programada en 100 para 2026, entregar 38,98 unidades porcentuales equivale también a 38,98 % de cumplimiento anual. Esa igualdad numérica depende de que el denominador sea 100. Con programación 80 y entrega 40, el cumplimiento es 50 %, no 40 %. El ejemplo de 38,98 existe en el archivo: la actividad «Implementar 100 Porciento de las actividades de seguimiento y control [...] MIPG», clasificada Constante, registra programado 100, entregado 38,98 y porcentaje entregado 38,98 en 2026.

Para un creciente con niveles programados 15, 51, 87 y 120, el objetivo final es 120; 87 es un hito de 2026, no una entrega adicional a los años anteriores. Una diferencia de niveles puede expresar nuevas unidades, pero si los niveles son porcentajes su diferencia se expresa en puntos porcentuales. Si hay revisiones, cambios de definición o reducción de cobertura, una diferencia negativa es una incidencia para examinar, no un valor que deba sustituirse automáticamente por cero.

En el Sector Seguridad, Convivencia y Justicia (014), las 86 actividades se distribuyen en 24 Suma, 40 Constante y 22 Creciente, sin Decreciente. En el conjunto distrital sí existen seis Decreciente. Sus descripciones incluyen tiempo promedio de atención, prevalencias de nutrición, tasa de morbilidad y beneficiarios con renovación de matrícula; por tanto, no corresponde describirlos todos como un mismo stock. La ausencia sectorial no autoriza eliminar la categoría del esquema del proyecto.

### 15.3. Cómo distinguir porcentajes y cantidades sin perder información

1. **Porcentajes de cumplimiento identificados por esquema:** las diez columnas MAG cuyo sufijo es PORCCOMPROMISOS o PORCENTREGADO contienen porcentajes en escala 0–100. Para análisis pueden tener campos derivados en escala proporcional (`valor_original / 100`). Un cumplimiento 150 % se convierte en 1,5, no en 1. El valor fuente debe seguir disponible. La incoherencia del porcentaje comprometido 2024 se conserva y se marca; dividir por 100 no la resuelve.
2. **Magnitudes físicas:** PROGRAMADO, COMPROMISOS_CONTRATADO y ENTREGADO de MAG conservan la unidad de cada actividad. No se dividen automáticamente por 100. Cuando la unidad sea explícitamente Porciento se puede agregar una representación proporcional, pero debe distinguirse de la fracción de cumplimiento y documentarse la unidad resultante.
3. **Recursos:** REC y RECK se mantienen en millones de pesos y con su base de precios. No se normalizan como porcentajes. Si se calculan ratios financieros, deben tener campos y denominadores propios.
4. **IPC:** sus tasas ya se expresan como fracciones. Factor y tasa son columnas distintas; no se escalan de nuevo.
5. **Unidad de magnitud:** el Excel no trae una columna independiente de unidad de medida. Extraer palabras de la descripción puede generar candidatos, pero no prueba su validez. Se recomienda un catálogo revisable con `unidad_medida`, `naturaleza_magnitud` (cantidad, porcentaje, tasa, duración u otra), `estado_validacion_unidad` y referencia al texto de origen. Una unidad no resuelta debe permanecer identificada como pendiente; no inferirse solamente de que el número sea menor o mayor que 100.

Ejemplo real de por qué no basta el tipo: entre las 553 actividades Constante hay programaciones anuales de 1 estrategia, 1 plan y 4 canales. Además, 6 tienen magnitud programada 2026 superior a 100. Esas cifras no son automáticamente porcentajes de cumplimiento superiores a 100.

### 15.4. No separar las bases maestras por tipo de anualización

Se recomienda conservar una sola tabla maestra de actividades con la columna de tipo normalizada y documentada. Crear cuatro Excel independientes introduciría mantenimiento repetido, favorecería divergencias de esquema y no separaría por sí mismo cantidades de porcentajes. Las vistas o filtros por tipo son útiles para examinar reglas, no necesitan convertirse en cuatro bases maestras.

Las separaciones útiles responden a **unidades de observación diferentes**:
- Actividad y atributos: una fila por actividad y corte.
- Vigencia y mediciones: tabla anual derivada, si se necesita, con una fila por actividad–vigencia–corte. Solo debe crearse cuando aporte valor al análisis; no sustituye los totales reportados del original.
- Indicador y narrativa: registros del PDF por entidad–meta–indicador–corte, con procedencia y página.
- Tablas de referencia: diccionario, IPC y notas de erratas.

Si se deriva la tabla anual, tendrá 6.620 filas para las cuatro vigencias del corte actual. Los totales reportados permanecen separados de los años, porque son otro horizonte y algunos no coinciden con las sumas. Se debe poder reconstruir los valores originales; mantener cero de origen no obliga a interpretarlo como desempeño cero. La ejecución cero de un año futuro tampoco debe evaluarse como incumplimiento actual.

### 15.5. Qué hojas conviene llevar a Silver

| Contenido | Recomendación | Motivo |
|---|---|---|
| datos | Conservar íntegro en `gestion_inversion_actividades.csv` | Es la base maestra de 1.655 filas y 78 campos originales; las variables derivadas deben identificarse |
| DiccionarioDeDatos | Conservar los 78 campos, sus descripciones y observaciones en `diccionario_gestion_inversion.csv`; incorporar las notas introductorias en la documentación del notebook | Una exportación que solo tome la primera fila no conservaría su estructura ni las notas |
| IPC | Conservar las cuatro vigencias, tasas, operaciones y factores en `ipc_gestion_inversion.csv` | Permite interpretar y auditar precios constantes, sin recalcular factores de memoria |
| FeDeErratas | Conservar las ocho notas completas en `fe_erratas_gestion_inversion.csv` | Sustenta incidencias que no pueden resolverse con datos y diccionario solamente |
| Seis tablas dinámicas | No duplicarlas como bases maestras; utilizarlas como controles de conciliación en el notebook | Sus valores se pueden reproducir desde datos; contienen agregados y subtotales que duplicarían cifras si se concatenan |
| TablasDinamicasIncluidas | Mantener en el original y describir su función en el notebook | Es un índice de consulta, no observaciones adicionales |
| PDF: indicadores y relatos | Extraer en una fase separada si el objetivo de la tesis incorpora la información cualitativa o el seguimiento detallado de indicadores | Aporta contenido que no existe en datos; exige conservar texto, contexto, página y relaciones |

La salida mínima recomendada del **Excel** son cuatro CSV, no dos. No es obligatorio convertir cada hoja de consulta a un CSV si los originales se conservan y el alcance queda documentado. Omitir una tabla dinámica reproducible en Silver es diferente de eliminar datos primarios. En cambio, descartar IPC o erratas perdería información necesaria para interpretar y controlar la base.

Si la tesis mantiene el componente cualitativo previsto, el PDF no puede omitirse por considerar suficiente el Excel. Las narrativas y los cuadros de indicador deben tener su propio proceso de extracción; no es necesario resolver ambos procesos en una única tabla o en la misma celda de código.

### 15.6. Orden propuesto de implementación

Primero, preparar las cuatro tablas del Excel con conservación de valores y controles de identidad, unidades, anualización, porcentajes, IPC y erratas. Segundo, validar la extracción y asignación de los bloques del PDF, conservando los relatos sin completar los textos cortados. Tercero, definir la tabla analítica de la tesis y sus enlaces, según la pregunta de investigación. Este orden permite avanzar con datos verificables sin decidir antes de tiempo un agregado físico universal ni una corrección sustantiva no respaldada.

## 16. Contraste ampliado de magnitudes físicas y porcentajes con el PDF

Se contrastaron las **1.655 filas de actividad** mediante coordenadas del PDF y la clave entidad–proyecto–actividad, incluidas las continuaciones entre páginas. Los 1.655 tipos de anualización impresos coinciden con los del Excel después de normalizar espacios.

Se compararon **28,141 celdas numéricas explícitas** de magnitud (cantidades y porcentajes), de las cuales 10,290 llevan signo % en el PDF. Las posiciones se validaron contra los encabezados y contra la presencia o ausencia del signo %. Se usó tolerancia de 0,0050001 para admitir redondeo a dos decimales. Las celdas en blanco no se transformaron en cero ni se consideran valores contrastados.

Resultado: 26,775 coincidencias dentro de tolerancia y 1,366 discrepancias. Esto compara los valores publicados y su presentación; no reemplaza una validación externa de las magnitudes ni prueba que cada unidad inferida de la descripción esté correctamente definida.

| Columna | Celdas discrepantes |
| --- | --- |
| PROYECTO_ACTIVIDAD_MAGTOTAL_COMPROMISOS_CONTRATADO | 2 |
| PROYECTO_ACTIVIDAD_MAGTOTAL_ENTREGADO | 1 |
| PROYECTO_ACTIVIDAD_MAGTOTAL_PORCCOMPROMISOS | 2 |
| PROYECTO_ACTIVIDAD_MAGTOTAL_PORCENTREGADO | 1 |
| PROYECTO_ACTIVIDAD_MAGVIG1_COMPROMISOS_CONTRATADO | 2 |
| PROYECTO_ACTIVIDAD_MAGVIG1_ENTREGADO | 2 |
| PROYECTO_ACTIVIDAD_MAGVIG1_PORCCOMPROMISOS | 1318 |
| PROYECTO_ACTIVIDAD_MAGVIG1_PORCENTREGADO | 2 |
| PROYECTO_ACTIVIDAD_MAGVIG1_PROGRAMADO | 1 |
| PROYECTO_ACTIVIDAD_MAGVIG2_COMPROMISOS_CONTRATADO | 7 |
| PROYECTO_ACTIVIDAD_MAGVIG2_ENTREGADO | 8 |
| PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS | 7 |
| PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO | 7 |
| PROYECTO_ACTIVIDAD_MAGVIG2_PROGRAMADO | 2 |
| PROYECTO_ACTIVIDAD_MAGVIG3_COMPROMISOS_CONTRATADO | 1 |
| PROYECTO_ACTIVIDAD_MAGVIG3_ENTREGADO | 1 |
| PROYECTO_ACTIVIDAD_MAGVIG3_PORCCOMPROMISOS | 1 |
| PROYECTO_ACTIVIDAD_MAGVIG3_PROGRAMADO | 1 |

### Porcentaje comprometido de 2024

De las 1,318 discrepancias observadas en esa columna, el porcentaje del PDF reproduce comprometido/programado × 100 en **1,317 casos**, usando los valores del Excel y admitiendo el redondeo del reporte. Esto aporta evidencia adicional para distinguir una cantidad de su porcentaje y para localizar un problema de representación o cálculo en el Excel. No se aplicó ninguna sustitución automática.

Ejemplo verificable: Personería, página 1, proyecto 008201, actividad 1: programado 3.300, comprometido 3.381; el PDF imprime 102,45 % y el Excel almacena 0 en MAGVIG1_PORCCOMPROMISOS. El cociente es 102,454545... %. Aquí el PDF y el cociente respaldan un valor derivado de cumplimiento, pero el dato original debe conservarse y la procedencia de cualquier campo corregido debe registrarse.

### Regla de decisión propuesta

1. Conservar siempre el valor original de Excel.
2. Registrar valor publicado en PDF, página y coordenadas o ubicación de tabla cuando se use como evidencia.
3. Calcular un valor de control solo cuando se conoce el denominador y la fórmula aplicable.
4. Distinguir coincidencia por redondeo, discrepancia respaldada por fórmula, discrepancia con errata y caso no resuelto.
5. Si se autoriza una corrección analítica, guardarla como campo derivado con regla y fuente. No sustituir silenciosamente el original ni asumir que todos los campos del PDF prevalecen.
6. Para unidades físicas como Porciento, tasas, minutos o personas, verificar además la descripción. Un valor impreso sin signo % puede ser una magnitud cuya unidad es porcentaje; el formato del PDF ayuda, pero no basta para construir el catálogo de unidades.

### Muestra de otras discrepancias

| Página | Proyecto | Actividad | Columna | PDF | Excel | Errata |
| --- | --- | --- | --- | --- | --- | --- |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG1_PROGRAMADO | 4.0 | 3.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG1_COMPROMISOS_CONTRATADO | 4.0 | 3.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG1_ENTREGADO | 3.0 | 0.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG1_PORCENTREGADO | 75.0 | 0.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG2_PROGRAMADO | 0.0 | 3.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG2_ENTREGADO | 1.0 | 3.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG3_PROGRAMADO | 2.0 | 0.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG3_COMPROMISOS_CONTRATADO | 2.0 | 0.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG3_PORCCOMPROMISOS | 100.0 | 0.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGVIG3_ENTREGADO | 2.0 | 3.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGTOTAL_COMPROMISOS_CONTRATADO | 9.0 | 6.0 | 7 - VD |
| 18 | 007981 | 10 | PROYECTO_ACTIVIDAD_MAGTOTAL_PORCCOMPROMISOS | 150.0 | 100.0 | 7 - VD |
| 57 | 008085 | 5 | PROYECTO_ACTIVIDAD_MAGVIG2_COMPROMISOS_CONTRATADO | 23262.64 | 11631.32 | 1 - SDHT |
| 57 | 008085 | 5 | PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS | 200.0 | 100.0 | 1 - SDHT |
| 57 | 008085 | 5 | PROYECTO_ACTIVIDAD_MAGVIG2_ENTREGADO | 23262.64 | 11631.32 | 1 - SDHT |
| 57 | 008085 | 5 | PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO | 200.0 | 100.0 | 1 - SDHT |
| 60 | 008084 | 4 | PROYECTO_ACTIVIDAD_MAGVIG2_PROGRAMADO | 6.0 | 11.0 | 1 - SDHT |
| 61 | 008084 | 2 | PROYECTO_ACTIVIDAD_MAGVIG2_COMPROMISOS_CONTRATADO | 44510.0 | 22255.0 | 1 - SDHT |
| 61 | 008084 | 2 | PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS | 200.0 | 100.0 | 1 - SDHT |
| 61 | 008084 | 2 | PROYECTO_ACTIVIDAD_MAGVIG2_ENTREGADO | 44510.0 | 22255.0 | 1 - SDHT |
| 61 | 008084 | 2 | PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO | 200.0 | 100.0 | 1 - SDHT |
| 137 | 007934 | 2 | PROYECTO_ACTIVIDAD_MAGVIG1_COMPROMISOS_CONTRATADO | 755.0 | 1005.0 | 3 - IPES |
| 137 | 007934 | 2 | PROYECTO_ACTIVIDAD_MAGVIG1_ENTREGADO | 755.0 | 1005.0 | 3 - IPES |
| 137 | 007934 | 2 | PROYECTO_ACTIVIDAD_MAGVIG1_PORCENTREGADO | 100.0 | 755.0 | 3 - IPES |
| 137 | 007927 | 3 | PROYECTO_ACTIVIDAD_MAGVIG2_COMPROMISOS_CONTRATADO | 5.03 | 6.0 | 3 - IPES |
| 137 | 007927 | 3 | PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS | 83.83 | 100.0 | 3 - IPES |
| 137 | 007927 | 3 | PROYECTO_ACTIVIDAD_MAGVIG2_ENTREGADO | 5.03 | 6.0 | 3 - IPES |
| 137 | 007927 | 3 | PROYECTO_ACTIVIDAD_MAGVIG2_PORCENTREGADO | 83.83 | 100.0 | 3 - IPES |
| 147 | 008141 | 37 | PROYECTO_ACTIVIDAD_MAGVIG2_COMPROMISOS_CONTRATADO | 100.0 | 1.0 | 6 - SDS/FFDS |
| 147 | 008141 | 37 | PROYECTO_ACTIVIDAD_MAGVIG2_PORCCOMPROMISOS | 10000.0 | 100.0 | 6 - SDS/FFDS |

## 17. Catálogo provisional de unidades para revisión del investigador

Se preparó [revision_unidades_gestion_inversion.csv](revision_unidades_gestion_inversion.csv), con delimitador `|` y una fila por cada una de las 1.655 actividades. Es un instrumento de revisión, no una nueva salida Silver ni una clasificación metodológica aprobada.

Incluye códigos de entidad, proyecto y actividad, descripción completa del Excel, descripción reconstruida de la celda de actividad del PDF, página, unidad literal candidata en ambos documentos, programación anual, entrega 2026 y porcentaje de cumplimiento fuente. Las cuatro columnas finales permiten registrar naturaleza, unidad y escala confirmadas por el investigador y sus observaciones.

La clasificación propuesta distingue 578 actividades con porcentaje explícito en la descripción, 998 con cantidades/productos probables, 72 con magnitudes de longitud o área (dos sin dimensión resuelta), cinco tasas, una duración en minutos y una puntuación/índice cuya escala no se conoce. La unidad literal candidata concuerda entre las descripciones del Excel y del PDF después de reconstruir títulos fragmentados. Esto no demuestra que la escala numérica registrada sea correcta ni convierte todas las propuestas en decisiones definitivas.

Hay cinco dudas expresas: proyecto 008141 actividades 36 y 39 (denominador de las tasas), proyecto 007984 actividad 1 y proyecto 008013 actividad 4 (metros lineales o cuadrados), y proyecto 008103 actividad 4 (escala de satisfacción). Esos casos aparecen primero. Los restantes registros siguen siendo revisables: la ausencia de una duda detectada por estas reglas no equivale a validación humana de su ficha metodológica.

El signo % del PDF identifica los porcentajes de cumplimiento en sus columnas. La magnitud física puede imprimirse sin ese signo aunque su descripción diga Porciento. Por eso se combina la descripción, el encabezado, el contexto de actividad y los valores; no se usa el formato numérico ni el tipo de anualización como única evidencia.

No se modificó el notebook ni los CSV Silver al generar este catálogo. Antes de convertir magnitudes físicas porcentuales se debe confirmar su escala y separar su significado del cociente de cumplimiento.

## Fuentes locales

Las conclusiones se apoyan en los originales y en sus diccionarios/notas, no en conocimiento supuesto de datos internos. Las referencias de página remiten al PDF identificado al comienzo; las referencias de fila incluyen el encabezado del Excel.

- [Reporte PDF de Gestión-Inversión](<Capa Bronce/2026-06-30/gestion_inversion/planacciongestioninversion20260630_entidades_a2ddd1077b.pdf>)
- [Libro Excel de Gestión-Inversión](<Capa Bronce/2026-06-30/gestion_inversion/planacciongestioninversion20260630_ver20260728_75dd904475.xlsx>)


## 18. Auditoría de las salidas por vigencia y ejemplo de cicloinfraestructura

Se realizó una revisión independiente sobre las 1.655 actividades y las cuatro vigencias:

- 244.940 celdas fuente exportadas comparadas con el Excel original, sin cambios indebidos.
- 39.720 importes RECVIG/RECKVIG conservan sus valores y escala en millones; no se reescalaron ni recalcularon.
- 33.100 celdas MAGVIG originales conservadas, incluidos porcentajes publicados.
- 13.240 evaluaciones de compromiso y entrega verificadas contra la programación de su propia actividad y vigencia.
- Cuatro archivos de 1.655 filas y 40 campos, sin MAGTOTAL, RECKTOTAL ni mediciones de otra vigencia.
- Contraste geométrico de las 1.655 actividades con el PDF y de 23.883 celdas anuales explícitas de magnitudes/cumplimiento. Las discrepancias entre fuentes son incidencias originales; no se sustituyeron cifras del Excel por las del PDF.

**Ejemplo confirmado en PDF, página 41:** proyecto 007998, actividad 2, “Implementar 28 kilómetros de señalización y/o demarcación de cicloinfraestructura”. En 2026 programa 12,12 km, compromete 12,12 km y entrega 0,13 km. El compromiso recalculado es 1 (100 %) y la entrega recalculada es 0,0107260726072607 (aproximadamente 1,07 %). El PDF muestra 1,07 %. No hay que usar comprometido como si fuese entregado ni interpretar 0,010726 como 0,010726 %.

La descripción identifica la unidad y la meta del horizonte, pero no reemplaza el valor de entrega de la vigencia: los 28 km del texto no son el denominador anual de 2026. Un kilometraje fraccionario es válido: 0,13 km son 130 metros.

Los vacíos calculados se explican por 1.180 casos de programación no positiva en años no futuros y 3.310 resultados de 2027 no evaluables al corte de junio de 2026. Son celdas de resultado, no filas perdidas. No se detectaron desplazamientos de columnas; textos con punto y coma requieren importar únicamente con | y comillas como calificador.

Esta auditoría valida conservación y reproducción de fórmulas adoptadas. No demuestra por sí sola la veracidad de las magnitudes autorreportadas ni subsana las diferencias de versión o erratas de las fuentes.


## Control semántico de posibles mezclas de escala en MAGVIG

Una comprobación aritmética no demuestra que comprometido y entregado estén en la misma escala que programado. Una actividad de una estrategia puede registrar 63,7 por un posible porcentaje cargado en una celda de cantidad; también existen sobrejecuciones válidas. La coma decimal no identifica por sí misma un porcentaje.

Este control identifica candidatas cuya descripción expresa una unidad final, cuya programación anual está entre 0 y 1 y cuya ejecución o compromiso está entre 1 y 100. No aplica una corrección automática: se debe confirmar si cada columna representa cantidad, fracción de producto o porcentaje de cumplimiento. Una misma fila puede mezclar escalas entre comprometido y entregado.

Los resultados exportados conservan las magnitudes fuente. Los índices calculados sobre una fila señalada son reproducibles aritméticamente, pero no deben darse por validados semánticamente hasta resolver la escala. El PDF puede repetir el mismo error de registro y no constituye una confirmación independiente solo por coincidir.

Casos de referencia del corte 2026: proyecto 007926 actividad 3, 63,7 frente a 1; proyecto 008100 actividad 5, 47 frente a 1; proyecto 008180 actividad 3, comprometido 24 pero entregado 0,49 frente a 1. Si los primeros valores se confirman como porcentajes de una unidad, su magnitud comparable sería valor/100. No se generaliza esa operación a todas las cifras superiores a la meta.

El proyecto 008028 actividad 6, eventos programados 10 y entrega 47,7, requiere distinguir entre 47,7 unidades reportadas y un eventual 47,7 % del programa anual. El archivo publica 477 %, por lo que no confirma la segunda interpretación. El proyecto 008050 actividad 4, archivo sonoro, sí conserva una entrega fraccionaria anual 0,46 sobre 0,3 programado; la relación anual es 1,5333 y el acumulado de entregas 2024–2026 es 0,86.


</details>




### Antecedentes adicionales de la interpretación temporal y la anualización

## 4. Cómo separo conversión, medición y diagnóstico temporal

La versión anterior mezclaba una conversión de unidades con correcciones inferidas para una lista de actividades y fechas. Eliminé esa lista. Ahora todas las actividades pasan por las mismas reglas y cada medida produce tres resultados:

| Sufijo | Qué contiene | Cómo lo utilizo |
|---|---|---|
| `_NORMALIZADO` | Porcentaje publicado dividido exactamente entre cien | Auditoría y lectura del dato oficial; puede conservar anomalías de origen |
| `_PARA_ANALISIS` | La misma proporción, solo cuando pasa los controles técnicos | Uso analítico condicionado a cobertura y estado; no es una estimación ni una corrección oficial |
| `_ESTADO_CALIDAD` | El motivo que habilita o impide el uso | Distingo impedimentos aritméticos o de aplicabilidad de alertas exploratorias que no excluyen por sí solas |

El total físico solo se habilita para Suma; para los otros tipos marco `TOTAL_NO_APLICABLE_ANUALIZACION`, aunque una edición futura rellene sus totales. Conservo siempre el total publicado.

Una proporción de 1 equivale a 100 %. No aplico min–max, no limito los resultados a uno y no invierto porcentajes por el solo hecho de que la anualización sea Decreciente. Los sobrecumplimientos que pasan los controles permanecen por encima de uno.

Compruebo denominador, valores finitos, fechas y concordancia entre porcentaje y magnitud con una tolerancia de 0,0051 puntos porcentuales y relativa de 1e-9. Una discrepancia impide habilitar la medida sin modificar el valor publicado. También señalo una posible mezcla de escala de un producto único. Es una pista textual, no una unidad certificada: esa alerta por sí sola no modifica ni excluye la proporción.

Para el contraste temporal exijo la misma identidad de plan, entidad, proyecto, actividad, vigencia, definición y tipo de anualización y una programación comparable. Excluyo el corte objetivo, los posteriores, las observaciones de la misma fecha o posteriores y las ediciones que el portal publicó después del corte evaluado. Una edición sin fecha de disponibilidad no actúa como antecedente.

Comparo como máximo tres antecedentes distintos y coherentes. Un salto grande compatible con un factor decimal 10 o 100 se marca para confirmar; no lo divido automáticamente ni lo uso por sí solo para dejar vacía la medida analítica. Una actividad puede concentrar su ejecución al cierre o realizar entregas discontinuas. Los umbrales son criterios de alerta, no probabilidades ni pruebas de que un dato sea falso. Si no tengo historia comparable, lo hago explícito y mantengo los controles del archivo actual.

En particular, no uso septiembre para decidir sobre junio. Aunque la revisión retrospectiva ayuda a comprender un caso, las decisiones ejecutables excluyen esa información futura. La referencia se carga desde este notebook y se comprueba su huella antes de utilizarla.


### 4.1. Por qué la anualización cambia la interpretación y no la conversión

Al explorar junio de 2026 encontré 927 actividades Suma, 553 Constantes, 169 Crecientes y 6 Decrecientes en el conjunto distrital. Por ello no confundo el comportamiento de una actividad con el de otra ni traslado al total distrital la ausencia de Decrecientes que pueda observar en un sector particular.

Contrasté esta interpretación con el [manual de programación del Plan de Acción de la SDP, M-IN-004, versión 9](https://www.sdp.gov.co/sites/default/files/05_1410_manualprogramacion_plan_de_accion-v9.pdf) y el [manual de reprogramación M-IN-006, versión 9](https://www.sdp.gov.co/sites/default/files/10_1410_manual_reprogr_actuliz_plan_accion-v9.pdf), además del diccionario de cada Excel. Los uso para interpretar la anualización; no sustituyo la fórmula publicada de una actividad por la de indicadores de otro instrumento.

| Tipo | Cómo interpreto las vigencias | Qué evito en la limpieza |
|---|---|---|
| Suma | Las magnitudes anuales son aditivas si conservan unidad, definición y cobertura. En el ejemplo 15 + 35 + 6,27 sobre una meta total de 100 obtengo 56,27 % acumulado. | No sumo los porcentajes anuales: sus denominadores pueden ser distintos. Tampoco sumo sucesivos cortes del mismo año. |
| Constante | Cada vigencia repite un estándar. Un 38,98 % de 2026 se interpreta frente a su programación de 2026. | No convierto cuatro programaciones de 100 % en una meta de 400 %, ni agrego cumplimientos anteriores al de 2026. |
| Creciente | Las magnitudes son niveles acumulados: un nivel posterior incorpora el anterior; 15, 51, 87 y 120 describen una trayectoria hacia 120. | No sumo esos niveles. Una diferencia 20 − 10 = 10 solo representa incremento si unidad, definición, programación y versión son comparables. |
| Decreciente | Interpreto niveles que buscan reducirse, sin sumarlos. | No invierto automáticamente el porcentaje ni uso 1 − p. Para construir logro de reducción necesitaría línea base, meta y fórmula certificadas. |

El manual M-IN-004, páginas 17–18, contempla que en Constante la programación del primer año pueda ser inferior por corresponder a un semestre. No impongo igualdad ciega de todos los años ni interpreto esa diferencia como un error. Distingo las reglas de programación de la evolución de lo efectivamente entregado.

Los cortes trimestrales son instantáneas y pueden revisar valores de años anteriores. Una diferencia entre archivos no equivale necesariamente a una ejecución del trimestre. Un cambio de programación, definición, unidad o fe de erratas puede romper la comparabilidad. No exijo crecimiento lineal ni monotonicidad a la entrega o al compromiso como prueba automática de verdad.

Mi decisión es conservar el porcentaje oficial y dividirlo exactamente entre cien, sin recalcularlo desde una trayectoria supuesta. La conversión es la misma para los cuatro tipos porque cambia la representación, no la anualización: 38,98 % se representa como 0,3898. No fuerzo el intervalo [0, 1]: un sobrecumplimiento puede superar 1. En una actividad Decreciente, que el cociente magnitud/programado sea coherente no demuestra que mida correctamente el logro de reducción.

Mantengo separadas magnitud comprometida/contratada y magnitud entregada, y las distingo de los giros financieros. Contrastar el porcentaje con su magnitud sirve para descubrir discrepancias, no para reemplazar el valor oficial. Tampoco doy el mismo peso a actividades heterogéneas por compartir una escala proporcional: una media simple no constituye automáticamente un indicador de avance sectorial.

En la siguiente celda compruebo los totales físicos únicamente en Suma y describo los ceros de totales en los otros tipos. Este contraste es diagnóstico: no reconstruyo ni sobrescribo `MAGTOTAL`. Una diferencia puede obedecer a revisión, redondeo o medición y requiere examinar la fuente.


#### Lo que encontré al comprobar los totales de Suma

En junio y septiembre de 2026 encontré, en cada archivo, una diferencia entre la programación total y la suma de las vigencias, seis en compromisos y seis en entrega. Todas las filas afectadas contienen referencias a fe de erratas de SDHT o IPES. Esta coincidencia respalda la necesidad de consultar las correcciones, pero no identifica por sí sola qué celda es correcta.

Por ejemplo, la actividad `0118 / 008084 / 4` publica programación anual 6, 11, 16 y 20, cuya suma es 53, y programación total 48. La hoja `FeDeErratas` del original de junio relaciona el radicado SDHT 2-2026-6747 y SDP 1-2026-05672 con una solicitud de ajuste de magnitudes y recursos. No contiene una tabla completa que permita sustituir inequívocamente cada cifra. Por ello no presento 53 como una corrección certificada de 48 ni reconstruyo todos los porcentajes desde esa suma.

Esta observación no cambia la definición aditiva de Suma: distingue la regla metodológica de la conciliación efectiva de una publicación. Conservo los componentes y el total, la referencia `FE_DE_ERRATAS` y los originales de Bronce. Un porcentaje total coherente con sus propios numerador y denominador todavía puede estar asociado a componentes que no concilian; pasar el control aritmético no resuelve esa diferencia. Antes de un análisis acumulado debo revisar esa evidencia, sin sumar porcentajes ni sustituir totales automáticamente.



</details>
